In [1]:
print('start')

start


In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [4]:
#Monomeric models
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    df.columns = [clean_name(col) for col in df.columns]
    return df

In [5]:
#Monomer composition
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_RRCK.csv')
df_mc_train = clean_feature_names(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_RRCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(140, 385)
(140,)
(35, 385)
(35,)
0.5020103392671724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.167886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 6
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning]

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.2791680941772403


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2991,0.4210,0.5469,0.2358,0.5629,0.5860,0.2300,0.3438,0.4796,0.5020,0.7086,0.6285
LGBMRegressor,0.3071,0.4470,0.5542,0.2153,0.4656,0.4493,0.2842,0.4346,0.5331,0.3848,0.6450,0.5956
XGBRegressor,0.2517,0.3715,0.5017,0.3569,0.6355,0.6461,0.2207,0.3296,0.4698,0.5222,0.7255,0.6241
DecisionTreeRegressor,0.3586,0.4478,0.5989,0.0837,0.5167,0.5410,0.2181,0.3210,0.4670,0.5279,0.7280,0.6550
RandomForestRegressor,0.2425,0.3800,0.4925,0.3803,0.6180,0.6357,0.2260,0.3503,0.4754,0.5107,0.7242,0.6414
GradientBoostingRegressor,0.2701,0.3886,0.5197,0.3098,0.5644,0.6006,0.2110,0.3470,0.4593,0.5433,0.7491,0.6607
AdaBoostRegressor,0.2939,0.4461,0.5421,0.2492,0.5129,0.4935,0.2701,0.4303,0.5197,0.4153,0.7262,0.6971
SVR,0.2799,0.4183,0.5291,0.2847,0.5366,0.5157,0.2361,0.3515,0.4859,0.4888,0.7021,0.6695
LinearRegression,0.4740,0.5067,0.6885,-0.2110,0.4154,0.3817,0.3116,0.4142,0.5582,0.3253,0.6435,0.6217
KNeighborsRegressor,0.2490,0.3784,0.4990,0.3637,0.6267,0.6165,0.3223,0.4223,0.5677,0.3023,0.5630,0.4995


In [6]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.788799999999997, -5.13999999999999, -5.139...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.123099999999997, -6.269999999999991, -6.1...","[-6.076819999999997, -6.24901999999999, -6.225...","[0.04618603252066436, 0.319420803330024, 0.417..."
1,LGBMRegressor,"[-5.742027732510554, -5.700094420501769, -5.70...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.742027732510554, -5.742027732510554, -6.2...","[-5.81770737661578, -5.81770737661578, -6.3145...","[0.046600469799401745, 0.046600469799401745, 0..."
2,XGBRegressor,"[-5.9843497, -5.1450787, -5.1450787, -5.665011...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1708407, -6.259389, -5.8048453, -6.143747...","[-6.209151, -6.258704, -5.9441133, -6.1893244,...","[0.078049764, 0.3154796, 0.15457404, 0.1275968..."
3,DecisionTreeRegressor,"[-6.13, -5.14, -5.14, -5.57, -5.28, -5.57, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -6.27, -6.42, -6.89, -5.4, -5.14, -6....","[-6.026, -6.27, -6.218, -6.779999999999999, -6...","[0.12737346662472515, 0.3225523213371748, 0.59..."
4,RandomForestRegressor,"[-5.9400999999999975, -5.314999999999993, -5.3...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.105699999999998, -6.153349999999996, -6.3...","[-6.109895999999998, -6.120031999999996, -6.09...","[0.032013997938401526, 0.16820722783518963, 0...."
5,GradientBoostingRegressor,"[-5.990264054932226, -5.269259417032786, -5.26...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.065897537564234, -6.065897537564234, -6.6...","[-6.064839199589315, -6.027947290796168, -6.29...","[0.08552865774390354, 0.10527631627584394, 0.3..."
6,AdaBoostRegressor,"[-6.009642857142856, -5.694156626506025, -5.69...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0369736842105235, -6.0369736842105235, -6...","[-6.139282655713357, -6.1022693357725855, -6.0...","[0.07563802713272358, 0.07765604649612239, 0.1..."
7,SVR,"[-5.8112235889124175, -5.240067495196884, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.902418916444308, -5.795745822077137, -6.2...","[-5.91362387774158, -5.863231626216331, -6.144...","[0.036374248073893715, 0.10350850620034273, 0...."
8,LinearRegression,"[-5.510357810448829, -5.59695861516724, -5.596...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.481012890669073, -6.203411202248113, -6.2...","[-6.474080592588871, -6.2976974305266635, -6.2...","[0.20206885438060623, 0.30551107029350855, 0.7..."
9,KNeighborsRegressor,"[-5.716666666666666, -5.18, -5.18, -5.60333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -6.136666666666667, -6.4...","[-6.0233333333333325, -6.11, -6.08333333333333...","[0.14883249346534266, 0.12452576707921424, 0.3..."


In [7]:
result_df.to_csv('results/Monomeric/Monomer_comp_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_prediction_data_RRCK.csv')

In [8]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [9]:
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_RRCK.csv')
df_mc_train = clean_feature_names(df_mc_train)
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_RRCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(140, 67)
(140,)
(35, 67)
(35,)
0.5055171613723969
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.293365 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 35
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 6
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


-0.4384290120785843


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2996,0.4203,0.5474,0.2345,0.5621,0.5811,0.2284,0.3416,0.4779,0.5055,0.7111,0.6268
LGBMRegressor,0.3071,0.4470,0.5542,0.2153,0.4656,0.4493,0.2842,0.4346,0.5331,0.3848,0.6450,0.5956
XGBRegressor,0.2517,0.3715,0.5017,0.3569,0.6355,0.6461,0.2207,0.3296,0.4698,0.5222,0.7255,0.6241
DecisionTreeRegressor,0.3545,0.4380,0.5954,0.0943,0.5242,0.5373,0.1851,0.2895,0.4302,0.5993,0.7755,0.6878
RandomForestRegressor,0.2458,0.3815,0.4958,0.3719,0.6115,0.6298,0.2265,0.3512,0.4759,0.5096,0.7234,0.6463
GradientBoostingRegressor,0.2717,0.3892,0.5212,0.3058,0.5633,0.6000,0.2114,0.3469,0.4598,0.5422,0.7485,0.6570
AdaBoostRegressor,0.3000,0.4491,0.5477,0.2335,0.4954,0.4731,0.2775,0.4385,0.5268,0.3992,0.7035,0.6672
SVR,0.2799,0.4183,0.5291,0.2847,0.5366,0.5157,0.2361,0.3515,0.4859,0.4888,0.7020,0.6695
LinearRegression,0.4740,0.5067,0.6885,-0.2110,0.4154,0.3821,0.3116,0.4142,0.5582,0.3253,0.6435,0.6217
KNeighborsRegressor,0.2493,0.3795,0.4993,0.3629,0.6264,0.6152,0.3240,0.4229,0.5692,0.2985,0.5604,0.4995


In [10]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.778199999999997, -5.13999999999999, -5.139...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.111999999999997, -6.270399999999991, -6.2...","[-6.0844799999999974, -6.244359999999991, -6.2...","[0.03353102444006066, 0.318427678445197, 0.416..."
1,LGBMRegressor,"[-5.742027732510554, -5.700094420501769, -5.70...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.742027732510554, -5.742027732510554, -6.2...","[-5.81770737661578, -5.81770737661578, -6.3145...","[0.046600469799401745, 0.046600469799401745, 0..."
2,XGBRegressor,"[-5.9843497, -5.1450787, -5.1450787, -5.665011...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1708407, -6.259389, -5.8048453, -6.143747...","[-6.209151, -6.258704, -5.9441133, -6.1893244,...","[0.078049764, 0.3154796, 0.15457404, 0.1275968..."
3,DecisionTreeRegressor,"[-6.13, -5.14, -5.14, -5.57, -5.32, -5.57, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -6.27, -5.45, -6.89, -5.4, -5.14, -5....","[-6.183999999999999, -6.13, -6.174, -6.7799999...","[0.25842600488340955, 0.42712995680471766, 0.4..."
4,RandomForestRegressor,"[-5.9313499999999975, -5.307599999999992, -5.3...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.104199999999999, -6.129349999999996, -6.2...","[-6.115952666666665, -6.118638666666661, -6.07...","[0.019883260005452805, 0.16240736533106226, 0...."
5,GradientBoostingRegressor,"[-5.990264054932226, -5.269259417032786, -5.26...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.065897537564234, -6.065897537564234, -6.6...","[-6.064839199589315, -6.027947290796168, -6.29...","[0.08552865774390354, 0.10527631627584394, 0.3..."
6,AdaBoostRegressor,"[-5.908970588235293, -5.597010869565219, -5.59...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.114999999999998, -6.114999999999998, -6.0...","[-6.136172760800842, -6.136172760800842, -6.04...","[0.12170703174820145, 0.12170703174820145, 0.0..."
7,SVR,"[-5.811223075663906, -5.240067304259033, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.902417919160923, -5.795744668779475, -6.2...","[-5.913621302540199, -5.863235277811801, -6.14...","[0.03637539664267794, 0.10352254675450849, 0.2..."
8,LinearRegression,"[-5.5103578104488244, -5.596958615167241, -5.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.48101289066907, -6.203411202248114, -6.28...","[-6.474080592588872, -6.297697430526664, -6.25...","[0.20206885438060684, 0.30551107029350877, 0.7..."
9,KNeighborsRegressor,"[-5.716666666666666, -5.18, -5.18, -5.60333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -6.136666666666667, -6.4...","[-6.0233333333333325, -6.11, -6.08333333333333...","[0.14883249346534266, 0.12452576707921424, 0.3..."


In [11]:
const_col

['Ala_indol_2_yl_',
 'dAla_indol_2_yl_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 '2Abz',
 'Aib',
 'Aoc_2_',
 '5_Ava',
 'Bal',
 'Me_Bal',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'dCha',
 'D',
 'Asp_piperidide',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'dPhe_4_F_',
 'Phe_4_CF3_',
 'Phe_4_NO2_',
 'Phe_CHF2_',
 'dPhe_3_4_diF_',
 'Et_Phe',
 'H2NEt_Phe',
 'Me_Phe_3_Cl_',
 'Me_Phe_4_Cl_',
 'Me_Phe_a_b_dehydro_',
 'G',
 'Bn_Gly',
 'Bn_4_Cl__Gly',
 'Bu_Gly',
 'Et_Gly',
 'EtOEt_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'Pr_Gly',
 'PhEt_Gly',
 'cHexCH2_Gly',
 'isoamyl_Gly',
 'pentyl_Gly',
 '3_pyridylethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'H',
 'Me_Hph',
 'bHph',
 'Hph_2_Cl_',
 'Hph_3_Cl_',
 'Hph_4_Cl_',
 'Hse_Et_',
 'dHyp',
 'Hyp_Et_',
 'dI',
 'Me_dI',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'K',
 'dK',
 'meK',
 'Me_dK',
 'Lys_Ac_',
 'Lys_Cbz_',
 'Lys_iPr_',
 'Lys_Me_',
 'Me_Lys_Me_',
 'Lys_Me2_',
 'Lys_Tfa_',
 'aMeLeu

In [12]:
result_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_prediction_data_RRCK.csv')

In [13]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [14]:
df_train = pd.read_csv('features/Monomeric/Train_mon_comp_RRCK.csv')
df_mc_train = clean_feature_names(df_train)
df_mc_train = df_mc_train.drop(['ID','SMILES','Permeability'],axis=1)
df_mc, const_col = remove_low_variance_columns(df_mc_train)
X_train = df_mc
y_train = df_train['Permeability']
print(X_train.shape)
print(y_train.shape)

df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_RRCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(140, 9)
(140,)
(35, 9)
(35,)
0.3093157403495549
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.069256 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 4
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-2.11784336213666


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3683,0.4727,0.6069,0.0589,0.4295,0.4381,0.3190,0.4692,0.5648,0.3093,0.5964,0.5509
LGBMRegressor,0.3110,0.4513,0.5577,0.2053,0.4548,0.4147,0.2800,0.4360,0.5291,0.3939,0.6574,0.6427
XGBRegressor,0.3899,0.4810,0.6244,0.0037,0.4107,0.4184,0.4068,0.5333,0.6378,0.1194,0.4287,0.3967
DecisionTreeRegressor,0.4808,0.5184,0.6934,-0.2285,0.3286,0.3675,0.3472,0.5077,0.5892,0.2484,0.5750,0.5288
RandomForestRegressor,0.3254,0.4483,0.5704,0.1686,0.4642,0.4545,0.2996,0.4492,0.5474,0.3513,0.6194,0.5790
GradientBoostingRegressor,0.3120,0.4394,0.5585,0.2029,0.4984,0.5046,0.3137,0.4641,0.5601,0.3209,0.5848,0.5156
AdaBoostRegressor,0.2980,0.4405,0.5458,0.2387,0.4904,0.4809,0.2916,0.4438,0.5400,0.3687,0.6541,0.6724
SVR,0.3400,0.4677,0.5831,0.1313,0.4113,0.4094,0.3340,0.4563,0.5779,0.2769,0.5503,0.4865
LinearRegression,0.3632,0.4996,0.6026,0.0721,0.3461,0.3332,0.3207,0.4639,0.5663,0.3056,0.5682,0.5387
KNeighborsRegressor,0.3398,0.4670,0.5829,0.1319,0.4335,0.4188,0.3032,0.4599,0.5506,0.3436,0.6691,0.6695


In [15]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.134999999999994, -5.13999999999999, -5.139...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.134999999999994, -6.134999999999994, -6.1...","[-6.237166666666668, -6.237166666666668, -6.23...","[0.07661882565300386, 0.07661882565300386, 0.3..."
1,LGBMRegressor,"[-5.60659224777032, -5.633452559685933, -5.633...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.60659224777032, -5.60659224777032, -6.264...","[-5.674823495914751, -5.674823495914751, -6.25...","[0.07125095056547585, 0.07125095056547585, 0.0..."
2,XGBRegressor,"[-6.134915, -5.1393714, -5.1393714, -5.5981593...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.134915, -6.134915, -5.448362, -5.5981593,...","[-6.2368407, -6.2368407, -5.4420767, -5.648992...","[0.076408885, 0.076408885, 0.16449516, 0.05047..."
3,DecisionTreeRegressor,"[-6.135000000000001, -5.14, -5.14, -5.59777777...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.135000000000001, -6.135000000000001, -6.3...","[-6.237166666666667, -6.237166666666667, -6.68...","[0.0766188256529975, 0.0766188256529975, 0.202..."
4,RandomForestRegressor,"[-6.098335952380951, -5.248833333333327, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.098335952380951, -6.098335952380951, -6.0...","[-6.210934578643578, -6.210934578643578, -6.10...","[0.08247339118230439, 0.08247339118230439, 0.2..."
5,GradientBoostingRegressor,"[-6.030221985267229, -5.086368104421397, -5.08...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.030221985267229, -6.030221985267229, -5.5...","[-6.139127599890716, -6.139127599890716, -5.81...","[0.0718074821235166, 0.0718074821235166, 0.257..."
6,AdaBoostRegressor,"[-5.869999999999999, -5.555246913580247, -5.55...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.869999999999999, -5.869999999999999, -5.9...","[-5.953927156177157, -5.953927156177157, -5.96...","[0.12279674375304707, 0.12279674375304707, 0.0..."
7,SVR,"[-5.6596818703680025, -5.240174804825071, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.6596818703680025, -5.6596818703680025, -6...","[-5.738553876517672, -5.738553876517672, -6.28...","[0.07869585293903066, 0.07869585293903066, 0.1..."
8,LinearRegression,"[-5.623356395830175, -5.560588095793635, -5.56...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.623356395830175, -5.623356395830175, -5.9...","[-5.629842064930592, -5.629842064930592, -5.91...","[0.0665386149312745, 0.0665386149312745, 0.085..."
9,KNeighborsRegressor,"[-6.136666666666667, -5.18, -5.18, -5.63, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.136666666666667, -6.136666666666667, -6.2...","[-6.258666666666667, -6.258666666666667, -6.38...","[0.13198653129938848, 0.13198653129938848, 0.2..."


In [16]:
result_df.to_csv('results/Monomeric/Monomer_comp_LVR_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_LVR_prediction_data_RRCK.csv')

In [17]:
#AA composition
df_aac_train = pd.read_csv('features/Monomeric/Train_aac_RRCK.csv')
X_train = df_aac_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_aac_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_aac_test = pd.read_csv('features/Monomeric/Test_aac_RRCK.csv')
X_test = df_aac_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_aac_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(140, 21)
(140,)
(35, 21)
(35,)
0.3703046050421719
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.179635 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 8
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-2.4935918861800572


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3354,0.4618,0.5791,0.1431,0.4566,0.4704,0.2908,0.4320,0.5393,0.3703,0.6100,0.5603
LGBMRegressor,0.2984,0.4561,0.5462,0.2376,0.4882,0.4777,0.2981,0.4377,0.5460,0.3547,0.6014,0.6200
XGBRegressor,0.3139,0.4420,0.5602,0.1981,0.5110,0.4848,0.2905,0.4357,0.5389,0.3711,0.6152,0.6038
DecisionTreeRegressor,0.3274,0.4567,0.5722,0.1634,0.5176,0.4696,0.2671,0.4167,0.5169,0.4216,0.6625,0.6473
RandomForestRegressor,0.2970,0.4386,0.5449,0.2413,0.5041,0.5207,0.2776,0.4150,0.5268,0.3991,0.6350,0.6024
GradientBoostingRegressor,0.2949,0.4335,0.5430,0.2466,0.5210,0.5311,0.3082,0.4410,0.5552,0.3326,0.5847,0.5623
AdaBoostRegressor,0.3035,0.4514,0.5509,0.2246,0.4766,0.4800,0.3046,0.4444,0.5519,0.3406,0.6017,0.6103
SVR,0.3301,0.4475,0.5746,0.1565,0.4442,0.4555,0.2926,0.4541,0.5409,0.3666,0.6200,0.5910
LinearRegression,0.3526,0.4739,0.5938,0.0990,0.3726,0.4329,0.2988,0.4366,0.5466,0.3531,0.5988,0.5951
KNeighborsRegressor,0.4093,0.5002,0.6397,-0.0457,0.2999,0.3130,0.3391,0.4587,0.5823,0.2659,0.5177,0.4813


In [18]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.552799999999998, -5.11000000000001, -5.110...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1603999999999965, -6.269999999999991, -6....","[-6.058469999999996, -6.269999999999989, -6.23...","[0.1267661058800823, 0.32255232133717116, 0.35..."
1,LGBMRegressor,"[-5.948857836992893, -5.63283632081995, -5.632...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.948857836992893, -5.948857836992893, -6.0...","[-6.112626537876496, -6.134690667507222, -6.13...","[0.08605666571045421, 0.10203377141490169, 0.0..."
2,XGBRegressor,"[-6.245441, -5.110323, -5.110323, -6.1843934, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.157541, -6.2696843, -6.235048, -6.49473, ...","[-5.862521, -6.269765, -6.2892585, -6.376748, ...","[0.25338367, 0.3213482, 0.3039692, 0.4134986, ..."
3,DecisionTreeRegressor,"[-6.27, -5.109999999999999, -5.109999999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -6.27, -6.89, -6.89, -6.42, -5.109999...","[-6.034000000000001, -6.27, -6.798, -6.8439999...","[0.11825396399275576, 0.3225523213371748, 0.11..."
4,RandomForestRegressor,"[-6.008099999999995, -5.152753571428571, -5.15...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.032283333333331, -6.171749999999996, -6.3...","[-6.052384666666665, -6.223011999999994, -6.31...","[0.05447261275581073, 0.15826674311427322, 0.2..."
5,GradientBoostingRegressor,"[-6.0925625807361286, -5.195509152137764, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.217454573020618, -6.200374638359659, -6.3...","[-5.982501753643666, -6.221314173057616, -6.24...","[0.19680275692611301, 0.1993464449651446, 0.45..."
6,AdaBoostRegressor,"[-6.08296296296296, -5.551807228915661, -5.551...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.08296296296296, -6.08296296296296, -6.224...","[-6.109685206228957, -6.117670132699544, -6.18...","[0.10407179983167296, 0.10088429036659788, 0.3..."
7,SVR,"[-5.745217198007522, -5.398046103648319, -5.39...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.042954106354762, -6.01036573846243, -6.09...","[-5.888558925856929, -5.963806788900119, -6.16...","[0.09170115544958658, 0.10477987849998283, 0.2..."
8,LinearRegression,"[-5.7914971360650025, -5.689133557568333, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.480031796341198, -6.123135088956289, -5.9...","[-6.111161890870153, -6.020507828366531, -6.14...","[0.19239778135976768, 0.07898561826678437, 0.2..."
9,KNeighborsRegressor,"[-5.53, -5.13, -5.13, -5.876666666666668, -4.9...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.136666666666667, -6.136666666666667, -6.0...","[-6.082, -6.11, -6.2346666666666675, -5.915999...","[0.1775462381090256, 0.12452576707921424, 0.22..."


In [19]:
aac_comp.to_csv('results/Monomeric/AAC_comp_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_prediction_data_RRCK.csv')

In [20]:
#Constant column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_RRCK.csv')
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_RRCK.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(140, 15)
(140,)
(35, 15)
(35,)
0.3767452636550458
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.178541 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 8
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-2.024217335236652


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3330,0.4591,0.5770,0.1493,0.4599,0.4767,0.2879,0.4317,0.5365,0.3767,0.6150,0.5729
LGBMRegressor,0.2984,0.4561,0.5462,0.2376,0.4882,0.4777,0.2981,0.4377,0.5460,0.3547,0.6014,0.6200
XGBRegressor,0.3139,0.4420,0.5602,0.1981,0.5110,0.4848,0.2905,0.4357,0.5389,0.3711,0.6152,0.6038
DecisionTreeRegressor,0.3302,0.4575,0.5746,0.1563,0.5190,0.4765,0.2903,0.4320,0.5388,0.3714,0.6313,0.6236
RandomForestRegressor,0.2971,0.4388,0.5450,0.2410,0.5035,0.5207,0.2800,0.4176,0.5291,0.3938,0.6307,0.5979
GradientBoostingRegressor,0.2977,0.4356,0.5456,0.2394,0.5154,0.5241,0.3072,0.4397,0.5543,0.3349,0.5862,0.5570
AdaBoostRegressor,0.3154,0.4609,0.5616,0.1940,0.4452,0.4239,0.3125,0.4502,0.5590,0.3234,0.5794,0.6080
SVR,0.3301,0.4475,0.5746,0.1565,0.4442,0.4562,0.2927,0.4542,0.5410,0.3664,0.6198,0.5910
LinearRegression,0.3526,0.4739,0.5938,0.0990,0.3726,0.4329,0.2988,0.4366,0.5466,0.3531,0.5988,0.5951
KNeighborsRegressor,0.4150,0.5030,0.6442,-0.0603,0.2970,0.3203,0.3412,0.4524,0.5841,0.2614,0.5171,0.4730


In [21]:
aac_comp.to_csv('results/Monomeric/AAC_comp_const_rem_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_const_rem_prediction_data_RRCK.csv')

In [22]:
#LVR column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_RRCK.csv')
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
X_train, const_col = remove_low_variance_columns(X_train)

y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_RRCK.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_mc = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_mc, X_train,y_train, X_test,  y_test)
result_df

(140, 7)
(140,)
(35, 7)
(35,)
0.39578373070223705
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.183668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 54
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 7
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-2.7789592187388146


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3299,0.4536,0.5744,0.1570,0.4581,0.4613,0.2791,0.4410,0.5283,0.3958,0.6410,0.6462
LGBMRegressor,0.3469,0.4957,0.5890,0.1136,0.3417,0.3193,0.3476,0.4865,0.5896,0.2475,0.5422,0.5717
XGBRegressor,0.3635,0.4628,0.6029,0.0712,0.4026,0.4223,0.2915,0.4483,0.5399,0.3688,0.6160,0.6260
DecisionTreeRegressor,0.4014,0.4924,0.6335,-0.0255,0.3768,0.3955,0.2761,0.4370,0.5255,0.4022,0.6502,0.6329
RandomForestRegressor,0.3224,0.4477,0.5678,0.1763,0.4443,0.4558,0.2774,0.4304,0.5267,0.3993,0.6481,0.6729
GradientBoostingRegressor,0.3111,0.4485,0.5578,0.2051,0.4854,0.4814,0.3173,0.4622,0.5633,0.3131,0.5820,0.6202
AdaBoostRegressor,0.3480,0.4795,0.5899,0.1108,0.3426,0.3490,0.3438,0.4917,0.5864,0.2556,0.5439,0.6192
SVR,0.3446,0.4704,0.5870,0.1196,0.3997,0.4059,0.2878,0.4495,0.5365,0.3769,0.6330,0.6290
LinearRegression,0.4157,0.5151,0.6447,-0.0620,0.1505,0.2514,0.4376,0.5517,0.6615,0.0526,0.2779,0.4426
KNeighborsRegressor,0.4007,0.5041,0.6330,-0.0239,0.3006,0.2906,0.3420,0.4667,0.5848,0.2595,0.5233,0.5379


In [23]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.173599999999992, -5.11000000000001, -5.110...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.107599999999996, -6.269999999999991, -6.1...","[-6.057939999999997, -6.269999999999989, -6.18...","[0.1255580718233592, 0.32255232133717116, 0.37..."
1,LGBMRegressor,"[-5.783433338649384, -5.579452961762609, -5.57...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.75274206127544, -5.783433338649384, -6.06...","[-5.832775273247767, -5.888438566802146, -5.94...","[0.09423144374301569, 0.09173504143157721, 0.1..."
2,XGBRegressor,"[-6.2194376, -5.110135, -5.110135, -5.8090334,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.180749, -6.269203, -5.9770517, -6.04998, ...","[-6.1149263, -6.270471, -5.9365587, -6.089128,...","[0.1531494, 0.32102263, 0.22417374, 0.22699317..."
3,DecisionTreeRegressor,"[-6.27, -5.109999999999999, -5.109999999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -6.27, -6.89, -6.89, -6.42, -5.109999...","[-6.084, -6.27, -6.286, -6.632000000000001, -6...","[0.17083325203250097, 0.3225523213371748, 0.79..."
4,RandomForestRegressor,"[-6.193299999999996, -5.152913571428571, -5.15...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.024799999999999, -6.163049999999996, -6.3...","[-6.002097333333332, -6.1870919999999945, -6.2...","[0.04503946725558279, 0.1367527449669636, 0.23..."
5,GradientBoostingRegressor,"[-6.176967787280019, -5.217717286103449, -5.21...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.045531508399683, -6.176967787280019, -6.6...","[-6.047872539843193, -6.212846643503316, -6.39...","[0.1091629372651069, 0.19219829619495196, 0.33..."
6,AdaBoostRegressor,"[-6.014615384615384, -5.6130357142857115, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.906862745098037, -6.014615384615384, -5.9...","[-5.84713283040489, -5.8969548229548225, -6.01...","[0.0853167113527657, 0.1110243927016457, 0.222..."
7,SVR,"[-6.00998697975932, -5.24974252476008, -5.2497...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.833244353019012, -5.8599834984255, -6.350...","[-5.83976898764365, -5.895527386375006, -6.243...","[0.039778839607465595, 0.04357487848385047, 0...."
8,LinearRegression,"[-5.597559444999849, -5.663978407880999, -5.66...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.682534756118463, -5.564459579280318, -5.7...","[-5.802687994737708, -5.6697604486293205, -5.9...","[0.09913526662566563, 0.10276513133258128, 0.2..."
9,KNeighborsRegressor,"[-6.06, -5.13, -5.13, -5.43, -5.50333333333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.136666666666667, -5.9366666666666665, -5....","[-6.084666666666666, -5.960000000000001, -5.98...","[0.10234147633182639, 0.048350571638583674, 0...."


In [24]:
result_df.to_csv('results/Monomeric/AAC_comp_LVR_results_RRCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_LVR_prediction_data_RRCK.csv')

In [25]:
#Atomic models
df_train = pd.read_csv('features/Atomic/Train_all_atomic_desc_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Atomic/Test_all_atomic_desc_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_degree = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_degree, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 23)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 23)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.082596 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 134
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 7
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.7845243743433084


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2356,0.4115,0.4854,0.3980,0.6350,0.6050,0.2806,0.4080,0.5297,0.3925,0.6280,0.5731
DecisionTreeRegressor,0.2968,0.4306,0.5448,0.2417,0.6249,0.5869,0.1858,0.3337,0.4311,0.5977,0.7786,0.7410
RandomForestRegressor,0.1754,0.3409,0.4188,0.5519,0.7464,0.7280,0.1978,0.3226,0.4447,0.5718,0.7614,0.7183
GradientBoostingRegressor,0.1785,0.3366,0.4225,0.5438,0.7414,0.7133,0.1956,0.3367,0.4422,0.5766,0.7610,0.7326
AdaBoostRegressor,0.2202,0.3961,0.4692,0.4374,0.6730,0.6283,0.2265,0.3795,0.4759,0.5096,0.7380,0.6451
XGBRegressor,0.1875,0.3548,0.4331,0.5208,0.7286,0.7104,0.1968,0.3323,0.4436,0.5740,0.7607,0.7334
ExtraTreesRegressor,0.1551,0.3182,0.3938,0.6037,0.7780,0.7543,0.2029,0.3335,0.4504,0.5608,0.7532,0.7142
LinearRegression,0.3482,0.4808,0.5901,0.1103,0.4304,0.4329,0.2647,0.3972,0.5145,0.4269,0.6594,0.6011
KNeighborsRegressor,0.2327,0.3780,0.4824,0.4055,0.6555,0.6446,0.3034,0.3845,0.5509,0.3430,0.6161,0.5586
SVR,0.2322,0.3985,0.4818,0.4068,0.6390,0.5998,0.3014,0.4267,0.5490,0.3475,0.6008,0.5130


In [26]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.949393173388269, -5.546069928465827, -5.54...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.031290839426711, -6.059811349339949, -6.0...","[-6.144794996718112, -6.150499098700759, -6.14...","[0.06501401321131696, 0.05533615865115521, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -5.109999999999999, -5.109999999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.89, -5.76, -6.42, -6.89, -6.42, -5.109999...","[-6.548, -5.964, -6.406000000000001, -6.376, -...","[0.3505082024717823, 0.4080000000000002, 0.356..."
2,RandomForestRegressor,"[-6.072599999999994, -5.15291357142857, -5.152...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.177549999999994, -5.985399999999994, -6.0...","[-6.272569999999998, -6.109639999999994, -6.13...","[0.06966351699419265, 0.17628175855714623, 0.0..."
3,GradientBoostingRegressor,"[-5.963867437878151, -5.1864946643608185, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.61171971970209, -5.895025833200626, -6.04...","[-6.501123739992556, -6.062966097515329, -6.09...","[0.14472726714027276, 0.26242662098524877, 0.1..."
4,AdaBoostRegressor,"[-5.978620689655173, -5.686825396825394, -5.68...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.993437499999998, -5.978620689655173, -5.9...","[-6.269625408496732, -6.189565619412517, -6.17...","[0.1423638683002113, 0.11245885619769815, 0.09..."
5,XGBRegressor,"[-5.878516, -5.110041, -5.110041, -5.706893, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.464718, -5.7628493, -5.8696265, -6.604839...","[-6.195545, -5.95288, -6.2261534, -6.4206476, ...","[0.25556564, 0.3796274, 0.2464462, 0.19767247,..."
6,ExtraTreesRegressor,"[-6.052299999999995, -5.11000000000001, -5.110...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.986099999999999, -5.7599999999999945, -5....","[-6.135419999999999, -5.8935399999999944, -6.0...","[0.0798188298586246, 0.26707999999999926, 0.16..."
7,LinearRegression,"[-6.264077206996636, -6.079162072509698, -6.07...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.091853717097905, -6.051163193934663, -6.3...","[-6.0933554204985665, -6.081205897586668, -6.3...","[0.10981637478298084, 0.056471027891719365, 0...."
8,KNeighborsRegressor,"[-5.920000000000001, -5.13, -5.13, -6.10666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.920000000000001, -5.9...","[-6.068, -6.0680000000000005, -6.0146666666666...","[0.1273071526313868, 0.12730715263138598, 0.19..."
9,SVR,"[-5.930864916766745, -5.948606342142662, -5.94...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.883148727678627, -5.860342696349137, -6.0...","[-5.991325915488167, -5.9480578488344085, -6.0...","[0.07042430518694906, 0.07223868312792038, 0.2..."


In [27]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_RRCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_RRCK.csv')

In [28]:
#Atomic + monomeric_composition based features
df1 = pd.read_csv('features/Monomeric/Train_mon_comp_RRCK.csv')
df2 = pd.read_csv('features/Atomic/Train_all_atomic_desc_RRCK.csv')
df_train = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_train

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_Cl,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0.090909,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,0,74,12,0,0,0,0,106,0,1
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0.090909,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,0,73,13,0,0,0,0,107,0,1
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0.090909,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,0,73,12,0,0,0,0,106,0,1
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0.090909,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,0,73,12,0,0,0,0,106,0,1
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0.181818,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,0,72,12,0,0,0,0,106,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0.166667,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,35,6,0,6,0,0,60,1,1
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0.166667,0.166667,0.000000,0.000000,0.0,0.0,0.0,...,0,34,6,0,6,0,0,60,1,1
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0.000000,0.000000,0.333333,0.000000,0.0,0.0,0.0,...,0,38,6,0,0,0,0,54,0,1
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0.000000,0.000000,0.333333,0.000000,0.0,0.0,0.0,...,0,37,6,0,0,0,0,54,0,1


In [29]:
df1 = pd.read_csv('features/Monomeric/Test_mon_comp_RRCK.csv')
df2 = pd.read_csv('features/Atomic/Test_all_atomic_desc_RRCK.csv')
df_test = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_test

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_Cl,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.090909,0.090909,0.000000,0.000000,0.000000,0.0,0.0,...,0,74,12,0,0,0,0,111,0,1
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.090909,0.090909,0.000000,0.000000,0.000000,0.0,0.0,...,0,73,12,0,0,0,0,106,0,1
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.000000,0.000000,0.000000,0.200000,0.000000,0.0,0.0,...,0,64,10,0,6,0,0,100,1,1
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.000000,0.000000,0.200000,0.100000,0.000000,0.0,0.0,...,0,60,10,0,6,0,0,100,1,1
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.000000,0.000000,0.111111,0.222222,0.000000,0.0,0.0,...,0,58,9,0,6,0,0,91,1,1
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.000000,0.000000,0.125000,0.125000,0.000000,0.0,0.0,...,0,56,8,0,6,0,0,82,1,1
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0,55,8,0,6,0,0,82,1,1
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.000000,0.000000,0.000000,0.000000,0.166667,0.0,0.0,...,0,41,6,0,23,0,0,97,1,1
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0,51,8,0,6,0,0,81,1,1
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.000000,0.000000,0.000000,0.000000,0.166667,0.0,0.0,...,0,39,6,0,23,0,0,81,1,1


In [30]:
import re
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    df.columns = [clean_name(col) for col in df.columns]
    return df

In [31]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [32]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 408)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 408)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000572 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 169
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 13
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.2277433769659465


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2264,0.3941,0.4758,0.4216,0.6588,0.6248,0.2414,0.3616,0.4913,0.4774,0.6919,0.5814
DecisionTreeRegressor,0.3473,0.4373,0.5893,0.1127,0.5324,0.5172,0.1935,0.3080,0.4399,0.5811,0.7732,0.7279
RandomForestRegressor,0.1789,0.3338,0.4230,0.5428,0.7415,0.7369,0.1958,0.3040,0.4425,0.5760,0.7629,0.7072
GradientBoostingRegressor,0.1745,0.3183,0.4178,0.5541,0.7444,0.7378,0.1814,0.2968,0.4259,0.6072,0.7810,0.7185
AdaBoostRegressor,0.2192,0.3913,0.4682,0.4400,0.6878,0.6703,0.2263,0.3518,0.4757,0.5102,0.7384,0.6850
XGBRegressor,0.1833,0.3354,0.4281,0.5318,0.7340,0.7275,0.1952,0.3012,0.4418,0.5775,0.7604,0.7178
ExtraTreesRegressor,0.2004,0.3408,0.4476,0.4880,0.7012,0.7054,0.2142,0.3319,0.4628,0.5363,0.7327,0.6634
LinearRegression,1.6269,0.8080,1.2755,-3.1567,0.4027,0.4544,1.1728,0.6590,1.0830,-1.5392,0.2770,0.4224
KNeighborsRegressor,0.2714,0.3913,0.5210,0.3065,0.5887,0.5844,0.3532,0.4487,0.5943,0.2352,0.5227,0.4413
SVR,0.2512,0.3991,0.5012,0.3581,0.6008,0.5831,0.2461,0.3566,0.4961,0.4671,0.6852,0.6287


In [33]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_RRCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_RRCK.csv')

In [34]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 82)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 82)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000581 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 169
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 13
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


-0.5280727179864115


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2264,0.3941,0.4758,0.4216,0.6588,0.6248,0.2414,0.3616,0.4913,0.4774,0.6919,0.5814
DecisionTreeRegressor,0.3618,0.4415,0.6015,0.0756,0.5330,0.5191,0.2043,0.3179,0.4520,0.5577,0.7608,0.7189
RandomForestRegressor,0.1787,0.3350,0.4227,0.5435,0.7430,0.7411,0.1960,0.3051,0.4427,0.5757,0.7630,0.7095
GradientBoostingRegressor,0.1776,0.3236,0.4214,0.5463,0.7393,0.7314,0.1819,0.2946,0.4265,0.6062,0.7803,0.7203
AdaBoostRegressor,0.2099,0.3836,0.4582,0.4637,0.7091,0.7005,0.2105,0.3461,0.4588,0.5443,0.7695,0.7003
XGBRegressor,0.1833,0.3354,0.4281,0.5318,0.7340,0.7275,0.1952,0.3012,0.4418,0.5775,0.7604,0.7178
ExtraTreesRegressor,0.2083,0.3455,0.4564,0.4678,0.6893,0.6916,0.2126,0.3289,0.4611,0.5397,0.7348,0.6714
LinearRegression,1.6269,0.8080,1.2755,-3.1567,0.4027,0.4544,1.1728,0.6590,1.0830,-1.5392,0.2770,0.4224
KNeighborsRegressor,0.2716,0.3913,0.5212,0.3060,0.5888,0.5846,0.3531,0.4484,0.5942,0.2355,0.5230,0.4413
SVR,0.2512,0.3991,0.5012,0.3581,0.6008,0.5831,0.2462,0.3566,0.4962,0.4670,0.6851,0.6287


In [35]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.8983121656569715, -5.517529782171612, -5.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.991209156010933, -6.00744586627661, -6.35...","[-6.106255840028917, -6.109503182082053, -6.37...","[0.06745546933238637, 0.06200991367774981, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -5.14, -5.14, -5.74, -5.08, -5.57, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.7, -5.76, -6.13, -6.89, -6.13, -5.14, -5....","[-6.140000000000001, -5.964, -6.606, -6.843999...","[0.3031831129861952, 0.4080000000000002, 0.287..."
2,RandomForestRegressor,"[-6.140066666666664, -5.178599999999994, -5.17...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.155799999999996, -6.009399999999994, -6.3...","[-6.208299999999998, -6.0799599999999945, -6.3...","[0.045225302652387854, 0.16618028282561, 0.196..."
3,GradientBoostingRegressor,"[-6.035788489538349, -5.220821912225317, -5.22...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.507462629707532, -5.993767922224659, -5.9...","[-6.403946211654349, -6.101842961119112, -6.16...","[0.15938352108390086, 0.22111561097838617, 0.2..."
4,AdaBoostRegressor,"[-6.138888888888889, -5.417403846153846, -5.41...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.32, -6.140714285714284, -6.25692307692307...","[-6.338009304421068, -6.231249776149774, -6.19...","[0.08543022905079516, 0.07546603318021115, 0.1..."
5,XGBRegressor,"[-6.1299644, -5.140197, -5.140197, -5.6841226,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.387762, -5.7630253, -6.0062733, -6.387274...","[-6.1205673, -5.9375486, -6.294853, -6.336074,...","[0.16335614, 0.3494585, 0.22054584, 0.23873347..."
6,ExtraTreesRegressor,"[-6.054499999999997, -5.13999999999999, -5.139...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9784999999999995, -5.716299999999995, -6....","[-6.025779999999999, -5.858879999999994, -6.22...","[0.09346013909683633, 0.23791709816656656, 0.3..."
7,LinearRegression,"[-4.0, -5.510747474132141, -5.510747474132141,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.8818023788740605, -5.8551348740746905, -6...","[-6.75934663585623, -5.467799119377704, -6.118...","[1.4687076373823993, 0.7778812518845597, 0.618..."
8,KNeighborsRegressor,"[-5.716666666666666, -5.13, -5.13, -5.60333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-6.0233333333333325, -5.862, -6.4760000000000...","[0.14883249346534266, 0.06875398978321959, 0.1..."
9,SVR,"[-5.859634016716038, -5.267404519121085, -5.26...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9472015920776595, -5.891286824538007, -6....","[-5.950882839883007, -5.917264722731953, -6.21...","[0.031117433087564598, 0.0628201267179523, 0.2..."


In [36]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_const_rem_RRCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_const_rem_RRCK.csv')

In [37]:
const_col

['Ala_indol_2_yl_',
 'dAla_indol_2_yl_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 '2Abz',
 'Aib',
 'Aoc_2_',
 '5_Ava',
 'Bal',
 'Me_Bal',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'dCha',
 'D',
 'Asp_piperidide',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'dPhe_4_F_',
 'Phe_4_CF3_',
 'Phe_4_NO2_',
 'Phe_CHF2_',
 'dPhe_3_4_diF_',
 'Et_Phe',
 'H2NEt_Phe',
 'Me_Phe_3_Cl_',
 'Me_Phe_4_Cl_',
 'Me_Phe_a_b_dehydro_',
 'G',
 'Bn_Gly',
 'Bn_4_Cl__Gly',
 'Bu_Gly',
 'Et_Gly',
 'EtOEt_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'Pr_Gly',
 'PhEt_Gly',
 'cHexCH2_Gly',
 'isoamyl_Gly',
 'pentyl_Gly',
 '3_pyridylethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'H',
 'Me_Hph',
 'bHph',
 'Hph_2_Cl_',
 'Hph_3_Cl_',
 'Hph_4_Cl_',
 'Hse_Et_',
 'dHyp',
 'Hyp_Et_',
 'dI',
 'Me_dI',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'K',
 'dK',
 'meK',
 'Me_dK',
 'Lys_Ac_',
 'Lys_Cbz_',
 'Lys_iPr_',
 'Lys_Me_',
 'Me_Lys_Me_',
 'Lys_Me2_',
 'Lys_Tfa_',
 'aMeLeu

In [38]:
#Fingerprints models
#All fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)

df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 20188)
y_train shape:  (140,)
X_test shape:  (35, 20188)
y_test shape:  (35,)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008673 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3857
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 779
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1762,0.3367,0.4198,0.5497,0.7454,0.7389,0.2334,0.3804,0.4831,0.4947,0.7109,0.6534
DecisionTreeRegressor,0.2258,0.3627,0.4752,0.4231,0.7224,0.7098,0.2302,0.3519,0.4797,0.5017,0.7119,0.6432
RandomForestRegressor,0.1698,0.3325,0.4121,0.5660,0.7614,0.7589,0.2616,0.3871,0.5115,0.4336,0.6642,0.6237
GradientBoostingRegressor,0.1437,0.3063,0.3791,0.6328,0.7955,0.7854,0.2417,0.3607,0.4916,0.4768,0.6933,0.6528
AdaBoostRegressor,0.1901,0.3612,0.4360,0.5144,0.7288,0.7245,0.2498,0.3866,0.4998,0.4591,0.7064,0.6408
XGBRegressor,0.1715,0.3232,0.4141,0.5618,0.7564,0.7539,0.2774,0.3857,0.5267,0.3994,0.6437,0.5869
ExtraTreesRegressor,0.1696,0.3125,0.4118,0.5668,0.7544,0.7518,0.2173,0.3560,0.4661,0.5296,0.7332,0.6909
LinearRegression,1.5337,0.9278,1.2384,-2.9186,0.2158,0.2667,1.2601,0.8426,1.1225,-1.7282,0.1911,0.2083
KNeighborsRegressor,0.3059,0.4131,0.5531,0.2184,0.5539,0.5873,0.3399,0.3992,0.5830,0.2641,0.5414,0.5040
SVR,0.2676,0.4098,0.5173,0.3164,0.5692,0.5706,0.2574,0.3853,0.5074,0.4427,0.6790,0.6058


In [39]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.118430764063204, -5.404426144239926, -5.40...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.158976695555745, -6.158976695555745, -5.9...","[-6.233119834742088, -6.224751960887954, -5.96...","[0.10510979278589411, 0.11134490160274464, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -5.35, -5.35, -5.57, -5.28, -5.57, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.76, -5.76, -5.45, -6.89, -5.14, -4.9, -5....","[-5.9079999999999995, -5.833999999999999, -5.9...","[0.18126224096595522, 0.14800000000000005, 0.8..."
2,RandomForestRegressor,"[-6.129399999999996, -5.124499999999997, -5.19...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.978599999999997, -5.880999999999997, -5.8...","[-6.026450666666664, -5.9624337777777745, -5.7...","[0.06614428762920861, 0.08357604991448755, 0.2..."
3,GradientBoostingRegressor,"[-6.120900213478999, -5.188031044624854, -5.19...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.961852655741442, -5.880907539921563, -5.8...","[-6.015576250970187, -5.923763184776156, -5.68...","[0.12063188811722786, 0.15123217333207928, 0.1..."
4,AdaBoostRegressor,"[-6.339, -5.331749999999999, -5.34882352941176...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.84657894736842, -5.84657894736842, -5.620...","[-6.029847039473684, -5.973380807788701, -5.86...","[0.1263851853617288, 0.16500910814256103, 0.34..."
5,XGBRegressor,"[-6.2386203, -4.9505663, -5.15673, -5.737165, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.975163, -5.7613835, -5.4318714, -6.202573...","[-5.998128, -5.831582, -5.5173693, -6.164233, ...","[0.12114101, 0.14135839, 0.1470024, 0.25590083..."
6,ExtraTreesRegressor,"[-6.073249999999997, -4.977499999999994, -5.14...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.963299999999996, -5.7599999999999945, -5....","[-5.954899999999997, -5.813419999999995, -5.98...","[0.06687056153495324, 0.10684000000000182, 0.3..."
7,LinearRegression,"[-5.042923384673522, -5.064454148471654, -5.58...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.774830176950562, -5.759999999999896, -5.6...","[-6.251020281615923, -6.325495685264309, -6.58...","[1.6388689541185455, 1.1309913705287293, 2.148..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.13, -5.18, -5.96666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-5.886000000000001, -5.886000000000001, -6.23...","[0.048735111686659234, 0.048735111686659234, 0..."
9,SVR,"[-5.821362062347875, -5.448868360530859, -5.40...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9645288760085675, -5.8603796064982845, -5...","[-5.964073156162476, -5.888585008127544, -6.07...","[0.048858996330037666, 0.056827326288470606, 0..."


In [40]:
result_df.to_csv('results/Fingerprints/Results_All_fingerprints_fp_RRCK.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_fingerprints_fp_RRCK.csv')

In [41]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [42]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [43]:
#All fingerprints constant removal
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 3674)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 3674)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006463 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3857
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 779
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1762,0.3367,0.4198,0.5497,0.7454,0.7389,0.2334,0.3804,0.4831,0.4947,0.7109,0.6534
DecisionTreeRegressor,0.2129,0.3483,0.4614,0.4561,0.7299,0.7245,0.2307,0.3604,0.4803,0.5005,0.7111,0.6563
RandomForestRegressor,0.1679,0.3315,0.4098,0.5710,0.7655,0.7608,0.2647,0.3893,0.5145,0.4270,0.6587,0.6328
GradientBoostingRegressor,0.1438,0.3079,0.3792,0.6326,0.7954,0.7845,0.2393,0.3620,0.4892,0.4820,0.6964,0.6520
AdaBoostRegressor,0.1689,0.3427,0.4110,0.5684,0.7746,0.7637,0.2650,0.4029,0.5148,0.4263,0.6769,0.6101
XGBRegressor,0.1715,0.3232,0.4141,0.5618,0.7564,0.7539,0.2774,0.3857,0.5267,0.3994,0.6437,0.5869
ExtraTreesRegressor,0.1710,0.3117,0.4135,0.5632,0.7520,0.7592,0.2195,0.3597,0.4685,0.5247,0.7300,0.6767
LinearRegression,1.5337,0.9278,1.2384,-2.9186,0.2158,0.2667,1.2601,0.8426,1.1225,-1.7282,0.1911,0.2083
KNeighborsRegressor,0.3059,0.4131,0.5531,0.2184,0.5539,0.5873,0.3399,0.3992,0.5830,0.2641,0.5414,0.5040
SVR,0.2676,0.4098,0.5173,0.3164,0.5692,0.5707,0.2576,0.3853,0.5075,0.4424,0.6787,0.6058


In [44]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.118430764063204, -5.404426144239926, -5.40...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.158976695555745, -6.158976695555745, -5.9...","[-6.233119834742088, -6.224751960887954, -5.96...","[0.10510979278589411, 0.11134490160274464, 0.0..."
1,DecisionTreeRegressor,"[-6.46, -4.9, -5.14, -5.57, -5.28, -5.57, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -5.45, -6.89, -5.14, -5.35, -5...","[-6.055999999999999, -5.833999999999999, -5.60...","[0.14800000000000005, 0.14800000000000005, 0.6..."
2,RandomForestRegressor,"[-6.098166666666664, -5.132799999999998, -5.19...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.956566666666664, -5.900899999999996, -5.8...","[-6.012440666666665, -5.962910666666663, -5.77...","[0.06930586159309005, 0.09039733719884793, 0.2..."
3,GradientBoostingRegressor,"[-6.109863589386005, -5.188031044624854, -5.19...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.961852655741442, -5.880907539921563, -5.9...","[-6.016586452674893, -5.925694440705868, -5.72...","[0.12672902476912296, 0.15506964223044148, 0.1..."
4,AdaBoostRegressor,"[-6.163809523809523, -5.319230769230769, -5.31...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.956666666666667, -5.916666666666667, -5.7...","[-6.004415151515151, -5.988915151515151, -5.62...","[0.1390722670546798, 0.1413145374476073, 0.087..."
5,XGBRegressor,"[-6.2386203, -4.9505663, -5.15673, -5.737165, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.975163, -5.7613835, -5.4318714, -6.202573...","[-5.998128, -5.831582, -5.5173693, -6.164233, ...","[0.12114101, 0.14135839, 0.1470024, 0.25590083..."
6,ExtraTreesRegressor,"[-6.055699999999996, -5.010099999999996, -5.18...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.999199999999997, -5.7599999999999945, -5....","[-5.969339999999998, -5.814319999999995, -5.93...","[0.0668340512014646, 0.10864000000000154, 0.35..."
7,LinearRegression,"[-5.042923384727018, -5.064454148471581, -5.58...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.774830176950311, -5.759999999999977, -5.6...","[-6.251020281628909, -6.325495685262761, -6.58...","[1.6388689541212556, 1.1309913705258796, 2.148..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.13, -5.18, -5.96666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-5.886000000000001, -5.886000000000001, -6.23...","[0.048735111686659234, 0.048735111686659234, 0..."
9,SVR,"[-5.8213383453185825, -5.4496486957072925, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.964509891911432, -5.860273771517639, -5.9...","[-5.964090157989421, -5.888553855648092, -6.07...","[0.04886747861913691, 0.05684267844760165, 0.3..."


In [45]:
result_df.to_csv('results/Fingerprints/Results_All_const_rem_fingerprints_RRCK.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_const_rem_fingerprints_RRCK.csv')

In [46]:
#Morgan fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/morgan_fp_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/morgan_fp_test_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_fp

X_train shape:  (140, 2048)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 2048)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.073376 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 162
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 54
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3098,0.4637,0.5566,0.2086,0.4571,0.4499,0.2875,0.4327,0.5362,0.3776,0.6723,0.6577
DecisionTreeRegressor,0.3921,0.4804,0.6262,-0.0019,0.4484,0.4763,0.2489,0.3896,0.4989,0.4612,0.7016,0.6739
RandomForestRegressor,0.2480,0.3838,0.4980,0.3664,0.6081,0.6227,0.2062,0.3482,0.4541,0.5536,0.7636,0.6821
GradientBoostingRegressor,0.2468,0.3716,0.4968,0.3694,0.6205,0.6624,0.2047,0.3485,0.4524,0.5568,0.7579,0.6730
AdaBoostRegressor,0.2920,0.4497,0.5403,0.2540,0.5173,0.5344,0.2714,0.4235,0.5210,0.4124,0.7391,0.6445
XGBRegressor,0.2733,0.3775,0.5227,0.3018,0.5984,0.6103,0.1818,0.3302,0.4263,0.6065,0.7838,0.7375
ExtraTreesRegressor,0.3703,0.4676,0.6085,0.0538,0.4601,0.4853,0.2425,0.3795,0.4925,0.4749,0.7099,0.6743
LinearRegression,0.7457,0.6404,0.8635,-0.9052,0.4100,0.4314,0.2839,0.3978,0.5328,0.3853,0.6974,0.6744
KNeighborsRegressor,0.2616,0.3898,0.5115,0.3317,0.6015,0.6184,0.3138,0.4205,0.5602,0.3206,0.5866,0.5408
SVR,0.2562,0.3921,0.5062,0.3453,0.5912,0.6100,0.2363,0.3467,0.4861,0.4885,0.7022,0.6468


In [47]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.668977564378123, -5.567113333967301, -5.56...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.668977564378123, -5.772479882489096, -5.8...","[-5.682257350954872, -5.766077440621659, -5.92...","[0.0761619562739919, 0.04964598602470373, 0.12..."
1,DecisionTreeRegressor,"[-6.13, -5.125, -5.536666666666666, -5.92, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -7.0, -6.89, -5.4, -5.125, -5....","[-6.078, -5.758, -6.9319999999999995, -6.60199...","[0.10399999999999991, 0.003999999999999915, 0...."
2,RandomForestRegressor,"[-5.903341666666665, -5.1755797619047605, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.037824999999998, -5.808562499999997, -6.5...","[-6.035551666666665, -5.831984166666663, -6.55...","[0.07785068422592249, 0.03621248520116355, 0.1..."
3,GradientBoostingRegressor,"[-6.001086627228719, -5.475383582882642, -5.53...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.054067893802907, -5.8413194861651405, -6....","[-6.034251411883529, -5.795875294056339, -6.52...","[0.14067940454463798, 0.03661098069189575, 0.3..."
4,AdaBoostRegressor,"[-5.78642857142857, -5.7794285714285705, -5.65...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.78642857142857, -5.78642857142857, -6.287...","[-5.924063492063492, -5.8945891330891325, -6.3...","[0.13123614661318933, 0.1242398278787174, 0.21..."
5,XGBRegressor,"[-5.86746, -5.1272273, -5.5348864, -6.4032, -5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1281967, -5.760055, -6.370854, -6.8852115...","[-6.065816, -5.798028, -6.7683573, -6.6160593,...","[0.12412535, 0.07568306, 0.2528337, 0.53971887..."
6,ExtraTreesRegressor,"[-6.129999999999996, -5.125, -5.53666666666667...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.129999999999996, -5.7599999999999945, -7....","[-6.093879999999998, -5.757999999999996, -6.92...","[0.07223999999999685, 0.003999999999997783, 0...."
7,LinearRegression,"[-5.695037389975651, -5.124999999999997, -5.53...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.267584366559335, -5.760000000000002, -8.4...","[-5.784948189434206, -6.008567684860715, -6.94...","[0.636565351472568, 0.4971353697214327, 0.8996..."
8,KNeighborsRegressor,"[-5.88, -5.13, -5.536666666666666, -5.96666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-5.886000000000001, -5.886000000000001, -6.56...","[0.048735111686659234, 0.048735111686659234, 0..."
9,SVR,"[-5.823333693287642, -5.279314788483228, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.904751450240738, -5.859961114638675, -6.0...","[-5.922790112633282, -5.8688854258811745, -6.2...","[0.04331664847651881, 0.048802714036317286, 0...."


In [48]:
df_morgan_fp.to_csv('results/Fingerprints/Results_Morgan_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Morgan_fp_RRCK.csv')

In [49]:
#Morgan count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/count_morgan_fp_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/count_morgan_fp_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_count_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_count_fp

X_train shape:  (140, 2048)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 2048)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000963 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 433
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 69
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2281,0.3883,0.4776,0.4172,0.6502,0.6647,0.2609,0.3911,0.5108,0.4351,0.6762,0.6243
DecisionTreeRegressor,0.3531,0.4311,0.5942,0.0979,0.5240,0.5496,0.2218,0.3298,0.4710,0.5197,0.7228,0.6740
RandomForestRegressor,0.2277,0.3608,0.4772,0.4183,0.6490,0.6741,0.2179,0.3367,0.4668,0.5282,0.7411,0.6957
GradientBoostingRegressor,0.2093,0.3346,0.4575,0.4651,0.6845,0.7224,0.2033,0.3101,0.4509,0.5598,0.7528,0.6935
AdaBoostRegressor,0.2531,0.4015,0.5031,0.3533,0.5999,0.6340,0.2570,0.4052,0.5070,0.4435,0.7127,0.6681
XGBRegressor,0.2708,0.3859,0.5204,0.3080,0.5931,0.6300,0.1849,0.3079,0.4301,0.5996,0.7764,0.7243
ExtraTreesRegressor,0.2164,0.3436,0.4652,0.4471,0.6776,0.7008,0.1905,0.3184,0.4365,0.5875,0.7725,0.6999
LinearRegression,0.4337,0.4832,0.6586,-0.1081,0.4609,0.4828,0.2999,0.4007,0.5476,0.3507,0.6248,0.5193
KNeighborsRegressor,0.2804,0.4067,0.5295,0.2835,0.6004,0.5995,0.3533,0.4037,0.5944,0.2352,0.5479,0.4852
SVR,0.2347,0.3694,0.4844,0.4004,0.6359,0.6402,0.2317,0.3386,0.4813,0.4984,0.7084,0.6697


In [50]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.922522141195354, -5.5935248739988594, -5.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.922522141195354, -5.946628687820004, -6.0...","[-5.997052993772945, -5.98473875157397, -6.060...","[0.056157914632666, 0.06133135190096682, 0.051..."
1,DecisionTreeRegressor,"[-6.13, -5.125, -5.095, -6.66, -5.28, -5.57, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -5.125, -6.89, -6.89, -5.125, ...","[-6.0040000000000004, -5.833999999999999, -6.2...","[0.1581897594662815, 0.14800000000000005, 0.91..."
2,RandomForestRegressor,"[-6.0395999999999965, -5.14212976190476, -5.13...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.039099999999998, -5.866199999999996, -6.1...","[-6.023981666666666, -5.8699333333333295, -6.0...","[0.05244211104743033, 0.04629834170296521, 0.4..."
3,GradientBoostingRegressor,"[-6.070509375508913, -5.206046840267878, -5.20...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.177444612961619, -5.885582248950986, -5.5...","[-6.106352150688259, -5.883262285176884, -5.88...","[0.04521840001248983, 0.06096974867193552, 0.6..."
4,AdaBoostRegressor,"[-5.955666666666665, -5.47052631578947, -5.470...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9082758620689635, -6.003529411764706, -6....","[-6.05626136899499, -6.058938930042956, -6.085...","[0.0769432791118382, 0.13595146606049094, 0.39..."
5,XGBRegressor,"[-6.0215397, -5.124855, -5.0960374, -5.726697,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.128629, -5.762154, -6.479854, -6.8641295,...","[-6.0601487, -5.846196, -6.224263, -6.580139, ...","[0.08825825, 0.17103489, 0.65319747, 0.458769,..."
6,ExtraTreesRegressor,"[-6.0448999999999975, -5.125, -5.0950000000000...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.057599999999997, -5.7599999999999945, -6....","[-6.025639999999998, -5.800299999999996, -5.89...","[0.045952871509839406, 0.08060000000000259, 0...."
7,LinearRegression,"[-6.53933388003154, -6.078091429558869, -5.386...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.946130449582519, -5.760000000000001, -6.7...","[-5.607265512232686, -5.773330054699166, -6.41...","[0.6093543662520853, 0.02666010939832857, 0.67..."
8,KNeighborsRegressor,"[-5.88, -5.13, -5.03, -5.966666666666666, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -5.9...","[-5.886000000000001, -5.886000000000001, -6.06...","[0.048735111686659234, 0.048735111686659234, 0..."
9,SVR,"[-5.912529752122063, -5.449718578178742, -5.31...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.949382061169545, -5.860314500274301, -5.9...","[-5.9613122411927195, -5.877322254176326, -6.0...","[0.03875295439445824, 0.04103648691273359, 0.3..."


In [51]:
df_morgan_count_fp.to_csv('results/Fingerprints/Results_Count_Morgan_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Count_Morgan_fp_RRCK.csv')

In [52]:
#AtomPairs2d fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2D_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2D_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2D_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2D_fp

X_train shape:  (140, 780)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 780)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000616 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 7
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3576,0.5028,0.5980,0.0864,0.3050,0.1944,0.3738,0.5094,0.6114,0.1908,0.4550,0.4314
DecisionTreeRegressor,0.3568,0.5041,0.5973,0.0884,0.3480,0.2756,0.3908,0.5163,0.6252,0.1538,0.3952,0.3575
RandomForestRegressor,0.3484,0.4938,0.5902,0.1099,0.3515,0.2650,0.3818,0.5088,0.6179,0.1735,0.4179,0.3688
GradientBoostingRegressor,0.3479,0.4961,0.5898,0.1112,0.3579,0.2773,0.3972,0.5179,0.6302,0.1401,0.3778,0.3306
AdaBoostRegressor,0.3366,0.4848,0.5802,0.1399,0.3853,0.2960,0.3914,0.5100,0.6256,0.1526,0.3946,0.3193
XGBRegressor,0.3498,0.4962,0.5914,0.1063,0.3512,0.2762,0.3918,0.5173,0.6259,0.1518,0.3917,0.3575
ExtraTreesRegressor,0.3506,0.4966,0.5921,0.1042,0.3532,0.2750,0.3924,0.5174,0.6264,0.1505,0.3914,0.3575
LinearRegression,0.3548,0.4984,0.5957,0.0934,0.3402,0.2639,0.3684,0.5000,0.6069,0.2025,0.4531,0.4311
KNeighborsRegressor,0.6240,0.6451,0.7899,-0.5943,0.2314,0.2634,0.5794,0.5951,0.7612,-0.2545,0.4034,0.3544
SVR,0.3761,0.5091,0.6133,0.0390,0.2562,0.2296,0.3940,0.5210,0.6277,0.1470,0.3871,0.3598


In [53]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.695198516918082, -5.900689312889046, -5.90...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.695198516918082, -5.695198516918082, -5.9...","[-5.755362818437669, -5.755362818437669, -5.83...","[0.05795331415783911, 0.05795331415783911, 0.0..."
1,DecisionTreeRegressor,"[-5.919999999999999, -5.885119047619048, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.8...","[-6.1049999999999995, -6.1049999999999995, -5....","[0.10359107640675995, 0.10359107640675995, 0.0..."
2,RandomForestRegressor,"[-5.912557063492062, -5.888873484508981, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.912557063492063, -5.912557063492063, -5.8...","[-6.094198063492063, -6.094198063492063, -5.79...","[0.10170945943067007, 0.10170945943067007, 0.0..."
3,GradientBoostingRegressor,"[-5.9108565064383916, -5.8850524333718575, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9108565064383916, -5.9108565064383916, -5...","[-6.0381300323776745, -6.0381300323776745, -5....","[0.10407359340775493, 0.10407359340775493, 0.0..."
4,AdaBoostRegressor,"[-5.892424242424244, -5.892424242424244, -5.89...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.892424242424244, -5.892424242424244, -5.8...","[-5.913162828334573, -5.913162828334573, -5.84...","[0.05512956752653186, 0.05512956752653186, 0.0..."
5,XGBRegressor,"[-5.9198146, -5.8851843, -5.8851843, -5.660944...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9198146, -5.9198146, -5.8851843, -5.88518...","[-6.1040225, -6.1040225, -5.8063817, -5.806381...","[0.10318088, 0.10318088, 0.06586575, 0.0658657..."
6,ExtraTreesRegressor,"[-5.919999999999999, -5.885119047619051, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.8...","[-6.105000000000007, -6.105000000000007, -5.80...","[0.1035910764067622, 0.1035910764067622, 0.065..."
7,LinearRegression,"[-5.817320261437911, -5.885119047619049, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.817320261437911, -5.817320261437911, -5.8...","[-5.936737511211287, -5.936737511211287, -5.80...","[0.09614297245670222, 0.09614297245670222, 0.0..."
8,KNeighborsRegressor,"[-5.919999999999999, -6.483333333333333, -6.48...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-6.120666666666667, -6.120666666666667, -6.45...","[0.10822610077466974, 0.10822610077466974, 0.3..."
9,SVR,"[-5.859740594910672, -5.790485888254378, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.859740594910672, -5.859740594910672, -5.7...","[-5.863254326727651, -5.863254326727651, -5.68...","[0.06364051592695305, 0.06364051592695305, 0.0..."


In [54]:
df_AtomPairs2D_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_AtomPairs2D_fp_RRCK.csv')

In [55]:
#AtomPairs2d Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2DCount_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2DCount_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2DCount_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2DCount_fp

X_train shape:  (140, 780)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 780)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000576 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 700
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 48
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.27953838874178394


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1813,0.3599,0.4258,0.5367,0.7368,0.7139,0.2687,0.4049,0.5184,0.4182,0.6481,0.5977
DecisionTreeRegressor,0.2908,0.4161,0.5392,0.2570,0.6191,0.5940,0.3091,0.3896,0.5560,0.3308,0.6025,0.5841
RandomForestRegressor,0.1561,0.3169,0.3951,0.6011,0.7788,0.7589,0.2846,0.3994,0.5335,0.3838,0.6263,0.5472
GradientBoostingRegressor,0.1484,0.3061,0.3852,0.6209,0.7889,0.7757,0.2896,0.3936,0.5381,0.3731,0.6224,0.5626
AdaBoostRegressor,0.1692,0.3371,0.4114,0.5676,0.7646,0.7482,0.2835,0.4094,0.5325,0.3861,0.6348,0.5216
XGBRegressor,0.1663,0.3237,0.4078,0.5750,0.7669,0.7427,0.2960,0.4070,0.5440,0.3593,0.6148,0.5703
ExtraTreesRegressor,0.1467,0.2967,0.3830,0.6253,0.7912,0.7631,0.2725,0.3932,0.5220,0.4101,0.6528,0.5678
LinearRegression,0.8600,0.5746,0.9274,-1.1973,0.3507,0.4933,0.4286,0.4399,0.6547,0.0721,0.6633,0.6656
KNeighborsRegressor,0.1809,0.3299,0.4253,0.5378,0.7466,0.7339,0.2657,0.3640,0.5154,0.4248,0.6693,0.6261
SVR,0.2395,0.3911,0.4894,0.3880,0.6330,0.6054,0.2596,0.3845,0.5095,0.4381,0.6855,0.5899


In [56]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.2914597770111085, -5.487803210361487, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.2914597770111085, -6.2914597770111085, -5...","[-6.245406244707006, -6.245406244707006, -5.89...","[0.08007274765323033, 0.08007274765323033, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -5.35, -5.35, -5.74, -5.28, -5.57, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.76, -5.76, -5.45, -5.45, -5.45, -4.9, -5....","[-6.090999999999999, -5.964, -5.484, -6.038, -...","[0.42327768663136506, 0.4080000000000002, 0.33..."
2,RandomForestRegressor,"[-6.192699999999997, -5.084799999999994, -5.22...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.976499999999995, -5.905599999999996, -5.6...","[-6.05306433333333, -6.006218499999997, -5.679...","[0.11096373424757766, 0.12311469664210742, 0.1..."
3,GradientBoostingRegressor,"[-6.1415954869586935, -5.112767664376697, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.854327254901895, -5.844347548274246, -5.4...","[-5.975416617173691, -5.97043278025986, -5.596...","[0.22450472855590356, 0.2272630333551912, 0.15..."
4,AdaBoostRegressor,"[-6.13, -5.378666666666666, -5.411944444444446...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.953333333333333, -5.911071428571428, -5.6...","[-6.058634733893558, -6.013655080213904, -5.65...","[0.18239908934794916, 0.20355053073171517, 0.1..."
5,XGBRegressor,"[-6.129904, -4.9026637, -5.1165605, -5.8740454...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.760669, -5.760669, -5.435936, -5.7015457,...","[-5.8890924, -5.8880777, -5.4723425, -5.838728...","[0.25269336, 0.25319707, 0.1720278, 0.26947358..."
6,ExtraTreesRegressor,"[-6.129999999999996, -5.039199999999997, -5.15...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.856799999999994, -5.7599999999999945, -5....","[-5.939719999999995, -5.832169999999995, -5.78...","[0.11159419608564032, 0.14434000000000147, 0.1..."
7,LinearRegression,"[-6.050472179879418, -5.603884146185817, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.905453490482493, -6.387737085907904...","[-6.6318537654464365, -5.969139521754951, -6.4...","[1.8102263691216138, 0.12918699944254972, 0.08..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -5.5...","[-6.120666666666667, -6.1033333333333335, -5.7...","[0.10822610077466886, 0.10364469220477374, 0.2..."
9,SVR,"[-6.030103757754398, -5.796192007403309, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.961446722154777, -5.9493495522940565, -5....","[-6.022717129923865, -5.967616733838695, -5.75...","[0.06310786025514921, 0.06429298936637717, 0.1..."


In [57]:
df_AtomPairs2DCount_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_Count_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_df_AtomPairs2D_Count_fp_RRCK.csv')

In [58]:
#EState fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/EState_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/EState_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_estate_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_estate_fp

X_train shape:  (140, 79)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 79)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000150 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.1930028382971648


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3942,0.5207,0.6279,-0.0072,0.1037,0.0369,0.4039,0.5566,0.6355,0.1256,0.4172,0.4099
DecisionTreeRegressor,0.3714,0.5193,0.6095,0.0510,0.2968,0.1778,0.3501,0.4929,0.5917,0.2421,0.5343,0.4554
RandomForestRegressor,0.3752,0.5197,0.6125,0.0414,0.2576,0.1832,0.3544,0.5037,0.5953,0.2327,0.5401,0.4629
GradientBoostingRegressor,0.3613,0.5046,0.6011,0.0768,0.3131,0.2055,0.3498,0.4981,0.5915,0.2426,0.5510,0.4688
AdaBoostRegressor,0.3755,0.5094,0.6128,0.0407,0.2655,0.1839,0.3529,0.4995,0.5941,0.2359,0.5376,0.5006
XGBRegressor,0.3718,0.5163,0.6098,0.0500,0.2907,0.1803,0.3503,0.4937,0.5918,0.2417,0.5353,0.4554
ExtraTreesRegressor,0.3683,0.5145,0.6069,0.0589,0.2993,0.1772,0.3501,0.4929,0.5917,0.2421,0.5343,0.4554
LinearRegression,0.3941,0.5284,0.6278,-0.0070,0.2312,0.1658,0.3719,0.5228,0.6099,0.1948,0.4962,0.4231
KNeighborsRegressor,0.9583,0.8176,0.9789,-1.4486,0.0050,-0.0126,0.7705,0.6830,0.8778,-0.6682,0.3989,0.3621
SVR,0.4058,0.5233,0.6370,-0.0368,0.1546,0.1574,0.3957,0.5387,0.6290,0.1433,0.4159,0.3704


In [59]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.479668763580871, -5.858037043636908, -5.85...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.858037043636908, -5.858037043636908, -5.8...","[-5.789190297625636, -5.789190297625636, -5.78...","[0.05388297382205942, 0.05388297382205942, 0.0..."
1,DecisionTreeRegressor,"[-6.78, -5.889285714285714, -5.889285714285714...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.6775, -5.6775, -5.889285714285714, -5.889...","[-5.536066666666667, -5.536066666666667, -5.85...","[0.10324913774189326, 0.10324913774189326, 0.0..."
2,RandomForestRegressor,"[-5.924616760461756, -5.893018164077924, -5.89...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.676985119047617, -5.676985119047617, -5.8...","[-5.5275327914862915, -5.5275327914862915, -5....","[0.10341214496944241, 0.10341214496944241, 0.1..."
3,GradientBoostingRegressor,"[-6.537062837025527, -5.880812088670516, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.68228967533611, -5.68228967533611, -5.880...","[-5.553947953081605, -5.553947953081605, -5.85...","[0.09099509301450226, 0.09099509301450226, 0.0..."
4,AdaBoostRegressor,"[-6.78, -5.95909090909091, -5.95909090909091, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.774444444444442, -5.774444444444442, -5.9...","[-5.6605690690690675, -5.6605690690690675, -6....","[0.08413359926681212, 0.08413359926681212, 0.1..."
5,XGBRegressor,"[-6.765839, -5.889221, -5.889221, -6.0967054, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.6775293, -5.6775293, -5.889221, -5.889221...","[-5.5361834, -5.5361834, -5.8596187, -5.859618...","[0.10337337, 0.10337337, 0.094005905, 0.094005..."
6,ExtraTreesRegressor,"[-6.779999999999983, -5.889285714285723, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.6775000000000055, -5.6775000000000055, -5...","[-5.536066666666669, -5.536066666666669, -5.85...","[0.10324913774189559, 0.10324913774189559, 0.0..."
7,LinearRegression,"[-5.4075962411430565, -5.889285714285715, -5.8...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.909676346926325, -5.909676346926325, -5.8...","[-5.911837680120995, -5.911837680120995, -5.85...","[0.0428961779488848, 0.0428961779488848, 0.093..."
8,KNeighborsRegressor,"[-5.919999999999999, -6.483333333333333, -6.48...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-5.725999999999999, -5.725999999999999, -6.45...","[0.1633863859417636, 0.1633863859417636, 0.374..."
9,SVR,"[-5.823427135660978, -5.549755058313525, -5.54...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.8585440924111, -5.8585440924111, -5.54975...","[-5.796646378383518, -5.796646378383518, -5.59...","[0.056265332839844966, 0.056265332839844966, 0..."


In [60]:
df_estate_fp.to_csv('results/Fingerprints/Results_EState_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_EState_fp_RRCK.csv')

In [61]:
#Extended fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Extended_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Extended_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_extended_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_extended_fp

X_train shape:  (140, 1024)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1024)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002279 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 495
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 165
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2988,0.4553,0.5467,0.2365,0.4931,0.4878,0.2700,0.4119,0.5196,0.4155,0.6574,0.5603
DecisionTreeRegressor,0.3583,0.4563,0.5986,0.0846,0.4816,0.4970,0.3414,0.4211,0.5843,0.2610,0.5253,0.4753
RandomForestRegressor,0.2961,0.4277,0.5441,0.2435,0.5198,0.5248,0.2895,0.3935,0.5381,0.3731,0.6117,0.5835
GradientBoostingRegressor,0.3006,0.4305,0.5483,0.2319,0.5304,0.5434,0.2852,0.3908,0.5340,0.3826,0.6239,0.5691
AdaBoostRegressor,0.2987,0.4576,0.5465,0.2368,0.4912,0.4525,0.2995,0.4224,0.5472,0.3517,0.6056,0.5282
XGBRegressor,0.3283,0.4382,0.5730,0.1612,0.4956,0.5059,0.3229,0.4069,0.5682,0.3010,0.5539,0.5023
ExtraTreesRegressor,0.3353,0.4404,0.5790,0.1433,0.5039,0.5053,0.3371,0.4102,0.5806,0.2701,0.5342,0.5000
LinearRegression,0.5338,0.5625,0.7306,-0.3638,0.3471,0.3872,0.3174,0.4484,0.5634,0.3127,0.5820,0.5692
KNeighborsRegressor,0.3242,0.4381,0.5694,0.1716,0.5065,0.4888,0.2873,0.3795,0.5360,0.3779,0.6192,0.5462
SVR,0.3292,0.4618,0.5737,0.1590,0.4130,0.4271,0.3626,0.4812,0.6021,0.2151,0.4672,0.3542


In [62]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.887604998438236, -5.768251425371539, -5.76...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.976650830584715, -5.976650830584715, -6.0...","[-6.11101822192603, -6.11101822192603, -6.1149...","[0.0727921437387155, 0.0727921437387155, 0.149..."
1,DecisionTreeRegressor,"[-5.945, -5.641, -5.641, -5.92, -5.18, -5.485,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.945, -5.945, -6.466666666666666, -5.05, -...","[-5.945, -5.945, -6.289999999999999, -5.324, -...","[0.11700427342623007, 0.11700427342623007, 0.6..."
2,RandomForestRegressor,"[-5.714507499999997, -5.64630770807495, -5.646...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.964774166666665, -5.9503741666666645, -6....","[-5.953614357142855, -5.954314357142855, -6.24...","[0.08567833661846622, 0.08525725678802575, 0.4..."
3,GradientBoostingRegressor,"[-5.804923649234437, -5.639533733603696, -5.63...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.910368283041541, -5.942314345970233, -5.9...","[-5.925012070636469, -5.95963017082137, -6.084...","[0.038058135419404476, 0.07120890580993977, 0...."
4,AdaBoostRegressor,"[-5.669090909090908, -5.799333333333332, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.669090909090908, -5.669090909090908, -6.1...","[-5.830905483405483, -5.8335721500721505, -6.0...","[0.17411068475430463, 0.17648522497398206, 0.1..."
5,XGBRegressor,"[-5.949641, -5.6408443, -5.6408443, -6.1065826...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.944353, -5.944353, -5.9433365, -5.0547266...","[-5.9437985, -5.9450274, -6.063922, -5.167375,...","[0.118768446, 0.11667814, 0.63047534, 0.227186..."
6,ExtraTreesRegressor,"[-5.881510000000002, -5.641000000000004, -5.64...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9254700000000025, -5.945000000000002, -6....","[-5.939613999999999, -5.9449999999999985, -6.2...","[0.12007938700709693, 0.11700427342623064, 0.6..."
7,LinearRegression,"[-5.996413063844512, -5.57269916413936, -5.572...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.826065265953842, -5.944999999999997, -6.0...","[-5.6529592316324475, -5.945, -6.3135503922938...","[0.18710421937985816, 0.11700427342622867, 1.0..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.783333333333334, -5.78...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-6.0680000000000005, -6.0680000000000005, -6....","[0.12730715263138598, 0.12730715263138598, 0.2..."
9,SVR,"[-5.724450175641171, -5.549977730185798, -5.54...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.855524540443174, -5.86002484485694, -5.67...","[-5.934128154984027, -5.946641720949228, -5.68...","[0.06763648858236884, 0.072274140715583, 0.085..."


In [63]:
df_extended_fp.to_csv('results/Fingerprints/Results_Extended_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Extended_fp_RRCK.csv')

In [64]:
#Fingerprinter fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Fingerprinter_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Fingerprinter_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_fingerprinter_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_fingerprinter_fp

X_train shape:  (140, 1024)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1024)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002342 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 465
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 155
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3158,0.4676,0.5619,0.1932,0.4526,0.4293,0.2961,0.4331,0.5441,0.3590,0.6100,0.5163
DecisionTreeRegressor,0.3678,0.4768,0.6065,0.0602,0.4529,0.4773,0.3233,0.4037,0.5686,0.3000,0.5651,0.5297
RandomForestRegressor,0.3286,0.4551,0.5733,0.1603,0.4601,0.4904,0.2954,0.4059,0.5435,0.3604,0.6011,0.5403
GradientBoostingRegressor,0.3399,0.4709,0.5830,0.1317,0.4510,0.4636,0.2599,0.3800,0.5098,0.4374,0.6662,0.5880
AdaBoostRegressor,0.3073,0.4648,0.5544,0.2147,0.4672,0.4458,0.3129,0.4314,0.5594,0.3225,0.5756,0.5647
XGBRegressor,0.3711,0.4775,0.6092,0.0518,0.4178,0.4609,0.3134,0.4154,0.5598,0.3215,0.5711,0.5021
ExtraTreesRegressor,0.3540,0.4671,0.5950,0.0955,0.4634,0.4854,0.3282,0.4094,0.5729,0.2894,0.5544,0.5218
LinearRegression,0.5332,0.5795,0.7302,-0.3624,0.2896,0.3019,0.3090,0.4220,0.5559,0.3310,0.5796,0.5188
KNeighborsRegressor,0.3273,0.4369,0.5721,0.1638,0.4976,0.4851,0.3249,0.4087,0.5700,0.2967,0.5580,0.4758
SVR,0.3339,0.4641,0.5778,0.1469,0.3992,0.4319,0.3671,0.4843,0.6059,0.2052,0.4554,0.3374


In [65]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.70235297404548, -5.782614853836748, -5.782...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.86616288879865, -5.86616288879865, -6.142...","[-6.013603625965208, -6.013603625965208, -6.17...","[0.09458419597799618, 0.09458419597799618, 0.1..."
1,DecisionTreeRegressor,"[-5.87, -5.707692307692308, -5.707692307692308...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.945, -5.945, -6.466666666666666, -5.05, -...","[-5.945, -5.945, -6.289999999999999, -5.324, -...","[0.11700427342623007, 0.11700427342623007, 0.6..."
2,RandomForestRegressor,"[-5.793300753968254, -5.698927181051395, -5.69...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.93271583333333, -5.940540833333331, -6.10...","[-5.969149960317457, -5.971795626984124, -6.22...","[0.08991753101578487, 0.08803038039039898, 0.4..."
3,GradientBoostingRegressor,"[-5.738238709709905, -5.7082174230024325, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.941621644337885, -5.941621644337885, -6.0...","[-5.962726651476012, -5.962726651476012, -6.14...","[0.10898150765782189, 0.10898150765782189, 0.3..."
4,AdaBoostRegressor,"[-5.685333333333333, -5.921145833333333, -5.92...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.859166666666667, -5.859166666666667, -6.1...","[-5.935088018490754, -5.935088018490754, -6.10...","[0.09135100267193903, 0.09135100267193903, 0.1..."
5,XGBRegressor,"[-5.8263836, -5.7078595, -5.7078595, -5.771013...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9442387, -5.9442387, -6.029044, -5.054029...","[-5.9449315, -5.944852, -6.062493, -5.187687, ...","[0.116928704, 0.11681036, 0.6736006, 0.2657563..."
6,ExtraTreesRegressor,"[-5.892470000000004, -5.707692307692307, -5.70...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.945000000000002, -5.945000000000002, -6.4...","[-5.945119999999999, -5.9449999999999985, -6.2...","[0.11700451957082743, 0.11700427342623064, 0.6..."
7,LinearRegression,"[-5.687159456934946, -5.644222598117005, -5.64...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.936977894749374, -5.945000000000002, -5.9...","[-6.003522939201055, -5.945, -6.19194300645747...","[0.18616404677437742, 0.11700427342623007, 1.1..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.783333333333334, -5.78...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-6.0680000000000005, -6.0680000000000005, -6....","[0.12730715263138598, 0.12730715263138598, 0.2..."
9,SVR,"[-5.723772778900627, -5.550434535193182, -5.55...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.864903338443836, -5.859775732964175, -5.6...","[-5.95180758888544, -5.952917437412518, -5.709...","[0.07680368448832098, 0.07768123705746854, 0.0..."


In [66]:
df_fingerprinter_fp.to_csv('results/Fingerprints/Results_Fingerprinter_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Fingerprinter_fp_RRCK.csv')

In [67]:
#GraphOnly fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Graphonly_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Graphonly_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_graph_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_graph_fp

X_train shape:  (140, 1024)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1024)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001568 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 162
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 54
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3383,0.4801,0.5817,0.1355,0.3797,0.3646,0.3412,0.4852,0.5841,0.2613,0.5316,0.4069
DecisionTreeRegressor,0.3430,0.4810,0.5856,0.1237,0.4088,0.4190,0.3330,0.4505,0.5771,0.2790,0.5301,0.4562
RandomForestRegressor,0.3282,0.4699,0.5729,0.1614,0.4247,0.4372,0.3288,0.4638,0.5734,0.2882,0.5474,0.4473
GradientBoostingRegressor,0.3298,0.4737,0.5743,0.1573,0.4266,0.4219,0.3268,0.4562,0.5717,0.2924,0.5473,0.4316
AdaBoostRegressor,0.3570,0.4947,0.5975,0.0878,0.3395,0.3417,0.3520,0.4821,0.5933,0.2378,0.5066,0.4036
XGBRegressor,0.3414,0.4822,0.5843,0.1276,0.4126,0.4219,0.3503,0.4705,0.5918,0.2417,0.4919,0.4002
ExtraTreesRegressor,0.3488,0.4865,0.5906,0.1088,0.3920,0.4055,0.3169,0.4554,0.5629,0.3139,0.5705,0.4774
LinearRegression,0.3786,0.5120,0.6153,0.0327,0.3583,0.3731,0.3060,0.4322,0.5532,0.3375,0.5864,0.4843
KNeighborsRegressor,0.5436,0.5887,0.7373,-0.3888,0.2415,0.2503,0.3732,0.4629,0.6109,0.1921,0.5875,0.5252
SVR,0.3412,0.4684,0.5841,0.1283,0.3701,0.4436,0.3656,0.5067,0.6047,0.2084,0.4641,0.3691


In [68]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.434507168841772, -5.932508049979732, -5.93...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.806601759766931, -5.806601759766931, -5.9...","[-5.883021137180017, -5.883021137180017, -5.86...","[0.08812839535469225, 0.08812839535469225, 0.0..."
1,DecisionTreeRegressor,"[-5.945, -5.884318181818181, -5.88431818181818...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.945, -5.945, -5.884318181818181, -5.88431...","[-5.945, -5.945, -5.837959090909091, -5.837959...","[0.11700427342623007, 0.11700427342623007, 0.0..."
2,RandomForestRegressor,"[-5.862026635447884, -5.881923087246164, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.799803476800975, -5.915810833333332, -5.8...","[-5.951408302197801, -5.987754055555554, -5.81...","[0.10175019816175011, 0.07663857720237956, 0.0..."
3,GradientBoostingRegressor,"[-5.792494360635206, -5.885216903581173, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.761456043928442, -5.938733261567922, -5.8...","[-5.905120883755697, -5.990268987537998, -5.83...","[0.20196809846202204, 0.11123767747915936, 0.0..."
4,AdaBoostRegressor,"[-5.750000000000001, -5.925645161290322, -5.92...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.750000000000001, -5.892, -5.9256451612903...","[-5.9994238993710685, -6.110399656946827, -5.8...","[0.1723927263431963, 0.16888223631625912, 0.06..."
5,XGBRegressor,"[-5.930187, -5.884234, -5.884234, -6.6551266, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.605061, -5.9445887, -5.884234, -5.884234,...","[-5.8624253, -5.9464655, -5.837906, -5.837906,...","[0.20053646, 0.117168225, 0.067567304, 0.06756..."
6,ExtraTreesRegressor,"[-5.907650000000001, -5.884318181818182, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.914950000000001, -5.945000000000002, -5.8...","[-5.9017800000000005, -5.9449999999999985, -5....","[0.18155358602902966, 0.11700427342623064, 0.0..."
7,LinearRegression,"[-5.546087684673854, -5.884318181818182, -5.88...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.729399684406731, -5.945, -5.8843181818181...","[-5.740095074916411, -5.945, -5.83795909090909...","[0.19724733307327846, 0.11700427342622896, 0.0..."
8,KNeighborsRegressor,"[-5.920000000000001, -6.483333333333333, -6.48...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.4...","[-6.120666666666667, -6.120666666666667, -6.45...","[0.10822610077466907, 0.10822610077466907, 0.3..."
9,SVR,"[-5.74230504136997, -5.740056655440672, -5.740...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.848735374182268, -5.859936589363013, -5.7...","[-5.965287823560496, -5.9718413456858865, -5.6...","[0.09678420077971603, 0.0932038456344451, 0.09..."


In [69]:
df_graph_fp.to_csv('results/Fingerprints/Results_Graphonly_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Graphonly_fp_RRCK.csv')

In [70]:
#KlekotaRoth fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRoth_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRoth_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRoth_fp

X_train shape:  (140, 4860)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 4860)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044119 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 126
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 42
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2630,0.4201,0.5128,0.3281,0.5735,0.5542,0.2578,0.3961,0.5078,0.4418,0.6794,0.5388
DecisionTreeRegressor,0.3361,0.4292,0.5798,0.1412,0.5259,0.5459,0.1644,0.3099,0.4055,0.6440,0.8057,0.7818
RandomForestRegressor,0.2676,0.3943,0.5173,0.3162,0.5756,0.5906,0.1998,0.3427,0.4469,0.5675,0.7644,0.6947
GradientBoostingRegressor,0.2984,0.4176,0.5463,0.2375,0.5177,0.5731,0.1715,0.3132,0.4142,0.6286,0.8129,0.7710
AdaBoostRegressor,0.3291,0.4741,0.5737,0.1590,0.4141,0.4572,0.2632,0.4248,0.5130,0.4302,0.7219,0.7047
XGBRegressor,0.2970,0.3921,0.5449,0.2413,0.5503,0.5733,0.1596,0.3173,0.3995,0.6545,0.8134,0.8095
ExtraTreesRegressor,0.3277,0.4195,0.5724,0.1628,0.5273,0.5405,0.1629,0.3127,0.4036,0.6473,0.8082,0.7914
LinearRegression,0.7066,0.6192,0.8406,-0.8054,0.1469,0.2503,0.2329,0.3806,0.4826,0.4957,0.7131,0.6784
KNeighborsRegressor,0.2981,0.4047,0.5460,0.2383,0.5601,0.5646,0.3144,0.3926,0.5607,0.3194,0.5909,0.5555
SVR,0.2770,0.4087,0.5263,0.2922,0.5491,0.5742,0.2623,0.3887,0.5122,0.4320,0.6601,0.5494


In [71]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.032206602537977, -5.608531002489637, -5.60...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0558035841430575, -6.140082641571797, -6....","[-6.001900918497446, -6.056278771894654, -6.06...","[0.06052322238377586, 0.05709545233757811, 0.0..."
1,DecisionTreeRegressor,"[-6.0, -5.372000000000001, -5.372000000000001,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0, -5.76, -5.4, -6.89, -5.4, -5.372000000...","[-6.0, -5.808, -6.202, -6.601999999999999, -6....","[0.08221921916437779, 0.09600000000000009, 0.5..."
2,RandomForestRegressor,"[-5.8198388095238105, -5.3279622041847015, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.965175476190478, -5.792099999999997, -6.3...","[-5.990511190476191, -5.869332761904759, -6.44...","[0.038541170922200794, 0.056341742438345055, 0..."
3,GradientBoostingRegressor,"[-6.142967971809197, -5.435716251479076, -5.43...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.001520777333828, -5.8511799937893745, -6....","[-5.993293953928779, -5.907245164257104, -6.52...","[0.043491743650249146, 0.07183742839955909, 0...."
4,AdaBoostRegressor,"[-5.848749999999999, -5.772441860465114, -5.77...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.821666666666666, -5.821666666666666, -6.5...","[-6.038452344416028, -6.0352297642038, -6.3697...","[0.171079943529238, 0.13525511361328274, 0.271..."
5,XGBRegressor,"[-6.0693593, -5.3719254, -5.3719254, -6.139991...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.99973, -5.760573, -6.5383587, -6.888205, ...","[-5.9998307, -5.8259516, -6.52757, -6.6025367,...","[0.082288675, 0.130732, 0.26000646, 0.57369125..."
6,ExtraTreesRegressor,"[-6.0, -5.372000000000007, -5.372000000000007,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0, -5.7599999999999945, -5.39999999999999...","[-6.0, -5.807999999999995, -6.23983999999999, ...","[0.08221921916437555, 0.09600000000000222, 0.5..."
7,LinearRegression,"[-6.202425775371226, -5.423841298226746, -5.42...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.023419884549755, -5.71316023090049, -6.98...","[-5.978502202712998, -5.791507674265375, -6.73...","[0.05331095028097982, 0.05449972907019517, 0.5..."
8,KNeighborsRegressor,"[-5.53, -5.13, -5.13, -6.05, -5.22666666666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-5.885999999999999, -5.885999999999999, -6.46...","[0.04873511168665874, 0.04873511168665874, 0.1..."
9,SVR,"[-5.918178810628106, -5.250003421101468, -5.25...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.903708624693929, -5.872354882827788, -6.0...","[-5.905215699589116, -5.8908654781312535, -6.2...","[0.03757184441800641, 0.04279141323197192, 0.3..."


In [72]:
df_KlekotaRoth_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_fp_RRCK.csv')

In [73]:
#KlekotaRoth Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRothCount_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRothCount_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRothCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRothCount_fp

X_train shape:  (140, 4860)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 4860)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000774 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 969
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 116
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1928,0.3552,0.4391,0.5074,0.7142,0.6984,0.2165,0.3485,0.4653,0.5312,0.7460,0.6939
DecisionTreeRegressor,0.2303,0.3537,0.4799,0.4116,0.6971,0.7079,0.2912,0.3802,0.5396,0.3696,0.6251,0.5506
RandomForestRegressor,0.1873,0.3349,0.4327,0.5216,0.7278,0.7366,0.2223,0.3455,0.4714,0.5188,0.7352,0.7048
GradientBoostingRegressor,0.1913,0.3341,0.4374,0.5112,0.7166,0.7331,0.2225,0.3597,0.4717,0.5183,0.7269,0.6772
AdaBoostRegressor,0.2118,0.3749,0.4602,0.4589,0.6906,0.7001,0.2465,0.3900,0.4965,0.4664,0.7318,0.6881
XGBRegressor,0.1956,0.3372,0.4423,0.5002,0.7192,0.7274,0.2147,0.3365,0.4634,0.5351,0.7370,0.6925
ExtraTreesRegressor,0.1891,0.3263,0.4349,0.5168,0.7255,0.7228,0.1989,0.3336,0.4460,0.5693,0.7598,0.6995
LinearRegression,0.6907,0.5906,0.8311,-0.7647,0.3467,0.4408,0.4588,0.4982,0.6773,0.0067,0.4734,0.4648
KNeighborsRegressor,0.2855,0.3850,0.5343,0.2705,0.6020,0.6087,0.3288,0.3693,0.5734,0.2881,0.5833,0.5234
SVR,0.2057,0.3444,0.4536,0.4743,0.6908,0.7166,0.2511,0.3649,0.5011,0.4564,0.6776,0.6175


In [74]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.119184184529056, -5.445938327321007, -5.44...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.119184184529056, -6.119184184529056, -5.9...","[-6.165766217927536, -6.138495180555687, -6.08...","[0.06329217006280465, 0.1017692309288973, 0.17..."
1,DecisionTreeRegressor,"[-6.13, -5.109999999999999, -5.109999999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.76, -5.76, -6.89, -6.89, -5.4, -5.1099999...","[-5.856, -5.781999999999999, -5.78049999999999...","[0.14347125147568768, 0.044000000000000136, 0...."
2,RandomForestRegressor,"[-5.979042857142854, -5.135686904761904, -5.13...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.937829999999996, -5.880664285714281, -6.0...","[-5.98936022222222, -5.930651603174601, -6.008...","[0.061768374625704586, 0.1102501104232144, 0.2..."
3,GradientBoostingRegressor,"[-6.031519745437267, -5.180342286107264, -5.18...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0531023782389335, -5.883920085519724, -5....","[-6.019779120187602, -5.914163290828158, -5.75...","[0.08133303565500677, 0.1338181225847639, 0.26..."
4,AdaBoostRegressor,"[-5.9832, -5.415384615384615, -5.4153846153846...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.926739130434782, -5.823, -5.7346666666666...","[-5.981555233494364, -5.908724031007752, -5.75...","[0.08718876718964787, 0.1187859027075617, 0.10..."
5,XGBRegressor,"[-5.977397, -5.109996, -5.109996, -6.114418, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0495257, -5.7620997, -5.3948736, -6.14841...","[-6.0764837, -5.834122, -6.0061407, -6.332, -6...","[0.050771978, 0.14621712, 0.45851794, 0.229434..."
6,ExtraTreesRegressor,"[-6.060699999999997, -5.11000000000001, -5.110...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.881099999999999, -5.7599999999999945, -6....","[-5.884079999999999, -5.800279999999995, -5.97...","[0.06191056129611496, 0.08056000000000303, 0.2..."
7,LinearRegression,"[-5.933785040263709, -5.798734547869986, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.098019388172929, -5.764753130498633, -6.6...","[-6.271595430807042, -5.7264482767396245, -6.5...","[0.4098598802350503, 0.047786814328427074, 0.5..."
8,KNeighborsRegressor,"[-5.816666666666666, -5.13, -5.13, -6.05, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-5.885999999999999, -5.885999999999999, -6.23...","[0.04873511168665874, 0.04873511168665874, 0.2..."
9,SVR,"[-6.022714679708261, -5.396799940519092, -5.39...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.944043359663352, -5.860161164314112, -6.1...","[-5.959339460388283, -5.881834411695735, -6.18...","[0.050926386833846225, 0.05135567625417608, 0...."


In [75]:
df_KlekotaRothCount_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_Count_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_Count_fp_RRCK.csv')

In [76]:
#MACCS fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/MACCS_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/MACCS_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_MACCS_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_MACCS_fp

X_train shape:  (140, 166)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 166)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000557 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 51
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 17
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.08186128105925083


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2667,0.4167,0.5164,0.3186,0.5653,0.5702,0.2746,0.4187,0.5240,0.4054,0.6559,0.5325
DecisionTreeRegressor,0.2536,0.3736,0.5035,0.3522,0.6319,0.6182,0.2902,0.4147,0.5387,0.3716,0.6315,0.6188
RandomForestRegressor,0.2374,0.3676,0.4872,0.3935,0.6389,0.6445,0.2557,0.3670,0.5057,0.4464,0.6694,0.6501
GradientBoostingRegressor,0.2279,0.3603,0.4774,0.4177,0.6519,0.6698,0.2492,0.3646,0.4992,0.4605,0.6798,0.6525
AdaBoostRegressor,0.2628,0.4241,0.5126,0.3286,0.5804,0.5922,0.2643,0.4106,0.5141,0.4279,0.6802,0.6494
XGBRegressor,0.2511,0.3755,0.5011,0.3584,0.6363,0.6243,0.2779,0.4033,0.5272,0.3983,0.6461,0.6425
ExtraTreesRegressor,0.2461,0.3672,0.4961,0.3711,0.6423,0.6271,0.2981,0.4118,0.5460,0.3546,0.6176,0.6145
LinearRegression,0.3650,0.4876,0.6041,0.0675,0.5068,0.5089,0.3858,0.4711,0.6212,0.1647,0.5152,0.4812
KNeighborsRegressor,0.2663,0.3834,0.5161,0.3196,0.5979,0.5742,0.2720,0.3961,0.5216,0.4111,0.6506,0.6409
SVR,0.2778,0.4123,0.5271,0.2902,0.5483,0.5611,0.2567,0.4064,0.5067,0.4442,0.6818,0.6252


In [77]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.66890047652899, -5.479628017319369, -5.479...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.7882128395421235, -5.906141644390784, -6....","[-5.841090007793424, -6.033045040432856, -5.94...","[0.04209681820376567, 0.1442144356273101, 0.21..."
1,DecisionTreeRegressor,"[-5.92, -5.5840000000000005, -5.53666666666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0, -5.76, -6.585, -6.585, -5.4, -5.584000...","[-6.0, -5.808, -6.557333333333332, -6.55733333...","[0.08221921916437779, 0.09600000000000009, 0.2..."
2,RandomForestRegressor,"[-5.899367619047618, -5.57683163364413, -5.488...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.952544761904762, -5.805469999999996, -6.6...","[-6.0095731428571435, -5.861368992063488, -6.5...","[0.06768453649675005, 0.0751197344456371, 0.30..."
3,GradientBoostingRegressor,"[-6.061031529588029, -5.62272607904561, -5.525...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.956099655070239, -5.815879060928144, -6.4...","[-5.997950890824354, -5.872184410794991, -6.46...","[0.05550864386637247, 0.05206722121634773, 0.2..."
4,AdaBoostRegressor,"[-5.808461538461539, -5.797441860465116, -5.67...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.874705882352941, -5.808461538461539, -6.3...","[-5.977874509803922, -5.7490097521185906, -6.3...","[0.10082706471025175, 0.05914376974785046, 0.1..."
5,XGBRegressor,"[-5.972935, -5.5846395, -5.536237, -5.830258, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9995956, -5.761741, -6.5845485, -6.584548...","[-6.000204, -5.7963877, -6.557092, -6.557092, ...","[0.08186429, 0.07101392, 0.25349748, 0.2534974..."
6,ExtraTreesRegressor,"[-5.995999999999999, -5.583999999999998, -5.53...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0, -5.7599999999999945, -6.58500000000000...","[-6.0, -5.807999999999995, -6.557333333333335,...","[0.08221921916437555, 0.09600000000000222, 0.2..."
7,LinearRegression,"[-5.739895847101699, -5.755385267037699, -5.35...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.981605483786856, -5.796789032426291, -6.3...","[-5.955910986126396, -5.8472826451746265, -6.3...","[0.05115145677113308, 0.06179689092431438, 0.1..."
8,KNeighborsRegressor,"[-5.916666666666667, -5.75, -5.536666666666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-6.068, -6.068, -6.572, -6.572, -5.6186666666...","[0.1273071526313868, 0.1273071526313868, 0.196..."
9,SVR,"[-5.798152973861482, -5.450017645793669, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.969899122083263, -5.860028456185003, -5.9...","[-5.996300931727073, -5.88108683550602, -6.016...","[0.028547376936454524, 0.04216919752634165, 0...."


In [78]:
df_MACCS_fp.to_csv('results/Fingerprints/Results_MACCS_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_MACCS_fp_RRCK.csv')

In [79]:
#PubChem fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/PubChem_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/PubChem_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_PubChem_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_PubChem_fp

X_train shape:  (140, 881)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 881)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000587 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 108
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 36
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2854,0.4491,0.5342,0.2708,0.5210,0.4884,0.3246,0.4477,0.5698,0.2971,0.5460,0.5490
DecisionTreeRegressor,0.3110,0.4443,0.5577,0.2053,0.5412,0.5173,0.3998,0.4943,0.6323,0.1344,0.4444,0.3726
RandomForestRegressor,0.2815,0.4237,0.5306,0.2807,0.5518,0.5500,0.3410,0.4576,0.5840,0.2616,0.5266,0.5118
GradientBoostingRegressor,0.2707,0.4196,0.5202,0.3085,0.5671,0.5649,0.3836,0.4888,0.6194,0.1694,0.4493,0.3931
AdaBoostRegressor,0.2878,0.4514,0.5365,0.2647,0.5147,0.4965,0.3213,0.4435,0.5668,0.3043,0.5529,0.5364
XGBRegressor,0.2850,0.4226,0.5338,0.2719,0.5731,0.5462,0.3830,0.4850,0.6189,0.1708,0.4701,0.4241
ExtraTreesRegressor,0.3053,0.4403,0.5526,0.2199,0.5467,0.5252,0.3923,0.4908,0.6264,0.1506,0.4555,0.3968
LinearRegression,0.3652,0.4947,0.6044,0.0668,0.4558,0.4313,0.4013,0.4976,0.6335,0.1312,0.4624,0.3900
KNeighborsRegressor,0.3641,0.4744,0.6034,0.0697,0.4495,0.4055,0.3244,0.4503,0.5696,0.2977,0.5623,0.5347
SVR,0.3172,0.4518,0.5632,0.1896,0.4660,0.4351,0.3618,0.4892,0.6015,0.2167,0.4713,0.3756


In [80]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.877593342270295, -5.854054651133188, -5.85...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.877593342270295, -5.877593342270295, -6.2...","[-5.894962576947178, -5.894962576947178, -6.17...","[0.05817567422263721, 0.05817567422263721, 0.1..."
1,DecisionTreeRegressor,"[-5.84, -5.6433333333333335, -5.64333333333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.3...","[-5.92, -5.92, -6.388, -6.388, -6.388, -5.5586...","[0.06008327554319941, 0.06008327554319941, 0.1..."
2,RandomForestRegressor,"[-5.899385476190471, -5.6452388347763325, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9193292857142845, -5.9193292857142845, -6...","[-5.933050246031746, -5.933050246031746, -6.37...","[0.050285809501475105, 0.050285809501475105, 0..."
3,GradientBoostingRegressor,"[-6.09121592034307, -5.693554898144942, -5.693...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.932963983091939, -5.932963983091939, -6.3...","[-5.919075863414546, -5.919075863414546, -6.34...","[0.058425851938594994, 0.058425851938594994, 0..."
4,AdaBoostRegressor,"[-5.838333333333334, -5.867555555555556, -5.86...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.867555555555556, -5.867555555555556, -6.0...","[-5.856794246031745, -5.856794246031745, -6.23...","[0.10167386090071227, 0.10167386090071227, 0.1..."
5,XGBRegressor,"[-6.2667603, -5.6434627, -5.6434627, -6.217673...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919853, -5.919853, -6.347627, -6.347627, ...","[-5.9201403, -5.9201403, -6.387726, -6.387726,...","[0.0601316, 0.0601316, 0.18851878, 0.18851878,..."
6,ExtraTreesRegressor,"[-6.109999999999997, -5.643333333333327, -5.64...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.3...","[-5.920000000000001, -5.920000000000001, -6.38...","[0.06008327554319769, 0.06008327554319769, 0.1..."
7,LinearRegression,"[-6.119077516718318, -5.74457307056692, -5.744...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.3...","[-5.92, -5.92, -6.345739084354158, -6.34573908...","[0.06008327554320103, 0.06008327554320103, 0.1..."
8,KNeighborsRegressor,"[-5.919999999999999, -5.68, -5.68, -6.10666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-5.8759999999999994, -5.8759999999999994, -6....","[0.053806236730615936, 0.053806236730615936, 0..."
9,SVR,"[-5.741234290239531, -5.566440056978002, -5.56...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.859638483595333, -5.859638483595333, -5.6...","[-5.86774387875923, -5.86774387875923, -5.7658...","[0.05945012563266775, 0.05945012563266775, 0.2..."


In [81]:
df_PubChem_fp.to_csv('results/Fingerprints/Results_PubChem_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_PubChem_fp_RRCK.csv')

In [82]:
#Substructure fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Substructure_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Substructure_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_Substructure_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_Substructure_fp

X_train shape:  (140, 307)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 307)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 2
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.19812516384040335


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3753,0.5195,0.6126,0.0410,0.2252,0.1510,0.3730,0.5228,0.6108,0.1924,0.4693,0.4044
DecisionTreeRegressor,0.3586,0.5004,0.5988,0.0839,0.3357,0.3175,0.3301,0.4634,0.5745,0.2854,0.5362,0.4906
RandomForestRegressor,0.3599,0.5026,0.5999,0.0806,0.3198,0.2992,0.3247,0.4596,0.5699,0.2969,0.5490,0.5140
GradientBoostingRegressor,0.3520,0.4980,0.5933,0.1005,0.3414,0.3211,0.3295,0.4634,0.5740,0.2867,0.5401,0.5088
AdaBoostRegressor,0.3645,0.5091,0.6037,0.0687,0.2766,0.2087,0.3417,0.4793,0.5846,0.2601,0.5352,0.5473
XGBRegressor,0.3616,0.5044,0.6014,0.0760,0.3291,0.3178,0.3252,0.4604,0.5703,0.2959,0.5476,0.5002
ExtraTreesRegressor,0.3571,0.4987,0.5976,0.0877,0.3378,0.3175,0.3339,0.4666,0.5778,0.2771,0.5280,0.4664
LinearRegression,0.3839,0.5246,0.6196,0.0190,0.2901,0.2570,0.3305,0.4695,0.5749,0.2844,0.5396,0.5099
KNeighborsRegressor,0.7005,0.6530,0.8369,-0.7897,-0.1404,-0.0951,0.5544,0.5830,0.7446,-0.2003,0.1830,0.2453
SVR,0.3760,0.4993,0.6132,0.0394,0.2703,0.3158,0.3650,0.5057,0.6042,0.2097,0.4630,0.4100


In [83]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.392253501766805, -5.84593362675211, -5.845...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.392253501766805, -5.392253501766805, -6.0...","[-5.41604212877003, -5.41604212877003, -6.0140...","[0.05233029593950212, 0.05233029593950212, 0.0..."
1,DecisionTreeRegressor,"[-5.3462499999999995, -5.789342105263157, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.920000000000001, -6.3...","[-5.92, -5.92, -6.350666666666667, -6.35066666...","[0.06008327554319941, 0.06008327554319941, 0.1..."
2,RandomForestRegressor,"[-5.576601396936396, -5.793422752387, -5.79342...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.896313730158729, -5.896313730158729, -6.3...","[-5.891305888888889, -5.891305888888889, -6.33...","[0.06767791457207359, 0.06767791457207359, 0.1..."
3,GradientBoostingRegressor,"[-5.621563623811183, -5.786995854272565, -5.78...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.895919080914084, -5.895919080914084, -6.3...","[-5.876878909011817, -5.876878909011817, -6.32...","[0.06725422210253314, 0.06725422210253314, 0.1..."
4,AdaBoostRegressor,"[-5.475686274509804, -5.799615384615385, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.475686274509804, -5.475686274509804, -5.9...","[-5.528734420893864, -5.528734420893864, -6.20...","[0.13117757875520122, 0.13117757875520122, 0.2..."
5,XGBRegressor,"[-5.4442263, -5.789339, -5.789339, -5.926975, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919818, -5.919818, -6.319899, -6.319899, ...","[-5.9197817, -5.9197817, -6.350275, -6.350275,...","[0.05998858, 0.05998858, 0.16119665, 0.1611966..."
6,ExtraTreesRegressor,"[-5.346250000000001, -5.7893421052631435, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.3...","[-5.920000000000001, -5.920000000000001, -6.35...","[0.06008327554319769, 0.06008327554319769, 0.1..."
7,LinearRegression,"[-5.390909090909091, -5.7875889121338915, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.92, -5.92, -6.331103556485357, -6.3311035...","[-5.92, -5.92, -6.269946846906746, -6.26994684...","[0.0600832755431984, 0.0600832755431984, 0.165..."
8,KNeighborsRegressor,"[-5.53, -5.68, -5.68, -6.1066666666666665, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -6.4...","[-5.801999999999999, -5.801999999999999, -6.46...","[0.10434132024807354, 0.10434132024807354, 0.3..."
9,SVR,"[-5.842015406591075, -5.740567348139334, -5.74...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.859981648218187, -5.859981648218187, -6.0...","[-5.8642021640691375, -5.8642021640691375, -6....","[0.06351279238616393, 0.06351279238616393, 0.2..."


In [84]:
df_Substructure_fp.to_csv('results/Fingerprints/Results_Substructure_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_fp_RRCK.csv')

In [85]:
#Substructure Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/SubstructureCount_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/SubstructureCount_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_SubstructureCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_SubstructureCount_fp

X_train shape:  (140, 307)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 307)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012267 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 156
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 13
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.56648659495803


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2519,0.4179,0.5019,0.3564,0.5980,0.5895,0.2610,0.4002,0.5109,0.4350,0.6701,0.6389
DecisionTreeRegressor,0.2679,0.4049,0.5176,0.3155,0.6388,0.6417,0.2044,0.3314,0.4521,0.5574,0.7473,0.6883
RandomForestRegressor,0.1881,0.3468,0.4336,0.5195,0.7220,0.7178,0.2213,0.3659,0.4704,0.5209,0.7338,0.6915
GradientBoostingRegressor,0.1932,0.3610,0.4396,0.5063,0.7131,0.7113,0.2186,0.3534,0.4676,0.5266,0.7272,0.6699
AdaBoostRegressor,0.2372,0.4185,0.4871,0.3939,0.6325,0.6307,0.2302,0.3914,0.4798,0.5016,0.7371,0.6960
XGBRegressor,0.2197,0.3667,0.4687,0.4386,0.6890,0.6801,0.2094,0.3584,0.4576,0.5466,0.7417,0.7265
ExtraTreesRegressor,0.1910,0.3366,0.4370,0.5120,0.7245,0.7160,0.2192,0.3683,0.4682,0.5254,0.7296,0.6841
LinearRegression,0.2802,0.4206,0.5293,0.2842,0.5843,0.6112,0.2310,0.3561,0.4806,0.4999,0.7265,0.7166
KNeighborsRegressor,0.2816,0.4119,0.5307,0.2805,0.5945,0.5755,0.2835,0.3741,0.5324,0.3862,0.6314,0.5662
SVR,0.2897,0.4297,0.5383,0.2598,0.5193,0.5305,0.2867,0.4113,0.5354,0.3793,0.6211,0.5371


In [86]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.999757002174316, -5.676379437470554, -5.67...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.999757002174316, -5.999757002174316, -6.0...","[-6.068173180129753, -6.068173180129753, -5.92...","[0.10577423029034855, 0.10577423029034855, 0.1..."
1,DecisionTreeRegressor,"[-6.13, -5.109999999999999, -5.109999999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -7.0, -6.89, -5.45, -5.1099999...","[-6.004, -5.781999999999999, -6.32, -6.7959999...","[0.1581897594662815, 0.044000000000000136, 0.7..."
2,RandomForestRegressor,"[-6.046199999999998, -5.13455357142857, -5.134...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.980599999999998, -5.875999999999996, -6.2...","[-6.0573333333333315, -5.991655999999997, -6.0...","[0.0882612208793363, 0.12889941715927394, 0.19..."
3,GradientBoostingRegressor,"[-6.01324760653106, -5.1670732059186175, -5.16...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.987912060259569, -5.8906148925104675, -6....","[-6.0352799913597, -5.947530303843656, -6.3566...","[0.07042532865444197, 0.09308287206062857, 0.2..."
4,AdaBoostRegressor,"[-5.9590000000000005, -5.609117647058822, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.85, -5.85, -6.305, -6.474385964912275, -5...","[-6.043324786324787, -6.041908045977012, -6.26...","[0.10282810182933298, 0.10349451329389163, 0.2..."
5,XGBRegressor,"[-6.233497, -5.1101074, -5.1101074, -5.54251, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9703326, -5.7627125, -5.982992, -6.406984...","[-5.9373083, -5.806074, -5.8542604, -6.2893753...","[0.07368349, 0.08783629, 0.3148697, 0.28490713..."
6,ExtraTreesRegressor,"[-6.1092999999999975, -5.11000000000001, -5.11...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9582, -5.7599999999999945, -6.04970000000...","[-5.930049999999997, -5.809289999999995, -5.89...","[0.07279270567852272, 0.09765762847827393, 0.1..."
7,LinearRegression,"[-5.779444867811629, -5.577945741762819, -5.57...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.230033885586894, -5.974541584837993, -6.3...","[-6.024921972473841, -5.986300656683083, -6.49...","[0.1348985969966482, 0.08900741753750842, 0.21..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.13, -5.13, -6.10666666...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-5.885999999999999, -5.885999999999999, -6.13...","[0.04873511168665874, 0.04873511168665874, 0.2..."
9,SVR,"[-5.890050276765128, -5.6668259528163185, -5.6...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.897794974496423, -5.860261852847329, -6.1...","[-5.941186750641086, -5.887600651315986, -6.12...","[0.051861414498372116, 0.059114133805871195, 0..."


In [87]:
df_SubstructureCount_fp.to_csv('results/Fingerprints/Results_Substructure_Count_fp_RRCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_Count_fp_RRCK.csv')

In [88]:
#Descriptors models
#2d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2drdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2drdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 217)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 217)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002299 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2783
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 114
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.5920559106401457


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1735,0.3398,0.4165,0.5567,0.7499,0.7387,0.2304,0.3509,0.4800,0.5011,0.7128,0.6504
DecisionTreeRegressor,0.3370,0.4306,0.5805,0.1389,0.5447,0.5540,0.1997,0.3239,0.4469,0.5676,0.7536,0.7149
RandomForestRegressor,0.1590,0.3107,0.3988,0.5937,0.7759,0.7606,0.2060,0.3279,0.4539,0.5540,0.7536,0.7057
GradientBoostingRegressor,0.1430,0.2936,0.3782,0.6345,0.7977,0.7864,0.2105,0.3379,0.4588,0.5442,0.7430,0.6891
AdaBoostRegressor,0.1725,0.3371,0.4153,0.5593,0.7599,0.7790,0.2263,0.3669,0.4757,0.5101,0.7411,0.6900
XGBRegressor,0.1981,0.3407,0.4451,0.4939,0.7166,0.7007,0.1974,0.3241,0.4442,0.5727,0.7585,0.6823
ExtraTreesRegressor,0.1351,0.2833,0.3676,0.6548,0.8118,0.7957,0.2119,0.3539,0.4603,0.5413,0.7434,0.6841
LinearRegression,3.0287,1.1523,1.7403,-6.7383,0.2753,0.3328,0.7967,0.6228,0.8926,-0.7249,0.4003,0.4121
KNeighborsRegressor,0.1758,0.3169,0.4193,0.5507,0.7774,0.7641,0.3123,0.3862,0.5588,0.3239,0.6012,0.5096
SVR,0.1452,0.3001,0.3810,0.6290,0.8037,0.7987,0.2491,0.3809,0.4991,0.4606,0.6843,0.6148


In [89]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.807090876443247, -5.471560088600416, -5.31...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1572334966963895, -6.18520858555811, -5.7...","[-6.176264485734626, -6.17290609909846, -5.921...","[0.056074078426976134, 0.033321083270437236, 0..."
1,DecisionTreeRegressor,"[-5.87, -4.9, -4.9, -5.92, -5.28, -5.57, -6.49...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.12, -5.76, -6.78, -6.78, -6.89, -5.05, -5...","[-6.314, -5.833999999999999, -6.118, -6.406000...","[0.40316745900432016, 0.14800000000000005, 0.7..."
2,RandomForestRegressor,"[-5.989999999999996, -5.100549999999995, -5.11...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0521916666666655, -5.8623999999999965, -5...","[-6.147361666666664, -5.981779999999996, -5.94...","[0.06902321360721059, 0.11551653388151893, 0.2..."
3,GradientBoostingRegressor,"[-5.922519579091258, -5.163547569997099, -5.09...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.01535503819749, -5.8240411892230055, -5.7...","[-6.289525646473161, -5.864125113212623, -5.77...","[0.2435795942913108, 0.13445065926404548, 0.35..."
4,AdaBoostRegressor,"[-5.994193548387098, -5.332857142857142, -5.33...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -6.002500000000001, -5.71111111111111...","[-6.125874474474474, -6.039156976744185, -5.91...","[0.09621389128644563, 0.12303721202765115, 0.3..."
5,XGBRegressor,"[-6.1582794, -5.104896, -5.0736814, -5.6518064...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.133024, -5.760761, -6.377087, -6.4075317,...","[-6.2894974, -5.9168625, -6.0944095, -6.314816...","[0.17454444, 0.3123131, 0.50034726, 0.35522482..."
6,ExtraTreesRegressor,"[-6.013999999999997, -5.085999999999997, -5.06...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.045999999999997, -5.7599999999999945, -5....","[-6.032094999999998, -5.824619999999995, -5.79...","[0.0638337379447578, 0.12924000000000183, 0.17..."
7,LinearRegression,"[-4.0, -5.203373168615531, -5.299963383921, -1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.59313520551142, -5.743502788625381, -4.0,...","[-5.706151262216137, -5.406737494850736, -5.09...","[1.5904517403924785, 0.7034134687902324, 1.366..."
8,KNeighborsRegressor,"[-5.919999999999999, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-5.885999999999999, -5.885999999999999, -5.73...","[0.04873511168665874, 0.04873511168665874, 0.2..."
9,SVR,"[-6.1795077482878025, -5.242626869366737, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.057675239869171, -5.859497832877092, -5.6...","[-6.06250371184444, -5.884599535267421, -5.728...","[0.03231470673983675, 0.0494638504750396, 0.13..."


In [90]:
result_df.to_csv('results/Descriptors/Results_2d_RDKit_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_RDKit_desc_RRCK.csv')

In [91]:
#2d Mordred descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df , prediction_df= train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1430)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1430)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38372
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1109
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1255,0.2832,0.3543,0.6793,0.8302,0.8177,0.1871,0.3151,0.4325,0.5949,0.7786,0.7262
DecisionTreeRegressor,0.2644,0.3887,0.5142,0.3245,0.6237,0.6120,0.2329,0.3805,0.4826,0.4958,0.7089,0.6772
RandomForestRegressor,0.1645,0.3191,0.4056,0.5796,0.7677,0.7601,0.1986,0.3248,0.4456,0.5701,0.7656,0.7205
GradientBoostingRegressor,0.1349,0.2785,0.3673,0.6552,0.8116,0.8057,0.1969,0.3249,0.4438,0.5736,0.7598,0.7229
AdaBoostRegressor,0.1792,0.3385,0.4234,0.5420,0.7422,0.7681,0.2045,0.3353,0.4522,0.5572,0.7729,0.7080
XGBRegressor,0.1649,0.3024,0.4060,0.5787,0.7634,0.7568,0.2080,0.3398,0.4560,0.5498,0.7435,0.7114
ExtraTreesRegressor,0.1357,0.2776,0.3683,0.6534,0.8120,0.8067,0.2028,0.3291,0.4503,0.5609,0.7528,0.7026
LinearRegression,0.9985,0.6888,0.9992,-1.5512,0.3491,0.4125,0.4459,0.4949,0.6678,0.0346,0.6863,0.6833
KNeighborsRegressor,0.1980,0.3281,0.4450,0.4940,0.7179,0.7226,0.2616,0.3599,0.5115,0.4336,0.6644,0.6061
SVR,0.1475,0.2846,0.3840,0.6232,0.7972,0.8099,0.2081,0.3315,0.4562,0.5495,0.7498,0.6951


In [92]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.062324024701217, -5.256802011886175, -5.21...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.169921035660709, -5.874938384263271, -6.5...","[-6.198632767289019, -5.924034037226564, -6.37...","[0.08983217654685756, 0.06214054959289572, 0.2..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -4.9, -5.57, -5.02, -5.41, -5.28...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -7.0, -5.45, -5.45, -5.35, -5....","[-5.938000000000001, -5.7219999999999995, -6.6...","[0.6551763121481116, 0.0759999999999998, 0.639..."
2,RandomForestRegressor,"[-6.125299999999997, -5.0670999999999955, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.030099999999996, -5.8941699999999955, -6....","[-6.156730666666664, -5.938923999999996, -6.28...","[0.08750059673193353, 0.07878252664138348, 0.2..."
3,GradientBoostingRegressor,"[-6.308015034928348, -4.999562488285104, -5.04...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.019807801421183, -5.783516122087968, -6.5...","[-6.147828168842628, -5.815899442496798, -6.31...","[0.1743392673150182, 0.07392576425689873, 0.17..."
4,AdaBoostRegressor,"[-6.011304347826085, -5.287758620689656, -5.28...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.015277777777777, -5.854716981132072, -6.4...","[-6.097881204906204, -5.9786431511283755, -6.1...","[0.08463626865277028, 0.27737371063761, 0.3456..."
5,XGBRegressor,"[-6.113192, -4.948682, -5.014025, -6.097707, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.103551, -5.760788, -6.361973, -6.6328917,...","[-6.19312, -5.7970376, -6.333668, -6.5796065, ...","[0.1079705, 0.07318984, 0.17642114, 0.14151901..."
6,ExtraTreesRegressor,"[-6.096299999999996, -4.980399999999994, -5.06...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.083399999999997, -5.7599999999999945, -6....","[-6.131307999999997, -5.821779999999995, -6.04...","[0.028440285793219947, 0.12356000000000157, 0...."
7,LinearRegression,"[-5.2388143371434905, -5.117832319325097, -4.9...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.861144603512355, -5.75999999999999, -7.34...","[-5.952744336643621, -5.699248739090869, -7.79...","[1.3215508212375624, 0.12150252181818973, 0.52..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -5.885999999999999, -6.2...","[0.12730715263138598, 0.04873511168665874, 0.2..."
9,SVR,"[-6.024673481301633, -5.250855129202577, -5.23...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.978599465555913, -5.980658123740004, -5.8...","[-6.049296634979823, -6.03810656246225, -6.025...","[0.06703973292976419, 0.06185553437018037, 0.2..."


In [93]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_desc_RRCK.csv')

In [94]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [95]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [96]:
#2d RDKit descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 161)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 161)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084963 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2783
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 114
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.6698758564097418


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1735,0.3398,0.4165,0.5567,0.7499,0.7387,0.2304,0.3509,0.4800,0.5011,0.7128,0.6504
DecisionTreeRegressor,0.3170,0.4228,0.5630,0.1901,0.5717,0.5890,0.1932,0.3088,0.4396,0.5816,0.7637,0.7153
RandomForestRegressor,0.1629,0.3126,0.4036,0.5837,0.7695,0.7519,0.2085,0.3316,0.4566,0.5486,0.7506,0.7068
GradientBoostingRegressor,0.1385,0.2866,0.3722,0.6461,0.8047,0.7956,0.2128,0.3407,0.4613,0.5394,0.7393,0.6705
AdaBoostRegressor,0.1800,0.3366,0.4242,0.5402,0.7426,0.7748,0.2237,0.3613,0.4730,0.5157,0.7425,0.6888
XGBRegressor,0.1981,0.3407,0.4451,0.4939,0.7166,0.7007,0.1974,0.3241,0.4442,0.5727,0.7585,0.6823
ExtraTreesRegressor,0.1305,0.2781,0.3613,0.6665,0.8186,0.8017,0.2147,0.3553,0.4634,0.5351,0.7388,0.6793
LinearRegression,3.0287,1.1523,1.7403,-6.7383,0.2753,0.3328,0.7967,0.6228,0.8926,-0.7249,0.4003,0.4121
KNeighborsRegressor,0.1762,0.3164,0.4198,0.5498,0.7770,0.7643,0.3074,0.3844,0.5545,0.3344,0.6089,0.5131
SVR,0.1452,0.3001,0.3810,0.6290,0.8037,0.7987,0.2493,0.3810,0.4993,0.4603,0.6840,0.6148


In [97]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.807090876443247, -5.471560088600416, -5.31...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1572334966963895, -6.18520858555811, -5.7...","[-6.176264485734626, -6.17290609909846, -5.921...","[0.056074078426976134, 0.033321083270437236, 0..."
1,DecisionTreeRegressor,"[-5.87, -4.9, -5.14, -5.92, -5.32, -5.57, -6.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.12, -5.76, -6.89, -6.89, -6.78, -4.9, -5....","[-6.409999999999999, -5.833999999999999, -6.2,...","[0.34876926470089076, 0.14800000000000005, 0.7..."
2,RandomForestRegressor,"[-5.924099999999997, -5.119049999999997, -5.10...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.047151666666666, -5.875849999999996, -5.8...","[-6.137885666666666, -5.984419999999997, -5.90...","[0.07227530355522559, 0.12438477639968769, 0.2..."
3,GradientBoostingRegressor,"[-5.935977652441868, -5.163547569997099, -5.09...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.021888686483729, -5.8240411892230055, -5....","[-6.282377148609578, -5.864125113212623, -5.77...","[0.23247461653183288, 0.13445065926404548, 0.3..."
4,AdaBoostRegressor,"[-5.8481250000000005, -5.3177777777777795, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.989999999999999, -5.989999999999999, -5.9...","[-6.043465854149877, -6.072954997093175, -5.96...","[0.1266136881127132, 0.18698093075989045, 0.39..."
5,XGBRegressor,"[-6.1582794, -5.104896, -5.0736814, -5.6518064...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.133024, -5.760761, -6.377087, -6.4075317,...","[-6.2894974, -5.9168625, -6.0944095, -6.314816...","[0.17454444, 0.3123131, 0.50034726, 0.35522482..."
6,ExtraTreesRegressor,"[-5.926599999999997, -5.1177, -5.0763999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.055299999999996, -5.7599999999999945, -5....","[-6.044279999999997, -5.829539999999996, -5.80...","[0.0532498037555071, 0.139080000000002, 0.2355..."
7,LinearRegression,"[-4.0, -5.203373168606692, -5.299963383917959,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.593135205390599, -5.743502788602104, -4.0...","[-5.70615126221636, -5.406737494850502, -5.099...","[1.5904517404242295, 0.7034134687902128, 1.366..."
8,KNeighborsRegressor,"[-5.919999999999999, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-5.885999999999999, -5.885999999999999, -5.73...","[0.04873511168665874, 0.04873511168665874, 0.2..."
9,SVR,"[-6.17950755564847, -5.242626738094077, -5.238...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.057675154695909, -5.859497809250259, -5.6...","[-6.062453427656054, -5.884529049979111, -5.72...","[0.03233794359962698, 0.049489255630979986, 0...."


In [98]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_const_rem_RRCK.csv')

In [101]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')
# X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
# X_test = X_test.select_dtypes(include=['number'])
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1184)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1184)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.079130 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38372
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1109
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1255,0.2832,0.3543,0.6793,0.8302,0.8177,0.1871,0.3151,0.4325,0.5949,0.7786,0.7262
DecisionTreeRegressor,0.2785,0.3920,0.5277,0.2885,0.6228,0.6159,0.2421,0.3845,0.4920,0.4758,0.6971,0.6279
RandomForestRegressor,0.1636,0.3204,0.4045,0.5820,0.7693,0.7608,0.1983,0.3240,0.4453,0.5708,0.7665,0.7145
GradientBoostingRegressor,0.1383,0.2809,0.3719,0.6467,0.8065,0.8058,0.1962,0.3262,0.4429,0.5753,0.7604,0.7299
AdaBoostRegressor,0.1639,0.3298,0.4049,0.5812,0.7693,0.7693,0.1983,0.3365,0.4453,0.5706,0.7771,0.7296
XGBRegressor,0.1649,0.3024,0.4060,0.5787,0.7634,0.7568,0.2080,0.3398,0.4560,0.5498,0.7435,0.7114
ExtraTreesRegressor,0.1414,0.2869,0.3761,0.6387,0.8024,0.7903,0.2037,0.3335,0.4513,0.5590,0.7524,0.7086
LinearRegression,0.9985,0.6888,0.9992,-1.5512,0.3491,0.4125,0.4459,0.4949,0.6678,0.0346,0.6863,0.6833
KNeighborsRegressor,0.1980,0.3281,0.4450,0.4940,0.7179,0.7226,0.2616,0.3599,0.5115,0.4336,0.6644,0.6061
SVR,0.1475,0.2846,0.3840,0.6232,0.7972,0.8099,0.2090,0.3319,0.4571,0.5476,0.7484,0.6905


In [102]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.062324024701217, -5.256802011886175, -5.21...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.169921035660709, -5.874938384263271, -6.5...","[-6.198632767289019, -5.924034037226564, -6.37...","[0.08983217654685756, 0.06214054959289572, 0.2..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.14, -5.53, -5.08, -5.57, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -7.0, -5.45, -5.4, -5.05, -5.4...","[-5.908, -5.781999999999999, -6.64800000000000...","[0.6153503067359274, 0.044000000000000136, 0.6..."
2,RandomForestRegressor,"[-6.073199999999994, -5.050566666666663, -5.13...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.055799999999996, -5.882066666666662, -6.4...","[-6.15844785281385, -5.951699761904758, -6.298...","[0.08595735055774197, 0.0827145698873871, 0.22..."
3,GradientBoostingRegressor,"[-6.230949260070783, -4.970626315686047, -5.00...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.058002540570984, -5.783516122087968, -6.4...","[-6.146217123953274, -5.8198458698299, -6.3693...","[0.09868645611741549, 0.0786780440625311, 0.14..."
4,AdaBoostRegressor,"[-5.926896551724136, -5.252000000000001, -5.25...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.216486486486484, -5.880499999999997, -6.4...","[-6.101414256361624, -5.952200732600732, -6.28...","[0.10986558190681875, 0.0989942703893136, 0.30..."
5,XGBRegressor,"[-6.113192, -4.948682, -5.014025, -6.097707, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.103551, -5.760788, -6.361973, -6.6328917,...","[-6.19312, -5.7970376, -6.333668, -6.5796065, ...","[0.1079705, 0.07318984, 0.17642114, 0.14151901..."
6,ExtraTreesRegressor,"[-6.041699999999996, -4.986999999999995, -5.05...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.134799999999996, -5.7599999999999945, -6....","[-6.142179999999997, -5.822739999999995, -6.05...","[0.03779841068087361, 0.1254800000000021, 0.20..."
7,LinearRegression,"[-5.238814337143454, -5.117832319325064, -4.97...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.861144603512498, -5.759999999999979, -7.3...","[-5.9527443366436, -5.6992487390908915, -7.797...","[1.3215508212375158, 0.12150252181816121, 0.52..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -5.885999999999999, -6.2...","[0.12730715263138598, 0.04873511168665874, 0.2..."
9,SVR,"[-6.02467349536651, -5.250855232810105, -5.230...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.978599479489121, -5.980658137685163, -5.8...","[-6.049293668791092, -6.038104843254184, -6.02...","[0.06704513542280281, 0.06187348246033691, 0.2..."


In [103]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_df_2d_Mordred_const_rem_RRCK.csv')

In [104]:
#2d RDKit descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_LVR_rdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_LVR_rdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 152)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 152)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2451
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 105
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.571570687516292


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1697,0.3419,0.4119,0.5665,0.7556,0.7481,0.2327,0.3590,0.4824,0.4962,0.7107,0.6692
DecisionTreeRegressor,0.3491,0.4368,0.5909,0.1079,0.5311,0.5291,0.1959,0.3234,0.4426,0.5759,0.7589,0.7068
RandomForestRegressor,0.1593,0.3115,0.3992,0.5929,0.7750,0.7554,0.2097,0.3355,0.4579,0.5460,0.7497,0.7169
GradientBoostingRegressor,0.1368,0.2873,0.3699,0.6504,0.8070,0.7941,0.2001,0.3408,0.4473,0.5669,0.7613,0.7155
AdaBoostRegressor,0.1795,0.3400,0.4237,0.5414,0.7437,0.7628,0.2256,0.3636,0.4750,0.5116,0.7415,0.6785
XGBRegressor,0.1945,0.3432,0.4410,0.5031,0.7201,0.7042,0.1913,0.3295,0.4374,0.5858,0.7693,0.7101
ExtraTreesRegressor,0.1339,0.2832,0.3659,0.6579,0.8130,0.7987,0.2136,0.3538,0.4622,0.5375,0.7389,0.6802
LinearRegression,2.5047,1.0026,1.5826,-5.3996,0.2508,0.3197,0.9230,0.6226,0.9607,-0.9984,0.4649,0.5461
KNeighborsRegressor,0.1699,0.3148,0.4122,0.5659,0.7855,0.7717,0.2863,0.3691,0.5351,0.3801,0.6424,0.5672
SVR,0.1461,0.3037,0.3823,0.6266,0.8026,0.7961,0.2502,0.3819,0.5002,0.4584,0.6828,0.6097


In [105]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.878022609123952, -5.331851304825098, -5.33...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.129924280719684, -6.155645929868835, -5.7...","[-6.148385141849053, -6.1697640500635345, -5.8...","[0.03931969805179674, 0.07433778213175972, 0.1..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.14, -5.92, -5.28, -5.57, -6.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.18, -5.76, -6.89, -6.89, -6.89, -4.9, -5....","[-6.196, -5.781999999999999, -6.204, -6.758, -...","[0.32867004731189015, 0.044000000000000136, 0...."
2,RandomForestRegressor,"[-5.9248999999999965, -5.107499999999998, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.029465714285712, -5.873699999999996, -5.9...","[-6.0915872380952365, -5.989122571428568, -5.9...","[0.0684328211209138, 0.12148909351703949, 0.23..."
3,GradientBoostingRegressor,"[-5.870350770550735, -5.116337188243337, -5.09...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.058346059995749, -5.805759829349192, -5.9...","[-6.240729165917439, -5.861925456863106, -5.89...","[0.19151689542864342, 0.13013925414277852, 0.3..."
4,AdaBoostRegressor,"[-5.823333333333334, -5.327547169811319, -5.32...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0025, -5.910754716981128, -5.714, -6.1000...","[-6.097186068111454, -6.041514430402624, -5.93...","[0.11073659805025593, 0.14343516508651336, 0.3..."
5,XGBRegressor,"[-6.1508675, -5.1388564, -5.1244125, -5.932036...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.012635, -5.7609954, -6.367195, -6.4316125...","[-6.1432524, -5.8254, -6.176009, -6.2425966, -...","[0.24231026, 0.12950747, 0.4390638, 0.29183906..."
6,ExtraTreesRegressor,"[-6.026499999999999, -5.100099999999998, -5.07...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.004499999999998, -5.7599999999999945, -5....","[-6.029354999999998, -5.830219999999995, -5.84...","[0.08421021078230344, 0.14044000000000167, 0.2..."
7,LinearRegression,"[-10.0, -5.148733727964662, -5.330117590747244...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.828404830962449, -5.74325907768175, -7.39...","[-5.709643596855509, -5.399221300126793, -7.07...","[1.578042639516875, 0.6996588787061936, 0.7720..."
8,KNeighborsRegressor,"[-5.919999999999999, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-5.885999999999999, -5.885999999999999, -5.73...","[0.04873511168665874, 0.04873511168665874, 0.2..."
9,SVR,"[-6.176628352841027, -5.243935597959921, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.059544898203488, -5.860375757284078, -5.6...","[-6.063154511192835, -5.885130263603474, -5.73...","[0.03317696526455292, 0.05034508789497488, 0.1..."


In [106]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_LVR_RRCK.csv')

In [108]:
#2d Mordred descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
results_df

X_train shape:  (140, 854)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 854)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071758 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26640
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 785
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1231,0.2831,0.3509,0.6855,0.8368,0.8314,0.2035,0.3239,0.4511,0.5595,0.7517,0.7042
DecisionTreeRegressor,0.2043,0.3390,0.4520,0.4779,0.7272,0.7157,0.2751,0.3895,0.5245,0.4043,0.6413,0.6034
RandomForestRegressor,0.1617,0.3186,0.4021,0.5868,0.7717,0.7621,0.2084,0.3320,0.4565,0.5488,0.7478,0.6961
GradientBoostingRegressor,0.1324,0.2870,0.3639,0.6616,0.8141,0.8097,0.2151,0.3364,0.4638,0.5343,0.7318,0.7034
AdaBoostRegressor,0.1993,0.3597,0.4465,0.4907,0.7045,0.7274,0.2057,0.3464,0.4536,0.5546,0.7565,0.7260
XGBRegressor,0.1679,0.3198,0.4097,0.5711,0.7595,0.7578,0.2165,0.3423,0.4653,0.5314,0.7302,0.7058
ExtraTreesRegressor,0.1333,0.2789,0.3650,0.6595,0.8150,0.8067,0.2017,0.3283,0.4492,0.5632,0.7543,0.7013
LinearRegression,1.0752,0.7144,1.0369,-1.7472,0.2836,0.3257,0.5567,0.5557,0.7461,-0.2052,0.6368,0.5727
KNeighborsRegressor,0.2073,0.3424,0.4553,0.4704,0.7055,0.7201,0.2312,0.3358,0.4808,0.4995,0.7090,0.6612
SVR,0.1524,0.2932,0.3904,0.6106,0.7884,0.7918,0.2155,0.3370,0.4642,0.5335,0.7377,0.6989


In [109]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.25368196693837, -5.313844324151128, -5.248...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.135327797654625, -5.852075995789215, -6.5...","[-6.169938082911362, -5.914917910970436, -6.38...","[0.10240050304219787, 0.11392125445081104, 0.2..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.05, -5.92, -5.28, -5.74, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -7.0, -6.89, -6.42, -4.9, -5.4...","[-6.326, -5.781999999999999, -6.6, -6.59200000...","[0.274269940022599, 0.044000000000000136, 0.58..."
2,RandomForestRegressor,"[-6.121699999999995, -5.093249999999996, -5.16...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.037466666666663, -5.891199999999994, -6.5...","[-6.166303999999996, -5.970939999999996, -6.30...","[0.07469870531080933, 0.09147750761799531, 0.2..."
3,GradientBoostingRegressor,"[-6.401541217193528, -4.946065205819494, -5.06...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.077674506374981, -5.793149125946914, -6.5...","[-6.206162212112746, -5.8313503530217305, -6.2...","[0.09042939085889025, 0.08528530647651236, 0.2..."
4,AdaBoostRegressor,"[-6.055918367346937, -5.198, -5.23642857142857...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.062307692307688, -5.948928571428567, -6.6...","[-6.1193496481701235, -5.975154540478676, -6.4...","[0.07669004274546602, 0.06082019985431565, 0.2..."
5,XGBRegressor,"[-6.1960697, -4.9505773, -5.049124, -6.1801257...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.137565, -5.7603493, -6.3802547, -6.512685...","[-6.349485, -5.8206167, -6.2471857, -6.3747854...","[0.14642315, 0.12039173, 0.38307732, 0.4371114..."
6,ExtraTreesRegressor,"[-6.147699999999997, -5.032499999999996, -5.04...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.124499999999995, -5.7599999999999945, -5....","[-6.156449999999997, -5.820939999999995, -6.02...","[0.07732484723554205, 0.12188000000000196, 0.1..."
7,LinearRegression,"[-4.785252196095102, -4.9791085479473125, -4.8...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.759999999999966, -6.849900610639199...","[-5.467941356598866, -5.648488819787468, -7.41...","[1.2449480708352287, 0.22302236042507456, 0.62..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -5.885999999999999, -6.4...","[0.12730715263138598, 0.04873511168665874, 0.3..."
9,SVR,"[-6.030211234755988, -5.2521702902598575, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.982997260546891, -5.98513532487088, -5.88...","[-6.051847909971086, -6.039107018773394, -6.01...","[0.06390616539872604, 0.05853899898928307, 0.2..."


In [110]:
results_df.to_csv('results/Descriptors/Results_2d_Mordred_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_LVR_RRCK.csv')

In [111]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_RRCK.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-1.1856,1.405647,198.6856,125.288752,0,0,117,53,64,...,104.587391,1.973347,37.376734,17.883978,19.492756,10001.0,88.0,5.289,266.0,1035
1,0,-1.2765,1.629452,204.7959,128.382338,0,0,120,54,66,...,106.403540,1.970436,37.505267,17.862423,19.642844,10348.0,92.0,4.380,272.0,1032
2,0,-1.5766,2.485668,194.4885,122.195166,0,0,114,52,62,...,102.770477,1.976355,37.538804,17.903031,19.635773,9622.0,84.0,4.406,260.0,1037
3,0,-1.9676,3.871450,190.2914,119.101580,0,0,111,51,60,...,100.952062,1.979452,37.702286,17.921964,19.780322,9274.0,80.0,3.523,254.0,1039
4,0,-1.4824,2.197510,194.7852,123.159959,0,0,115,52,63,...,102.729306,1.975564,37.904740,15.362539,22.542201,9437.0,86.0,4.882,260.0,1041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,0,-2.2959,5.271157,259.0984,144.945994,0,0,124,66,58,...,135.026472,2.045856,43.876374,15.452959,25.442427,18776.0,103.0,5.021,338.0,1860
136,0,-2.1936,4.811881,264.7951,148.039580,0,0,127,67,60,...,136.842632,2.042427,44.044180,15.431464,25.631764,19280.0,107.0,4.734,344.0,1861
137,0,-2.0160,4.064256,265.1417,151.319959,0,0,131,68,63,...,139.322655,2.048863,37.708724,15.472218,22.236506,20899.0,104.0,7.346,344.0,1849
138,0,-2.3982,5.751363,253.4017,141.852408,0,0,121,65,56,...,133.210563,2.049393,43.712429,15.474678,25.255779,18261.0,99.0,5.308,332.0,1851


In [112]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')
df 


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1215.857018,6.109834,38268,152,418.0,484.0,44.111111,19.277778
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1213.841368,6.161631,38268,152,418.0,484.0,44.111111,19.277778
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,100.268636,2.427280,4.854560,...,11.184019,125.341576,1201.841368,6.131844,37826,148,410.0,473.0,42.638889,19.277778
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,98.550044,2.444219,4.888335,...,11.198475,124.353468,1187.825718,6.154537,36408,148,408.0,472.0,43.000000,18.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,55.201917,2.419901,4.786578,...,10.503834,95.786335,626.379183,6.593465,6693,72,224.0,257.0,18.027778,10.055556
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,53.640751,2.419606,4.784653,...,10.483550,94.692583,612.363533,6.656125,6341,70,220.0,252.0,17.777778,9.722222
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,52.045329,2.439503,4.844153,...,10.570008,93.767368,606.410483,6.251654,5725,76,214.0,251.0,20.250000,9.638889
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,50.591075,2.439146,4.842060,...,10.560671,92.694207,592.394833,6.302073,5381,75,210.0,247.0,20.000000,9.388889


In [113]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1035,CC(C)C[C@@H]1NC(=O)CN(Cc2ccc(O)cc2)C(=O)[C@H]2...,-5.20,0,-1.1856,1.405647,198.6856,125.288752,0,0,...,6.328920,104.587391,1.973347,37.376734,17.883978,19.492756,10001.0,88.0,5.289,266.0
1,1032,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.26,0,-1.2765,1.629452,204.7959,128.382338,0,0,...,6.287494,106.403540,1.970436,37.505267,17.862423,19.642844,10348.0,92.0,4.380,272.0
2,1037,CC(C)C[C@@H]1NC(=O)CN(Cc2ccc(O)cc2)C(=O)[C@H]2...,-5.14,0,-1.5766,2.485668,194.4885,122.195166,0,0,...,6.372526,102.770477,1.976355,37.538804,17.903031,19.635773,9622.0,84.0,4.406,260.0
3,1039,CC(C)C[C@@H]1NC(=O)CN(Cc2ccc(O)cc2)C(=O)[C@H]2...,-5.54,0,-1.9676,3.871450,190.2914,119.101580,0,0,...,6.418490,100.952062,1.979452,37.702286,17.921964,19.780322,9274.0,80.0,3.523,254.0
4,1041,CC(C)C[C@@H]1NC(=O)CN(Cc2ccccn2)C(=O)[C@H]2CCC...,-4.74,0,-1.4824,2.197510,194.7852,123.159959,0,0,...,6.308556,102.729306,1.975564,37.904740,15.362539,22.542201,9437.0,86.0,4.882,260.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,1860,CN1C(=O)[C@@H](Cc2ccccc2)NC(=O)[C@H](CCCc2cccc...,-5.57,0,-2.2959,5.271157,259.0984,144.945994,0,0,...,7.342097,135.026472,2.045856,43.876374,15.452959,25.442427,18776.0,103.0,5.021,338.0
136,1861,CN1C(=O)[C@@H](Cc2ccccc2)N(C)C(=O)[C@H](CCCc2c...,-5.40,0,-2.1936,4.811881,264.7951,148.039580,0,0,...,7.279021,136.842632,2.042427,44.044180,15.431464,25.631764,19280.0,107.0,4.734,344.0
137,1849,O=C1CN(CCCc2ccccc2)C(=O)[C@H]2CCCN2C(=O)[C@H](...,-5.92,0,-2.0160,4.064256,265.1417,151.319959,0,0,...,7.003695,139.322655,2.048863,37.708724,15.472218,22.236506,20899.0,104.0,7.346,344.0
138,1851,O=C1CN(CCCc2ccccc2)C(=O)[C@H]2CCCN2C(=O)[C@H](...,-6.15,0,-2.3982,5.751363,253.4017,141.852408,0,0,...,7.408300,133.210563,2.049393,43.712429,15.474678,25.255779,18261.0,99.0,5.308,332.0


In [114]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,-2.3118,5.344419,324.4169,207.951609,0,0,...,6.109834,165.660911,1.926290,64.869552,30.340460,34.529092,38268.0,152.0,8.704,418.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,-1.9035,3.623312,324.6192,206.618023,0,0,...,6.161631,165.660911,1.926290,64.869552,30.340460,34.529092,38268.0,152.0,7.824,418.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,-3.1214,9.743138,318.9594,204.858023,0,0,...,6.131844,163.824465,1.927347,64.851242,30.333765,34.517477,37337.0,150.0,8.407,412.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,-2.5378,6.440429,318.5450,204.858023,0,0,...,6.131844,164.030378,1.929769,64.941671,30.873015,34.068656,37826.0,148.0,8.690,410.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,-2.8334,8.028156,316.0478,201.764437,0,0,...,6.154537,161.805840,1.926260,64.748500,30.296198,34.452302,36408.0,148.0,8.049,408.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,-2.5740,6.625476,161.8705,102.831650,0,0,...,6.593465,89.382185,1.986271,33.894145,15.321219,18.572926,6693.0,72.0,5.035,224.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,-2.2860,5.225796,158.9589,99.738064,0,0,...,6.656125,87.363105,1.985525,33.788693,15.284672,18.504021,6341.0,70.0,4.677,220.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,-3.7740,14.243076,153.1698,101.978822,0,0,...,6.251654,83.927240,1.951796,34.294802,15.216440,19.078362,5725.0,76.0,3.494,214.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,-3.4860,12.152196,150.2582,98.885236,0,0,...,6.302073,81.919106,1.950455,34.250954,15.200426,19.050528,5381.0,75.0,2.925,210.0


In [115]:
df_ordered.to_csv('features/Descriptors/Train_2d_padel_curated_RRCK.csv', index=False)

In [116]:
#2d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_RRCK.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-2.5450,6.477025,193.2906,119.101580,0,0,111,51,60,...,100.950923,1.979430,38.389707,17.922198,20.467508,9274.0,80.0,1.183,254.0,1040
1,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,2296
2,0,-1.5766,2.485668,194.4885,122.195166,0,0,114,52,62,...,102.769762,1.976342,37.537206,17.902556,19.634650,9648.0,84.0,4.406,260.0,1038
3,0,-1.3788,1.901089,199.0992,125.288752,0,0,117,53,64,...,104.585158,1.973305,37.329155,17.881349,19.447806,9989.0,88.0,4.667,266.0,1033
4,0,-2.2086,4.877914,191.7826,127.529510,0,0,122,52,70,...,101.677298,1.955333,37.702648,18.242792,19.459856,9672.0,84.0,5.962,254.0,1044
5,0,-2.0598,4.242776,259.4091,161.788612,0,0,152,68,84,...,132.963505,1.955346,48.807230,22.800249,26.006980,19257.0,124.0,4.656,344.0,1868
6,0,-2.4208,5.860273,160.1675,105.072408,0,0,100,44,56,...,85.761816,1.949132,34.302616,15.219294,19.083322,6071.0,77.0,4.002,220.0,2306
7,0,-8.3366,69.498900,240.1531,158.695026,0,0,149,67,82,...,131.666340,1.965169,48.993895,22.868615,26.125280,18483.0,125.0,3.909,332.0,1876
8,0,-8.1732,66.801198,270.3145,170.446198,0,0,160,74,86,...,144.541181,1.953259,60.018035,27.748959,32.269076,25441.0,143.0,0.955,372.0,1882
9,0,-1.6436,2.701421,172.9884,114.353166,0,0,109,47,62,...,91.623817,1.949443,34.446849,15.270649,19.176200,7129.0,81.0,5.437,234.0,2309


In [117]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1217.836283,6.181910,38268,152,418.0,484.0,44.111111,19.277778
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.282174,2.460904,4.898232,...,11.232960,131.870970,1094.710354,6.364595,28969,148,388.0,464.0,36.055556,18.000000
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,91.172927,2.459349,4.893348,...,11.208477,127.699425,1038.647754,6.491548,25441,143,372.0,447.0,35.055556,16.916667
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,87.929966,2.459011,4.889438,...,11.129422,124.389878,995.641940,6.382320,22326,133,354.0,422.0,32.472222,16.250000
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.784106,2.466654,4.910608,...,11.091041,121.222371,952.636126,6.267343,19257,124,344.0,406.0,32.333333,15.000000
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.484783,2.465318,4.908864,...,11.066935,120.067649,938.620476,6.299466,18483,125,332.0,396.0,29.638889,15.583333
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.177160,2.435750,4.819009,...,10.908540,119.586829,913.400825,7.486892,18371,100,334.0,383.0,21.673611,14.208333
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,78.841373,2.449488,4.853049,...,10.792633,115.248549,898.535032,6.558650,16763,102,302.0,347.0,24.361111,14.777778
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,80.291949,2.435508,4.818260,...,10.844022,117.353041,875.440404,7.060003,16811,95,322.0,368.0,19.951389,13.847222


In [118]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1040,CC(C)CN1CC(=O)N(CC(C)C)CC(=O)N(CC(C)C)CC(=O)N2...,-6.400,0,-2.5450,6.477025,193.2906,119.101580,0,0,...,6.418490,100.950923,1.979430,38.389707,17.922198,20.467508,9274.0,80.0,1.183,254.0
1,2296,CC(C)C[C@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](C...,-6.310,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
2,1038,CC(C)C[C@@H]1NC(=O)CN(Cc2ccc(O)cc2)C(=O)[C@H]2...,-5.440,0,-1.5766,2.485668,194.4885,122.195166,0,0,...,6.372526,102.769762,1.976342,37.537206,17.902556,19.634650,9648.0,84.0,4.406,260.0
3,1033,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.130,0,-1.3788,1.901089,199.0992,125.288752,0,0,...,6.328920,104.585158,1.973305,37.329155,17.881349,19.447806,9989.0,88.0,4.667,266.0
4,1044,CC(C)C[C@@H]1NC(=O)CN(CCCOC(C)C)C(=O)[C@H]2CCC...,-5.040,0,-2.2086,4.877914,191.7826,127.529510,0,0,...,6.020743,101.677298,1.955333,37.702648,18.242792,19.459856,9672.0,84.0,5.962,254.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,-2.0598,4.242776,259.4091,161.788612,0,0,...,6.267343,132.963505,1.955346,48.807230,22.800249,26.006980,19257.0,124.0,4.656,344.0
6,2306,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-4.750,0,-2.4208,5.860273,160.1675,105.072408,0,0,...,6.204261,85.761816,1.949132,34.302616,15.219294,19.083322,6071.0,77.0,4.002,220.0
7,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,-8.3366,69.498900,240.1531,158.695026,0,0,...,6.299466,131.666340,1.965169,48.993895,22.868615,26.125280,18483.0,125.0,3.909,332.0
8,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,-8.1732,66.801198,270.3145,170.446198,0,0,...,6.491548,144.541181,1.953259,60.018035,27.748959,32.269076,25441.0,143.0,0.955,372.0
9,2309,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-4.600,0,-1.6436,2.701421,172.9884,114.353166,0,0,...,6.077735,91.623817,1.949443,34.446849,15.270649,19.176200,7129.0,81.0,5.437,234.0


In [119]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,-3.3441,11.183005,322.0103,205.660023,0,0,...,6.181910,165.660911,1.926290,67.281618,32.752526,34.529092,38268.0,152.0,7.806,418.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,-3.1214,9.743138,318.9594,204.858023,0,0,...,6.131844,163.824465,1.927347,64.851242,30.333765,34.517477,37337.0,150.0,8.407,412.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,-9.3252,86.959355,281.9609,182.820542,0,0,...,6.364595,152.577949,1.956128,60.213907,27.820656,32.393251,28969.0,148.0,3.020,388.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,-8.1732,66.801198,270.3145,170.446198,0,0,...,6.491548,144.541181,1.953259,60.018035,27.748959,32.269076,25441.0,143.0,0.955,372.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,-8.3989,70.541521,256.6896,166.117405,0,0,...,6.382320,139.082785,1.958912,54.418777,25.276857,29.141920,22326.0,133.0,3.033,354.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,-2.0598,4.242776,259.4091,161.788612,0,0,...,6.267343,132.963505,1.955346,48.807230,22.800249,26.006980,19257.0,124.0,4.656,344.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,-8.3366,69.498900,240.1531,158.695026,0,0,...,6.299466,131.666340,1.965169,48.993895,22.868615,26.125280,18483.0,125.0,3.909,332.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,-0.2694,0.072576,246.8977,139.815201,0,0,...,7.486892,131.627090,2.025032,48.756497,18.515596,22.203405,18371.0,100.0,5.070,334.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,-8.0158,64.253050,220.6199,148.418682,0,0,...,6.558650,124.904121,1.982605,48.311777,20.403356,24.888056,16763.0,102.0,6.435,302.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,-0.4717,0.222501,246.2872,142.326373,0,0,...,7.060003,127.921577,2.030501,40.639057,15.453474,22.203344,16811.0,95.0,8.012,322.0


In [120]:
df_ordered.to_csv('features/Descriptors/Test_2d_padel_curated_RRCK.csv', index=False)

In [121]:
#3d Train descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_RRCK.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.258522,2.178582,3.014104,3.725775,4.504857,5.285143,5.992885,6.792336,7.583017,8.310595,...,0.418789,0.483201,0.441507,0.413971,30.483918,273.024911,901.437030,0.340883,1.338679,1032
1,1.261240,2.168987,2.989377,3.693289,4.513655,5.314368,6.005430,6.709361,7.363036,7.873015,...,0.382528,0.396407,0.482040,0.346020,26.757426,225.546792,829.961411,0.229523,1.224467,1043
2,1.269805,2.233919,3.042349,3.854016,4.684512,5.416980,6.124348,6.831340,7.496209,8.190869,...,0.318071,0.472096,0.552395,0.352037,39.621855,427.317271,1560.903774,0.381808,1.376528,1850
3,1.273232,2.237442,3.055163,3.877039,4.716471,5.457944,6.162207,6.762358,7.368559,8.044591,...,0.281164,0.464101,0.459530,0.412983,37.822777,387.038047,1466.033247,0.408817,1.336615,1848
4,1.261259,2.178294,3.013773,3.770215,4.648403,5.332800,6.003143,6.764511,7.640699,8.427945,...,0.417750,0.419671,0.426499,0.396497,32.701668,291.912460,759.000668,0.415158,1.242667,1035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,1.262141,2.193434,3.003030,3.738543,4.611331,5.402115,6.187503,6.936089,7.563316,8.295778,...,0.355228,0.511591,0.476491,0.399663,32.801596,275.542685,647.502616,0.432330,1.387745,5674
136,1.261997,2.182457,3.006600,3.739295,4.570212,5.431818,6.158368,6.877652,7.741932,8.581362,...,0.377663,0.419115,0.437242,0.463798,37.309710,361.952190,889.357732,0.435289,1.320155,46
137,1.260609,2.183179,3.016013,3.746739,4.590225,5.417016,6.154037,6.942950,7.776716,8.496377,...,0.393451,0.429257,0.442580,0.467995,32.966755,313.083436,1042.840197,0.354490,1.339833,5671
138,1.253933,2.164698,2.987200,3.680922,4.494437,5.317306,6.100451,6.887326,7.696507,8.498762,...,0.360449,0.576580,0.507186,0.375890,35.923849,362.518048,1240.362064,0.361975,1.459656,978


In [122]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')
df 

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1215.857018,6.109834,38268,152,418.0,484.0,44.111111,19.277778
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1213.841368,6.161631,38268,152,418.0,484.0,44.111111,19.277778
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,100.268636,2.427280,4.854560,...,11.184019,125.341576,1201.841368,6.131844,37826,148,410.0,473.0,42.638889,19.277778
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,98.550044,2.444219,4.888335,...,11.198475,124.353468,1187.825718,6.154537,36408,148,408.0,472.0,43.000000,18.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,55.201917,2.419901,4.786578,...,10.503834,95.786335,626.379183,6.593465,6693,72,224.0,257.0,18.027778,10.055556
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,53.640751,2.419606,4.784653,...,10.483550,94.692583,612.363533,6.656125,6341,70,220.0,252.0,17.777778,9.722222
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,52.045329,2.439503,4.844153,...,10.570008,93.767368,606.410483,6.251654,5725,76,214.0,251.0,20.250000,9.638889
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,50.591075,2.439146,4.842060,...,10.560671,92.694207,592.394833,6.302073,5381,75,210.0,247.0,20.000000,9.388889


In [123]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1032,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-5.2600,1.258522,2.178582,3.014104,3.725775,4.504857,5.285143,5.992885,...,0.475133,0.418789,0.483201,0.441507,0.413971,30.483918,273.024911,901.437030,0.340883,1.338679
1,1043,CC(C)C[C@@H]1NC(=O)CN(CN2CCOCC2)C(=O)[C@H]2CCC...,-5.0900,1.261240,2.168987,2.989377,3.693289,4.513655,5.314368,6.005430,...,0.437153,0.382528,0.396407,0.482040,0.346020,26.757426,225.546792,829.961411,0.229523,1.224467
2,1850,O=C1CN(CCCc2ccccc2)C(=O)[C@H]2CCCN2C(=O)[C@H](...,-5.7400,1.269805,2.233919,3.042349,3.854016,4.684512,5.416980,6.124348,...,0.587872,0.318071,0.472096,0.552395,0.352037,39.621855,427.317271,1560.903774,0.381808,1.376528
3,1848,O=C1CN(CCCc2ccccc2)C(=O)[C@H]2CCCN2C(=O)[C@H](...,-6.0500,1.273232,2.237442,3.055163,3.877039,4.716471,5.457944,6.162207,...,0.605878,0.281164,0.464101,0.459530,0.412983,37.822777,387.038047,1466.033247,0.408817,1.336615
4,1035,CC(C)C[C@@H]1NC(=O)CN(Cc2ccc(O)cc2)C(=O)[C@H]2...,-5.2000,1.261259,2.178294,3.013773,3.770215,4.648403,5.332800,6.003143,...,0.525689,0.417750,0.419671,0.426499,0.396497,32.701668,291.912460,759.000668,0.415158,1.242667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,5674,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-5.8050,1.262141,2.193434,3.003030,3.738543,4.611331,5.402115,6.187503,...,0.599659,0.355228,0.511591,0.476491,0.399663,32.801596,275.542685,647.502616,0.432330,1.387745
136,46,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.9200,1.261997,2.182457,3.006600,3.739295,4.570212,5.431818,6.158368,...,0.579197,0.377663,0.419115,0.437242,0.463798,37.309710,361.952190,889.357732,0.435289,1.320155
137,5671,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.2025,1.260609,2.183179,3.016013,3.746739,4.590225,5.417016,6.154037,...,0.509542,0.393451,0.429257,0.442580,0.467995,32.966755,313.083436,1042.840197,0.354490,1.339833
138,978,C/C1=C\[C@H](CC(C)C)NC(=O)[C@@H]2CCCN2C(=O)[C@...,-4.9500,1.253933,2.164698,2.987200,3.680922,4.494437,5.317306,6.100451,...,0.547534,0.360449,0.576580,0.507186,0.375890,35.923849,362.518048,1240.362064,0.361975,1.459656


In [124]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,1.253464,2.160869,3.010132,3.721973,4.521266,5.307678,6.013053,...,0.555662,0.361731,0.544173,0.495228,0.417788,55.253242,844.996910,3701.072059,0.376090,1.457189
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,1.255183,2.164464,3.011388,3.714492,4.553241,5.365293,6.064061,...,0.514250,0.383940,0.446697,0.436831,0.348132,50.660858,741.432722,3405.746601,0.347284,1.231660
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,1.253822,2.164320,3.003685,3.691482,4.500952,5.318275,5.993975,...,0.584567,0.352614,0.534974,0.490552,0.346062,57.530851,877.093731,3400.242320,0.405772,1.371588
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,1.254146,2.160962,3.000285,3.711056,4.551785,5.324193,6.029558,...,0.540927,0.396766,0.520911,0.514156,0.415753,54.389916,807.742283,3013.737615,0.406540,1.450820
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,1.255050,2.163883,3.008934,3.701704,4.508969,5.332843,6.044204,...,0.572679,0.345721,0.523047,0.544280,0.438872,53.667187,786.079599,3336.947160,0.377600,1.506199
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1.263749,2.198494,3.010260,3.780755,4.672715,5.496094,6.269202,...,0.558285,0.321297,0.508700,0.556253,0.295744,28.520547,232.063263,761.687250,0.337428,1.360698
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1.263070,2.200005,3.007454,3.772722,4.691634,5.471833,6.228680,...,0.492893,0.353000,0.440062,0.551455,0.349133,25.227357,193.693844,649.413156,0.268840,1.340650
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,1.259541,2.172428,3.016387,3.765681,4.460426,5.323072,6.052844,...,0.540801,0.390620,0.531371,0.520052,0.472508,25.525900,179.262689,445.738032,0.397132,1.523931
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,1.260646,2.174185,3.016437,3.762818,4.530641,5.318130,6.048567,...,0.434247,0.397872,0.462647,0.447369,0.419043,21.042045,138.352372,429.631984,0.248178,1.329059


In [125]:
df_ordered.to_csv('features/Descriptors/Train_3d_padel_curated_RRCK.csv', index=False)

In [126]:
#3d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_RRCK.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.261000,2.178801,3.011315,3.705388,4.549312,5.296390,6.074848,6.958239,7.532188,8.238341,...,0.392318,0.469028,0.490810,0.414457,30.179879,275.185557,992.573247,0.297895,1.374294,1040
1,1.275948,2.225858,3.041477,3.793361,4.634844,5.405932,6.144384,6.883769,7.601080,8.280489,...,0.395997,0.537446,0.535345,0.393729,37.635410,388.799321,1171.822077,0.401629,1.466520,1853
2,1.279863,2.230907,3.050999,3.785692,4.651890,5.470123,6.220941,6.967887,7.607413,8.312839,...,0.345681,0.461735,0.511862,0.384095,37.160707,404.790585,1629.498650,0.309589,1.357693,1859
3,1.258716,2.174121,3.019656,3.771190,4.480794,5.310028,6.026682,6.810671,7.570163,8.143837,...,0.420408,0.451162,0.484828,0.337762,38.674180,453.379291,1937.981514,0.300205,1.273752,1879
4,1.261325,2.173800,3.036389,3.796950,4.493464,5.297981,6.000064,6.828563,7.506464,8.225581,...,0.351295,0.412793,0.478748,0.389842,40.222215,455.898864,1728.491712,0.353195,1.281383,1882
5,1.259984,2.172058,3.017788,3.795102,4.534195,5.281763,6.018058,6.917362,7.724830,8.407464,...,0.378645,0.502782,0.546703,0.346323,37.901202,418.328104,1609.458674,0.336152,1.395808,1876
6,1.259196,2.174213,3.024953,3.781507,4.480843,5.251905,6.005996,6.936256,7.778258,8.578830,...,0.432467,0.491430,0.560578,0.414926,48.550470,662.854459,2481.778448,0.391672,1.466935,1883
7,1.258693,2.171442,3.026413,3.743651,4.498922,5.353520,6.058675,6.906317,7.812110,8.518435,...,0.353966,0.455681,0.504722,0.490337,42.046102,485.726415,1740.123264,0.377539,1.450740,1868
8,1.266435,2.191299,2.992767,3.761367,4.561961,5.415963,6.137118,6.883906,7.639428,8.325394,...,0.373649,0.534010,0.484321,0.388504,35.551011,350.087725,1106.888763,0.382397,1.406835,2332
9,1.267336,2.191433,2.996111,3.776332,4.589508,5.330308,6.012081,6.756147,7.504055,8.220074,...,0.399715,0.517190,0.554701,0.395091,33.141775,344.970427,1456.738694,0.239260,1.466982,2331


In [127]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1217.836283,6.181910,38268,152,418.0,484.0,44.111111,19.277778
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.282174,2.460904,4.898232,...,11.232960,131.870970,1094.710354,6.364595,28969,148,388.0,464.0,36.055556,18.000000
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,91.172927,2.459349,4.893348,...,11.208477,127.699425,1038.647754,6.491548,25441,143,372.0,447.0,35.055556,16.916667
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,87.929966,2.459011,4.889438,...,11.129422,124.389878,995.641940,6.382320,22326,133,354.0,422.0,32.472222,16.250000
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.784106,2.466654,4.910608,...,11.091041,121.222371,952.636126,6.267343,19257,124,344.0,406.0,32.333333,15.000000
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.484783,2.465318,4.908864,...,11.066935,120.067649,938.620476,6.299466,18483,125,332.0,396.0,29.638889,15.583333
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.177160,2.435750,4.819009,...,10.908540,119.586829,913.400825,7.486892,18371,100,334.0,383.0,21.673611,14.208333
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,78.841373,2.449488,4.853049,...,10.792633,115.248549,898.535032,6.558650,16763,102,302.0,347.0,24.361111,14.777778
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,80.291949,2.435508,4.818260,...,10.844022,117.353041,875.440404,7.060003,16811,95,322.0,368.0,19.951389,13.847222


In [128]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1040,CC(C)CN1CC(=O)N(CC(C)C)CC(=O)N(CC(C)C)CC(=O)N2...,-6.400,1.261000,2.178801,3.011315,3.705388,4.549312,5.296390,6.074848,...,0.472945,0.392318,0.469028,0.490810,0.414457,30.179879,275.185557,992.573247,0.297895,1.374294
1,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1.275948,2.225858,3.041477,3.793361,4.634844,5.405932,6.144384,...,0.538422,0.395997,0.537446,0.535345,0.393729,37.635410,388.799321,1171.822077,0.401629,1.466520
2,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1.279863,2.230907,3.050999,3.785692,4.651890,5.470123,6.220941,...,0.527378,0.345681,0.461735,0.511862,0.384095,37.160707,404.790585,1629.498650,0.309589,1.357693
3,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1.258716,2.174121,3.019656,3.771190,4.480794,5.310028,6.026682,...,0.446396,0.420408,0.451162,0.484828,0.337762,38.674180,453.379291,1937.981514,0.300205,1.273752
4,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1.261325,2.173800,3.036389,3.796950,4.493464,5.297981,6.000064,...,0.550835,0.351295,0.412793,0.478748,0.389842,40.222215,455.898864,1728.491712,0.353195,1.281383
5,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1.259984,2.172058,3.017788,3.795102,4.534195,5.281763,6.018058,...,0.512123,0.378645,0.502782,0.546703,0.346323,37.901202,418.328104,1609.458674,0.336152,1.395808
6,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1.259196,2.174213,3.024953,3.781507,4.480843,5.251905,6.005996,...,0.495314,0.432467,0.491430,0.560578,0.414926,48.550470,662.854459,2481.778448,0.391672,1.466935
7,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1.258693,2.171442,3.026413,3.743651,4.498922,5.353520,6.058675,...,0.564393,0.353966,0.455681,0.504722,0.490337,42.046102,485.726415,1740.123264,0.377539,1.450740
8,2332,CCC[C@@H]1C(=O)N(C)[C@@H](CC2CCCCC2)C(=O)N[C@@...,-6.220,1.266435,2.191299,2.992767,3.761367,4.561961,5.415963,6.137118,...,0.547949,0.373649,0.534010,0.484321,0.388504,35.551011,350.087725,1106.888763,0.382397,1.406835
9,2331,CC[C@@H]1C(=O)N(C)[C@@H](CC2CCCCC2)C(=O)N[C@@H...,-5.900,1.267336,2.191433,2.996111,3.776332,4.589508,5.330308,6.012081,...,0.426458,0.399715,0.517190,0.554701,0.395091,33.141775,344.970427,1456.738694,0.239260,1.466982


In [129]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,1.253128,2.162134,3.004789,3.714069,4.529507,5.297218,6.004380,...,0.553179,0.378046,0.621744,0.592669,0.362613,61.295253,1026.337021,4399.866189,0.396838,1.577026
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,1.253508,2.164196,3.003574,3.698403,4.496441,5.333041,6.027535,...,0.593681,0.333277,0.516828,0.508872,0.413331,55.396709,814.970696,3327.239818,0.390522,1.439031
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1.259196,2.174213,3.024953,3.781507,4.480843,5.251905,6.005996,...,0.495314,0.432467,0.491430,0.560578,0.414926,48.550470,662.854459,2481.778448,0.391672,1.466935
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1.261325,2.173800,3.036389,3.796950,4.493464,5.297981,6.000064,...,0.550835,0.351295,0.412793,0.478748,0.389842,40.222215,455.898864,1728.491712,0.353195,1.281383
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1.258716,2.174121,3.019656,3.771190,4.480794,5.310028,6.026682,...,0.446396,0.420408,0.451162,0.484828,0.337762,38.674180,453.379291,1937.981514,0.300205,1.273752
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1.258693,2.171442,3.026413,3.743651,4.498922,5.353520,6.058675,...,0.564393,0.353966,0.455681,0.504722,0.490337,42.046102,485.726415,1740.123264,0.377539,1.450740
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1.259984,2.172058,3.017788,3.795102,4.534195,5.281763,6.018058,...,0.512123,0.378645,0.502782,0.546703,0.346323,37.901202,418.328104,1609.458674,0.336152,1.395808
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1.279863,2.230907,3.050999,3.785692,4.651890,5.470123,6.220941,...,0.527378,0.345681,0.461735,0.511862,0.384095,37.160707,404.790585,1629.498650,0.309589,1.357693
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1.265278,2.193259,2.997694,3.767866,4.578901,5.437520,6.180221,...,0.433018,0.413344,0.440081,0.422826,0.386581,36.776762,417.957514,1822.583332,0.269542,1.249488
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1.275948,2.225858,3.041477,3.793361,4.634844,5.405932,6.144384,...,0.538422,0.395997,0.537446,0.535345,0.393729,37.635410,388.799321,1171.822077,0.401629,1.466520


In [130]:
df_ordered.to_csv('features/Descriptors/Test_3d_padel_curated_RRCK.csv', index=False)

In [131]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1444)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1444)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 33111
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 961
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1350,0.2918,0.3675,0.6550,0.8134,0.8063,0.2115,0.3293,0.4599,0.5422,0.7399,0.7062
DecisionTreeRegressor,0.2668,0.3833,0.5165,0.3184,0.6421,0.6380,0.2104,0.3438,0.4587,0.5444,0.7416,0.7224
RandomForestRegressor,0.1610,0.3155,0.4013,0.5885,0.7781,0.7672,0.1821,0.3118,0.4267,0.6059,0.7965,0.7639
GradientBoostingRegressor,0.1472,0.2985,0.3837,0.6239,0.7919,0.7771,0.1791,0.3058,0.4232,0.6123,0.7911,0.7573
AdaBoostRegressor,0.1580,0.3322,0.3974,0.5964,0.7913,0.7936,0.1944,0.3342,0.4409,0.5792,0.7827,0.7387
XGBRegressor,0.2083,0.3483,0.4564,0.4679,0.6916,0.7094,0.1861,0.3184,0.4314,0.5970,0.7832,0.7422
ExtraTreesRegressor,0.1343,0.2813,0.3664,0.6569,0.8140,0.8086,0.1778,0.3106,0.4217,0.6151,0.7947,0.7670
LinearRegression,7.6648,2.2783,2.7685,-18.5835,0.1288,0.1093,3.8051,1.4854,1.9507,-7.2381,0.1368,0.1960
KNeighborsRegressor,0.1969,0.3290,0.4437,0.4969,0.7267,0.7297,0.2407,0.3491,0.4906,0.4789,0.6936,0.6249
SVR,0.1653,0.3036,0.4066,0.5776,0.7629,0.7654,0.2490,0.3707,0.4990,0.4609,0.6804,0.6074


In [132]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.94532252162547, -5.314277716566065, -5.315...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.180207115353877, -5.946590137875205, -6.0...","[-6.196023623778077, -5.96170174747219, -6.273...","[0.09576708173827467, 0.06878552772770136, 0.1..."
1,DecisionTreeRegressor,"[-6.78, -5.1, -5.14, -5.76, -5.28, -5.76, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.35, -5.35, -5.45, -5.35, -5...","[-6.266, -5.781999999999999, -6.05200000000000...","[0.44066313664748485, 0.044000000000000136, 0...."
2,RandomForestRegressor,"[-6.0566999999999975, -5.254299999999996, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.038566666666663, -5.842199999999996, -6.1...","[-6.114357333333329, -5.9571599999999965, -6.1...","[0.08944961669131037, 0.1051099205593853, 0.17..."
3,GradientBoostingRegressor,"[-6.172299587643261, -5.18863168324436, -5.062...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.189530437070233, -5.78183577474684, -6.17...","[-6.290176901763243, -5.856588171892111, -6.00...","[0.09728846378672652, 0.1420657161512193, 0.13..."
4,AdaBoostRegressor,"[-5.863333333333333, -5.281269841269841, -5.31...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.036470588235295, -5.888571428571431, -6.6...","[-6.240617084154236, -5.917564972213141, -6.42...","[0.1395478131627219, 0.11150448040774001, 0.19..."
5,XGBRegressor,"[-6.357178, -5.5715084, -5.1270785, -5.725583,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.3006334, -5.760637, -6.2490525, -6.303871...","[-6.236652, -5.811528, -5.9847045, -6.3309493,...","[0.10456121, 0.1018184, 0.18370464, 0.3332288,..."
6,ExtraTreesRegressor,"[-6.025049999999997, -5.080699999999999, -5.13...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.102949999999997, -5.7599999999999945, -5....","[-6.136909999999998, -5.822679999999996, -6.04...","[0.0340944042329532, 0.12536000000000236, 0.13..."
7,LinearRegression,"[-10.0, -4.0, -4.0, -10.0, -8.178310088626231,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.759999999997942, -10.0, -10.0, -4.0...","[-7.6, -6.282091800989261, -9.625050890384994,...","[2.939387691339814, 1.0441836019799378, 0.7498..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -6.068, -6.4653333333333...","[0.12730715263138598, 0.1273071526313868, 0.33..."
9,SVR,"[-6.039475309461491, -5.2185022652648945, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.023141410130099, -5.8678146573019925, -6....","[-6.083900106357391, -5.953581823631754, -6.10...","[0.0562755930161941, 0.06949907984368305, 0.30..."


In [133]:
result_df.to_csv('results/Descriptors/Results_2D_padel_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_RRCK.csv')

In [134]:
#2d padel descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1022)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1022)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007957 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 33111
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 961
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1350,0.2918,0.3675,0.6550,0.8134,0.8063,0.2115,0.3293,0.4599,0.5422,0.7399,0.7062
DecisionTreeRegressor,0.2849,0.3948,0.5337,0.2721,0.6119,0.6257,0.1983,0.3219,0.4454,0.5706,0.7593,0.7158
RandomForestRegressor,0.1598,0.3165,0.3997,0.5917,0.7801,0.7701,0.1832,0.3133,0.4280,0.6033,0.7943,0.7630
GradientBoostingRegressor,0.1469,0.2981,0.3833,0.6247,0.7919,0.7800,0.1795,0.3074,0.4236,0.6114,0.7905,0.7542
AdaBoostRegressor,0.1671,0.3255,0.4087,0.5731,0.7671,0.7784,0.1997,0.3413,0.4469,0.5676,0.7796,0.7217
XGBRegressor,0.2083,0.3483,0.4564,0.4679,0.6916,0.7094,0.1861,0.3184,0.4314,0.5970,0.7832,0.7422
ExtraTreesRegressor,0.1382,0.2856,0.3717,0.6469,0.8069,0.7994,0.1798,0.3152,0.4240,0.6107,0.7907,0.7689
LinearRegression,7.6648,2.2783,2.7685,-18.5835,0.1288,0.1093,3.8051,1.4854,1.9507,-7.2381,0.1368,0.1960
KNeighborsRegressor,0.1969,0.3290,0.4437,0.4969,0.7267,0.7297,0.2407,0.3491,0.4906,0.4789,0.6936,0.6249
SVR,0.1653,0.3036,0.4066,0.5776,0.7629,0.7654,0.2513,0.3719,0.5013,0.4560,0.6765,0.6074


In [135]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.94532252162547, -5.314277716566065, -5.315...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.180207115353877, -5.946590137875205, -6.0...","[-6.196023623778077, -5.96170174747219, -6.273...","[0.09576708173827467, 0.06878552772770136, 0.1..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.05, -5.74, -5.28, -5.92, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.35, -5.35, -6.89, -5.2, -5....","[-6.215999999999999, -5.833999999999999, -6.02...","[0.38192145789415927, 0.14800000000000005, 0.5..."
2,RandomForestRegressor,"[-6.064399999999997, -5.238899999999996, -5.20...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.077833333333331, -5.837199999999996, -6.2...","[-6.128693333333331, -5.953659999999997, -6.17...","[0.0826760912638401, 0.10685385533522125, 0.17..."
3,GradientBoostingRegressor,"[-6.1651425485917155, -5.167262253976546, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.163670567143688, -5.78183577474684, -6.13...","[-6.308337507619461, -5.85891121693821, -5.983...","[0.1107125361464909, 0.1467109692558485, 0.114..."
4,AdaBoostRegressor,"[-5.863333333333332, -5.220303030303032, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.983181818181818, -5.862294117647059, -6.5...","[-6.202474458874459, -5.956749407944995, -6.25...","[0.20705618610643708, 0.2105907643648686, 0.42..."
5,XGBRegressor,"[-6.357178, -5.5715084, -5.1270785, -5.725583,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.3006334, -5.760637, -6.2490525, -6.303871...","[-6.236652, -5.811528, -5.9847045, -6.3309493,...","[0.10456121, 0.1018184, 0.18370464, 0.3332288,..."
6,ExtraTreesRegressor,"[-6.0602499999999955, -5.095899999999997, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.117699999999994, -5.7599999999999945, -6....","[-6.151819999999999, -5.821019999999995, -6.07...","[0.0664318718688562, 0.12204000000000227, 0.08..."
7,LinearRegression,"[-10.0, -4.0, -4.0, -10.0, -8.17831008859503, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.759999999997921, -10.0, -10.0, -4.0...","[-7.6, -6.282091800988016, -9.625050890381932,...","[2.939387691339814, 1.044183601977436, 0.74989..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -6.068, -6.4653333333333...","[0.12730715263138598, 0.1273071526313868, 0.33..."
9,SVR,"[-6.039475199305315, -5.218501962506984, -5.23...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.023141400320095, -5.8678139859322025, -6....","[-6.083876805080618, -5.953562311472967, -6.10...","[0.056295145237661735, 0.0695133761436379, 0.3..."


In [136]:
result_df.to_csv('results/Descriptors/Results_2D_padel_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_rem_RRCK.csv')

In [137]:
#2d padel descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 716)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 716)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006508 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 22083
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 661
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1274,0.2805,0.3569,0.6745,0.8266,0.8290,0.2178,0.3376,0.4667,0.5284,0.7306,0.6999
DecisionTreeRegressor,0.2779,0.3956,0.5272,0.2899,0.6327,0.6505,0.2278,0.3500,0.4773,0.5069,0.7130,0.6807
RandomForestRegressor,0.1590,0.3159,0.3987,0.5938,0.7795,0.7726,0.1937,0.3197,0.4401,0.5807,0.7769,0.7423
GradientBoostingRegressor,0.1272,0.2798,0.3566,0.6750,0.8219,0.8097,0.2096,0.3284,0.4579,0.5461,0.7413,0.7003
AdaBoostRegressor,0.1789,0.3322,0.4230,0.5429,0.7442,0.7662,0.2087,0.3475,0.4569,0.5481,0.7578,0.7298
XGBRegressor,0.1816,0.3309,0.4262,0.5360,0.7371,0.7322,0.1913,0.3174,0.4374,0.5858,0.7755,0.7419
ExtraTreesRegressor,0.1373,0.2860,0.3705,0.6492,0.8082,0.8025,0.1800,0.3083,0.4243,0.6103,0.7885,0.7551
LinearRegression,6.9923,2.1715,2.6443,-16.8653,0.0314,0.0254,3.1158,1.3136,1.7652,-5.7457,0.2298,0.2253
KNeighborsRegressor,0.1984,0.3319,0.4454,0.4932,0.7295,0.7372,0.2660,0.3581,0.5158,0.4241,0.6560,0.5752
SVR,0.1667,0.3069,0.4083,0.5741,0.7609,0.7609,0.2585,0.3784,0.5085,0.4402,0.6646,0.5955


In [138]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.149441149252347, -5.291781330520642, -5.27...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.104401663296806, -5.878059280498476, -6.2...","[-6.1677965084839705, -5.900480676987547, -6.2...","[0.11278845607688119, 0.016602310451171096, 0...."
1,DecisionTreeRegressor,"[-6.78, -4.9, -5.05, -5.86, -5.28, -5.92, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.3100000000000005, -5.76, -5.35, -6.310000...","[-6.434, -5.781999999999999, -6.008, -6.471999...","[0.3718924575734228, 0.044000000000000136, 0.6..."
2,RandomForestRegressor,"[-6.165399999999995, -5.243016666666664, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.099699999999997, -5.848299999999997, -6.2...","[-6.164826666666665, -5.940366666666663, -6.19...","[0.04595294888373514, 0.08707658187544694, 0.1..."
3,GradientBoostingRegressor,"[-6.262644698071333, -5.182628541776403, -5.06...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.025080052569988, -5.7766153988434255, -6....","[-6.185252986651207, -5.83888634300453, -6.216...","[0.16336798532960217, 0.10461993890988937, 0.1..."
4,AdaBoostRegressor,"[-6.196129032258064, -5.214374999999999, -5.29...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.868, -5.76, -6.483076923076923, -6.329166...","[-6.117171428571429, -5.89552380952381, -6.359...","[0.13332729101955149, 0.12916638358538218, 0.1..."
5,XGBRegressor,"[-6.3860435, -5.5937386, -5.077477, -5.781812,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.2857203, -5.760144, -6.370911, -6.4623394...","[-6.195822, -5.811945, -6.120382, -6.3182845, ...","[0.24399737, 0.10329227, 0.27604255, 0.3549720..."
6,ExtraTreesRegressor,"[-6.029299999999996, -5.099799999999997, -5.11...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1772999999999945, -5.7599999999999945, -6...","[-6.191654999999997, -5.820119999999996, -6.23...","[0.030267955001949818, 0.12024000000000186, 0...."
7,LinearRegression,"[-4.0, -4.0, -4.0, -10.0, -4.0, -10.0, -4.0, -...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.759999999998319, -10.0, -10.0, -4.0...","[-6.4, -6.607999999999516, -10.0, -10.0, -4.48...","[2.939387691339814, 1.696000000000242, 0.0, 0...."
8,KNeighborsRegressor,"[-5.920000000000001, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -6.068, -6.4653333333333...","[0.12730715263138598, 0.1273071526313868, 0.33..."
9,SVR,"[-6.061029074635051, -5.231092970263209, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.037484776320128, -5.8595733316251195, -6....","[-6.064738220645475, -5.908785016409529, -6.13...","[0.046025329817543655, 0.06582200061530798, 0...."


In [139]:
result_df.to_csv('results/Descriptors/Results_2D_padel_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_LVR_RRCK.csv')

In [140]:
#2d All descriptors
df_train_padel = pd.read_csv('features/Descriptors/Train_2d_padel_curated_RRCK.csv')
df_train_rdkit = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_RRCK.csv')
df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_RRCK.csv')

df_2d_train = df_train_rdkit.merge(df_train_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_train

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,15.193873,15.193873,0.130769,-1.621791,0.147476,26.802326,1216.662,...,6.109834,165.660911,1.926290,64.869552,30.340460,34.529092,38268.0,152.0,8.704,418.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,15.152762,15.152762,0.128114,-1.816236,0.134993,26.372093,1214.646,...,6.161631,165.660911,1.926290,64.869552,30.340460,34.529092,38268.0,152.0,7.824,418.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,6.131844,163.824465,1.927347,64.851242,30.333765,34.517477,37337.0,150.0,8.407,412.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,15.028142,15.028142,0.097424,-1.231351,0.116062,27.411765,1202.635,...,6.131844,164.030378,1.929769,64.941671,30.873015,34.068656,37826.0,148.0,8.690,410.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,15.068092,15.068092,0.128760,-1.611421,0.157205,27.083333,1188.608,...,6.154537,161.805840,1.926260,64.748500,30.296198,34.452302,36408.0,148.0,8.049,408.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,13.851851,13.851851,0.023647,-1.031406,0.304960,27.600000,626.799,...,6.593465,89.382185,1.986271,33.894145,15.321219,18.572926,6693.0,72.0,5.035,224.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,13.805528,13.805528,0.023859,-1.030988,0.318688,27.954545,612.772,...,6.656125,87.363105,1.985525,33.788693,15.284672,18.504021,6341.0,70.0,4.677,220.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,13.850263,13.850263,0.054388,-0.925132,0.468423,29.209302,606.809,...,6.251654,83.927240,1.951796,34.294802,15.216440,19.078362,5725.0,76.0,3.494,214.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,13.811409,13.811409,0.054806,-0.925415,0.488135,29.619048,592.782,...,6.302073,81.919106,1.950455,34.250954,15.200426,19.050528,5381.0,75.0,2.925,210.0


In [141]:
df_2d_train.to_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv', index=False)

In [142]:
df_test_padel = pd.read_csv('features/Descriptors/Test_2d_padel_curated_RRCK.csv')
df_test_rdkit = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_RRCK.csv')
df_test_mordred = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_RRCK.csv')

df_2d_test = df_test_rdkit.merge(df_test_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,15.144490,15.144490,0.113131,-1.744209,0.128505,27.116279,1218.634,...,6.181910,165.660911,1.926290,67.281618,32.752526,34.529092,38268.0,152.0,7.806,418.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,6.131844,163.824465,1.927347,64.851242,30.333765,34.517477,37337.0,150.0,8.407,412.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,14.943135,14.943135,0.009186,-1.218040,0.319427,27.500000,1095.438,...,6.364595,152.577949,1.956128,60.213907,27.820656,32.393251,28969.0,148.0,3.020,388.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,14.843366,14.843366,0.011626,-1.216334,0.396119,28.337838,1039.330,...,6.491548,144.541181,1.953259,60.018035,27.748959,32.269076,25441.0,143.0,0.955,372.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,14.986472,14.986472,0.005159,-1.183490,0.343286,27.394366,996.305,...,6.382320,139.082785,1.958912,54.418777,25.276857,29.141920,22326.0,133.0,3.033,354.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,15.005167,15.005167,0.011136,-1.161318,0.363344,26.720588,953.280,...,6.267343,132.963505,1.955346,48.807230,22.800249,26.006980,19257.0,124.0,4.656,344.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,14.929116,14.929116,0.022583,-1.148701,0.304153,26.582090,939.253,...,6.299466,131.666340,1.965169,48.993895,22.868615,26.125280,18483.0,125.0,3.909,332.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,14.694489,14.694489,0.002536,-1.355433,0.153787,21.769231,914.089,...,7.486892,131.627090,2.025032,48.756497,18.515596,22.203405,18371.0,100.0,5.070,334.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,14.686202,14.686202,0.009566,-1.067138,0.175293,25.412698,899.213,...,6.558650,124.904121,1.982605,48.311777,20.403356,24.888056,16763.0,102.0,6.435,302.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,14.705070,14.705070,0.072633,-1.128302,0.149796,22.047619,876.137,...,7.060003,127.921577,2.030501,40.639057,15.453474,22.203344,16811.0,95.0,8.012,322.0


In [143]:
df_2d_test.to_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv', index=False)

In [144]:
#2d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
# X_test = X_test.select_dtypes(include=['number'])
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 3091)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 3091)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020378 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 74261
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 2184
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1305,0.2846,0.3613,0.6665,0.8216,0.8105,0.2036,0.3249,0.4512,0.5592,0.7526,0.7014
DecisionTreeRegressor,0.2374,0.3723,0.4873,0.3934,0.6886,0.6682,0.1898,0.3121,0.4356,0.5891,0.7741,0.7628
RandomForestRegressor,0.1615,0.3171,0.4019,0.5873,0.7754,0.7674,0.1797,0.3081,0.4239,0.6109,0.7981,0.7643
GradientBoostingRegressor,0.1496,0.2984,0.3868,0.6177,0.7883,0.7856,0.1877,0.3212,0.4332,0.5936,0.7785,0.7530
AdaBoostRegressor,0.1739,0.3380,0.4170,0.5557,0.7545,0.7673,0.1908,0.3307,0.4368,0.5870,0.7911,0.7611
XGBRegressor,0.1957,0.3463,0.4423,0.5001,0.7150,0.6998,0.1770,0.3152,0.4208,0.6167,0.7969,0.7776
ExtraTreesRegressor,0.1379,0.2828,0.3714,0.6476,0.8074,0.8027,0.1860,0.3206,0.4312,0.5974,0.7802,0.7510
LinearRegression,6.9502,2.1322,2.6363,-16.7576,0.1291,0.1098,3.9329,1.5380,1.9832,-7.5149,0.1835,0.2001
KNeighborsRegressor,0.1883,0.3164,0.4339,0.5190,0.7418,0.7430,0.2550,0.3499,0.5049,0.4480,0.6747,0.6361
SVR,0.1543,0.2948,0.3928,0.6058,0.7836,0.7925,0.2433,0.3634,0.4933,0.4732,0.6890,0.6033


In [145]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.956479808031924, -5.329887295237425, -5.28...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.067159463001568, -5.8545754623055855, -6....","[-6.1992112558658246, -5.946457422101181, -6.2...","[0.09322025238605307, 0.08418488179240186, 0.0..."
1,DecisionTreeRegressor,"[-6.49, -4.9, -5.05, -5.86, -5.28, -5.86, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.35, -5.35, -6.58, -5.35, -5...","[-6.2780000000000005, -5.9399999999999995, -5....","[0.46040851425663276, 0.36000000000000015, 0.6..."
2,RandomForestRegressor,"[-6.078699999999998, -5.160299999999996, -5.18...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.094666666666666, -5.8663699999999945, -6....","[-6.157838333333331, -5.962473999999996, -6.20...","[0.0697448292786721, 0.0751474996523519, 0.179..."
3,GradientBoostingRegressor,"[-6.0483793056110455, -5.134619263897477, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.128547655151028, -5.788004697773655, -6.2...","[-6.212961514209725, -5.8253089350028, -6.0680...","[0.13834107579500182, 0.08226196809288563, 0.1..."
4,AdaBoostRegressor,"[-6.036428571428572, -5.224772727272727, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.036428571428572, -5.8975, -6.767142857142...","[-6.121808944099379, -5.9613, -6.2944217532467...","[0.08338833876663514, 0.1311631045683198, 0.47..."
5,XGBRegressor,"[-6.1849966, -5.5477777, -5.077396, -5.8497815...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.000277, -5.760592, -6.1616893, -6.232893,...","[-6.2132664, -5.784629, -6.1151605, -6.300622,...","[0.20145191, 0.04843194, 0.33927193, 0.4329340..."
6,ExtraTreesRegressor,"[-6.042199999999997, -5.003349999999994, -5.12...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.099349999999998, -5.7599999999999945, -6....","[-6.148449999999997, -5.825699999999996, -5.99...","[0.03600881836439458, 0.13140000000000213, 0.1..."
7,LinearRegression,"[-10.0, -4.0, -4.0, -10.0, -10.0, -10.0, -4.0,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.759999999999817, -10.0, -10.0, -4.0...","[-7.6133943420171395, -5.407999999999963, -10....","[2.9230597618025, 0.7039999999999815, 0.0, 2.4..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -5.885999999999999, -6.2...","[0.12730715263138598, 0.04873511168665874, 0.2..."
9,SVR,"[-6.0441649719303605, -5.193251195364211, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.001569905336069, -5.890687309842354, -5.8...","[-6.020983434444082, -5.921928235941475, -6.04...","[0.044974223465858956, 0.07009416265827476, 0...."


In [146]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_RRCK.csv')

In [147]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 2367)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 2367)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024376 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 74261
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 2184
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1305,0.2846,0.3613,0.6665,0.8216,0.8105,0.2036,0.3249,0.4512,0.5592,0.7526,0.7014
DecisionTreeRegressor,0.2534,0.3831,0.5033,0.3527,0.6763,0.6618,0.1640,0.3127,0.4050,0.6448,0.8155,0.7876
RandomForestRegressor,0.1605,0.3174,0.4007,0.5898,0.7771,0.7706,0.1754,0.3040,0.4188,0.6202,0.8038,0.7723
GradientBoostingRegressor,0.1456,0.2959,0.3815,0.6281,0.7950,0.7901,0.1859,0.3213,0.4312,0.5975,0.7820,0.7563
AdaBoostRegressor,0.1744,0.3389,0.4176,0.5544,0.7542,0.7650,0.1964,0.3483,0.4432,0.5748,0.7737,0.7289
XGBRegressor,0.1957,0.3463,0.4423,0.5001,0.7150,0.6998,0.1770,0.3152,0.4208,0.6167,0.7969,0.7776
ExtraTreesRegressor,0.1401,0.2835,0.3743,0.6420,0.8036,0.7983,0.1827,0.3222,0.4275,0.6044,0.7876,0.7570
LinearRegression,6.9502,2.1322,2.6363,-16.7576,0.1291,0.1098,3.9329,1.5380,1.9832,-7.5149,0.1835,0.2001
KNeighborsRegressor,0.1883,0.3164,0.4339,0.5190,0.7418,0.7430,0.2550,0.3499,0.5049,0.4480,0.6747,0.6361
SVR,0.1543,0.2948,0.3928,0.6058,0.7835,0.7926,0.2450,0.3642,0.4949,0.4696,0.6863,0.5984


In [148]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.956479808031924, -5.329887295237425, -5.28...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.067159463001568, -5.8545754623055855, -6....","[-6.1992112558658246, -5.946457422101181, -6.2...","[0.09322025238605307, 0.08418488179240186, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.05, -5.92, -5.28, -5.76, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.35, -5.35, -5.45, -5.35, -5...","[-6.202, -5.964, -5.635999999999999, -6.252000...","[0.40016996389034487, 0.4080000000000002, 0.58..."
2,RandomForestRegressor,"[-6.103049999999997, -5.177549999999997, -5.18...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.091699999999999, -5.884299999999996, -6.3...","[-6.179128333333331, -5.974059999999996, -6.22...","[0.08270486308420848, 0.09776217264361686, 0.2..."
3,GradientBoostingRegressor,"[-5.982793638672818, -5.152598124252644, -5.05...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.112121310369901, -5.788004697773655, -6.2...","[-6.198874449092026, -5.824753991460549, -6.07...","[0.13605617476019188, 0.0811538117655788, 0.19..."
4,AdaBoostRegressor,"[-5.883571428571429, -5.240272727272731, -5.27...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9923076923076914, -5.846052631578946, -6....","[-6.134758508158508, -5.9495604264156885, -6.4...","[0.1229656088731552, 0.17408922820331324, 0.21..."
5,XGBRegressor,"[-6.1849966, -5.5477777, -5.077396, -5.8497815...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.000277, -5.760592, -6.1616893, -6.232893,...","[-6.2132664, -5.784629, -6.1151605, -6.300622,...","[0.20145191, 0.04843194, 0.33927193, 0.4329340..."
6,ExtraTreesRegressor,"[-5.993449999999997, -5.079499999999997, -5.06...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.099049999999995, -5.7599999999999945, -5....","[-6.171009999999997, -5.828059999999995, -6.02...","[0.03866973493573598, 0.1361200000000018, 0.16..."
7,LinearRegression,"[-10.0, -4.0, -4.0, -10.0, -10.0, -10.0, -4.0,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.0, -5.7599999999998195, -10.0, -10.0, -4....","[-7.6133943420169174, -5.40799999999994, -10.0...","[2.9230597618027683, 0.7039999999999702, 0.0, ..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.1000000000000005, -5.0...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.0680000000000005, -5.885999999999999, -6.2...","[0.12730715263138598, 0.04873511168665874, 0.2..."
9,SVR,"[-6.044165042175122, -5.1932509058300935, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.001570136560791, -5.890687371157923, -5.8...","[-6.020977249388395, -5.921920643905485, -6.04...","[0.044978298174247554, 0.07009874394827098, 0...."


In [149]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_const_rem_RRCK.csv')

In [150]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1722)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1722)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012718 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51169
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1551
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1228,0.2746,0.3504,0.6862,0.8354,0.8321,0.2100,0.3269,0.4582,0.5454,0.7433,0.7101
DecisionTreeRegressor,0.2165,0.3494,0.4653,0.4469,0.7222,0.7116,0.2470,0.3638,0.4970,0.4653,0.6899,0.6761
RandomForestRegressor,0.1565,0.3145,0.3957,0.6000,0.7821,0.7757,0.1846,0.3111,0.4297,0.6003,0.7873,0.7600
GradientBoostingRegressor,0.1336,0.2807,0.3655,0.6587,0.8122,0.8003,0.1985,0.3342,0.4455,0.5702,0.7590,0.7468
AdaBoostRegressor,0.1652,0.3298,0.4065,0.5779,0.7712,0.7759,0.2031,0.3405,0.4507,0.5602,0.7639,0.7441
XGBRegressor,0.1753,0.3172,0.4187,0.5522,0.7483,0.7402,0.1745,0.3065,0.4177,0.6223,0.7990,0.7671
ExtraTreesRegressor,0.1379,0.2824,0.3713,0.6478,0.8072,0.8070,0.1882,0.3213,0.4339,0.5925,0.7772,0.7542
LinearRegression,7.4995,2.2488,2.7385,-18.1611,0.0167,0.0229,3.2726,1.4076,1.8090,-6.0853,0.2403,0.2549
KNeighborsRegressor,0.2045,0.3339,0.4522,0.4775,0.7204,0.7150,0.2721,0.3669,0.5216,0.4110,0.6475,0.5824
SVR,0.1569,0.3004,0.3961,0.5991,0.7801,0.7816,0.2543,0.3733,0.5042,0.4495,0.6708,0.6022


In [151]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.166008767477567, -5.361773205311217, -5.36...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.025936741955467, -5.828539364986243, -6.2...","[-6.134973803417346, -5.8959531209991445, -6.2...","[0.08206360796667703, 0.05163248433621551, 0.1..."
1,DecisionTreeRegressor,"[-6.78, -4.9, -5.05, -5.84, -5.19, -5.84, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.64, -5.86, -6.2, -4.9, -5.4...","[-6.180000000000001, -5.833999999999999, -5.78...","[0.5209990403062177, 0.14800000000000005, 0.55..."
2,RandomForestRegressor,"[-6.096899999999996, -5.193849999999998, -5.20...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.084799999999996, -5.8623999999999965, -6....","[-6.170634999999997, -5.9515999999999964, -6.2...","[0.05394914642513001, 0.09096058487059244, 0.1..."
3,GradientBoostingRegressor,"[-6.335128746379727, -5.22472348758606, -5.100...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.060214423452747, -5.775899183433765, -6.4...","[-6.168551184036916, -5.81464194701823, -6.150...","[0.1506635320973817, 0.0556463163806867, 0.204..."
4,AdaBoostRegressor,"[-6.1179487179487175, -5.1859259259259245, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.921999999999999, -5.854732142857142, -6.7...","[-6.138137534325769, -5.900379979769687, -6.48...","[0.11394072741634119, 0.12669915361139805, 0.2..."
5,XGBRegressor,"[-6.365387, -5.535933, -5.045331, -5.8579917, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.2491813, -5.7604127, -5.9024916, -6.36073...","[-6.1720104, -5.8169208, -5.99846, -6.3090906,...","[0.21899627, 0.11310927, 0.23714556, 0.2724734..."
6,ExtraTreesRegressor,"[-6.081499999999995, -5.029299999999995, -5.07...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.040299999999997, -5.7599999999999945, -6....","[-6.133329999999998, -5.825939999999996, -6.10...","[0.07267085798310148, 0.13188000000000175, 0.1..."
7,LinearRegression,"[-8.383595592298134, -4.0, -4.0, -10.0, -5.052...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-10.0, -5.760000000000042, -10.0, -4.0, -4.0...","[-8.8, -5.407999999999933, -10.0, -8.8, -7.382...","[2.4, 0.7039999999999663, 0.0, 2.4, 2.79025893..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-5.886000000000001, -5.885999999999999, -6.45...","[0.048735111686659234, 0.04873511168665874, 0...."
9,SVR,"[-6.059272344451147, -5.19293361224958, -5.192...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0119902291927625, -5.87988537644548, -5.9...","[-6.0209883938565785, -5.907531369272822, -6.0...","[0.045774551073771604, 0.06949702262595431, 0...."


In [152]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVR_RRCK.csv')

In [153]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 1722)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 1722)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015191 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51169
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1551
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1228,0.2746,0.3504,0.6862,0.8354,0.8321,0.2100,0.3269,0.4582,0.5454,0.7433,0.7101
DecisionTreeRegressor,0.2165,0.3494,0.4653,0.4469,0.7222,0.7116,0.2470,0.3638,0.4970,0.4653,0.6899,0.6761
RandomForestRegressor,0.1565,0.3145,0.3957,0.6000,0.7821,0.7757,0.1846,0.3111,0.4297,0.6003,0.7873,0.7600
GradientBoostingRegressor,0.1336,0.2807,0.3655,0.6587,0.8122,0.8003,0.1985,0.3342,0.4455,0.5702,0.7590,0.7468
AdaBoostRegressor,0.1652,0.3298,0.4065,0.5779,0.7712,0.7759,0.2031,0.3405,0.4507,0.5602,0.7639,0.7441
XGBRegressor,0.1753,0.3172,0.4187,0.5522,0.7483,0.7402,0.1745,0.3065,0.4177,0.6223,0.7990,0.7671
ExtraTreesRegressor,0.1379,0.2824,0.3713,0.6478,0.8072,0.8070,0.1882,0.3213,0.4339,0.5925,0.7772,0.7542
LinearRegression,7.4995,2.2488,2.7385,-18.1611,0.0167,0.0229,3.2726,1.4076,1.8090,-6.0853,0.2403,0.2549
KNeighborsRegressor,0.2045,0.3339,0.4522,0.4775,0.7204,0.7150,0.2721,0.3669,0.5216,0.4110,0.6475,0.5824
SVR,0.1569,0.3004,0.3961,0.5991,0.7801,0.7816,0.2543,0.3733,0.5042,0.4495,0.6708,0.6022


In [154]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.166008767477567, -5.361773205311217, -5.36...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.025936741955467, -5.828539364986243, -6.2...","[-6.134973803417346, -5.8959531209991445, -6.2...","[0.08206360796667703, 0.05163248433621551, 0.1..."
1,DecisionTreeRegressor,"[-6.78, -4.9, -5.05, -5.84, -5.19, -5.84, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.64, -5.86, -6.2, -4.9, -5.4...","[-6.180000000000001, -5.833999999999999, -5.78...","[0.5209990403062177, 0.14800000000000005, 0.55..."
2,RandomForestRegressor,"[-6.096899999999996, -5.193849999999998, -5.20...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.084799999999997, -5.862399999999996, -6.3...","[-6.170634999999998, -5.9515999999999964, -6.2...","[0.05394914642512989, 0.09096058487059244, 0.1..."
3,GradientBoostingRegressor,"[-6.335128746379727, -5.22472348758606, -5.100...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.060214423452747, -5.775899183433765, -6.4...","[-6.168551184036916, -5.81464194701823, -6.150...","[0.1506635320973817, 0.0556463163806867, 0.204..."
4,AdaBoostRegressor,"[-6.1179487179487175, -5.1859259259259245, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.921999999999999, -5.854732142857142, -6.7...","[-6.138137534325769, -5.900379979769687, -6.48...","[0.11394072741634119, 0.12669915361139805, 0.2..."
5,XGBRegressor,"[-6.365387, -5.535933, -5.045331, -5.8579917, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.2491813, -5.7604127, -5.9024916, -6.36073...","[-6.1720104, -5.8169208, -5.99846, -6.3090906,...","[0.21899627, 0.11310927, 0.23714556, 0.2724734..."
6,ExtraTreesRegressor,"[-6.081499999999997, -5.029299999999995, -5.07...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.040299999999995, -5.7599999999999945, -6....","[-6.133329999999998, -5.825939999999996, -6.10...","[0.07267085798310204, 0.13188000000000175, 0.1..."
7,LinearRegression,"[-8.383595592298134, -4.0, -4.0, -10.0, -5.052...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-10.0, -5.760000000000042, -10.0, -4.0, -4.0...","[-8.8, -5.407999999999933, -10.0, -8.8, -7.382...","[2.4, 0.7039999999999663, 0.0, 2.4, 2.79025893..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-5.886000000000001, -5.885999999999999, -6.45...","[0.048735111686659234, 0.04873511168665874, 0...."
9,SVR,"[-6.059272344451147, -5.19293361224958, -5.192...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0119902291927625, -5.87988537644548, -5.9...","[-6.0209883938565785, -5.907531369272822, -6.0...","[0.045774551073771604, 0.06949702262595431, 0...."


In [155]:
def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [156]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [157]:
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
X_train = df_train[selected_features] 
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test =df_test.dropna()
X_test =  df_test[X_train.columns]
y_test =  df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 202)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 202)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002857 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6268
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 183
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1510,0.2996,0.3886,0.6142,0.7860,0.7823,0.2119,0.3307,0.4603,0.5413,0.7404,0.6971
DecisionTreeRegressor,0.2303,0.3690,0.4799,0.4115,0.6833,0.6592,0.2044,0.3530,0.4521,0.5575,0.7531,0.6995
RandomForestRegressor,0.1498,0.3093,0.3871,0.6172,0.7962,0.7882,0.1961,0.3234,0.4428,0.5755,0.7761,0.7373
GradientBoostingRegressor,0.1242,0.2760,0.3524,0.6828,0.8268,0.8178,0.2129,0.3349,0.4614,0.5390,0.7430,0.7205
AdaBoostRegressor,0.1749,0.3417,0.4182,0.5532,0.7622,0.7613,0.2132,0.3468,0.4618,0.5383,0.7533,0.7067
XGBRegressor,0.1588,0.3118,0.3985,0.5943,0.7747,0.7630,0.2015,0.3398,0.4489,0.5637,0.7594,0.7500
ExtraTreesRegressor,0.1306,0.2799,0.3614,0.6664,0.8218,0.8131,0.1876,0.3188,0.4331,0.5938,0.7830,0.7434
LinearRegression,10.4546,2.7676,3.2334,-25.7115,0.1522,0.1552,4.0589,1.5888,2.0147,-7.7876,-0.0348,0.0192
KNeighborsRegressor,0.2006,0.3248,0.4479,0.4875,0.7281,0.7081,0.2555,0.3424,0.5055,0.4468,0.6843,0.6538
SVR,0.1468,0.2984,0.3832,0.6249,0.7943,0.7922,0.2432,0.3661,0.4932,0.4735,0.6893,0.5962


In [158]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.762644156761018, -5.3864500504045285, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.01813748508959, -5.995963769945327, -6.01...","[-6.1649685712020545, -6.09558189875182, -5.94...","[0.1297536178405518, 0.1456404661777826, 0.086..."
1,DecisionTreeRegressor,"[-6.13, -5.27, -5.05, -6.15, -5.1, -6.15, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.76, -5.76, -5.4, -6.7, -7.0, -4.9, -5.45,...","[-5.888, -5.781999999999999, -5.922, -6.502, -...","[0.1361469794009401, 0.044000000000000136, 0.6..."
2,RandomForestRegressor,"[-5.998749999999998, -5.30870476190476, -5.286...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0410299999999975, -5.825299999999996, -6....","[-6.078980999999998, -5.910359999999996, -6.13...","[0.04935062759479431, 0.09241256624507432, 0.1..."
3,GradientBoostingRegressor,"[-6.16383714820719, -5.210278294232454, -5.161...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.102042638290421, -5.800114489412835, -6.0...","[-5.985473845454921, -5.8289484600691965, -5.9...","[0.0938097664700221, 0.066789360145513, 0.0946..."
4,AdaBoostRegressor,"[-5.873333333333334, -5.258750000000001, -5.42...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.879636363636366, -5.79, -6.60499999999999...","[-6.007028293135436, -5.886619047619048, -6.19...","[0.09567169847171206, 0.10923382314060878, 0.3..."
5,XGBRegressor,"[-6.2127857, -5.4718657, -5.178146, -5.9181647...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.8619657, -5.7608423, -5.754847, -6.369037...","[-5.964306, -5.848433, -5.986495, -6.346134, -...","[0.15941793, 0.1753762, 0.27503854, 0.3724764,..."
6,ExtraTreesRegressor,"[-5.965149999999997, -5.076699999999999, -5.10...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.090399999999996, -5.7599999999999945, -6....","[-6.115079999999997, -5.808639999999995, -6.00...","[0.05393987022602137, 0.09728000000000243, 0.0..."
7,LinearRegression,"[-10.0, -4.0, -4.0, -10.0, -4.0, -4.0, -10.0, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-10.0, -5.759971439838409, -4.0, -4.0, -4.0,...","[-7.6, -6.60799867701528, -5.2, -5.2, -5.2, -6...","[2.939387691339814, 1.6960006615695524, 2.4, 2..."
8,KNeighborsRegressor,"[-5.919999999999999, -5.03, -5.03, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-6.068, -5.885999999999999, -6.06266666666666...","[0.1273071526313868, 0.04873511168665874, 0.21..."
9,SVR,"[-6.114124015893531, -5.06603668693899, -5.140...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.06153185145178, -5.8599607721581135, -5.8...","[-6.100755227123787, -5.888191435521163, -6.01...","[0.03143304383058189, 0.056286817369880435, 0...."


In [159]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_remove_corr_features_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVRremove_corr_features_RRCK.csv')

In [160]:
#3d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_RRCK.csv')
df_train = df_train.fillna(0)
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_RRCK.csv')
df_test = df_test.fillna(0)
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 11)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 11)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000512 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 429
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 11
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-4.629674134367108


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2976,0.4625,0.5455,0.2397,0.4923,0.4824,0.3384,0.4897,0.5817,0.2673,0.5187,0.4955
DecisionTreeRegressor,0.5844,0.5918,0.7645,-0.4931,0.2268,0.1942,0.2760,0.4306,0.5254,0.4024,0.6383,0.6572
RandomForestRegressor,0.2950,0.4587,0.5431,0.2464,0.5034,0.5028,0.2944,0.4534,0.5425,0.3627,0.6033,0.5849
GradientBoostingRegressor,0.3055,0.4529,0.5527,0.2194,0.5027,0.4769,0.3218,0.4789,0.5672,0.3034,0.5598,0.5263
AdaBoostRegressor,0.3041,0.4559,0.5515,0.2229,0.4784,0.4939,0.3195,0.4769,0.5653,0.3082,0.5616,0.5605
XGBRegressor,0.3628,0.5050,0.6023,0.0731,0.4172,0.4024,0.3356,0.4820,0.5793,0.2735,0.5472,0.5185
ExtraTreesRegressor,0.3035,0.4606,0.5509,0.2246,0.4963,0.4847,0.2860,0.4347,0.5348,0.3808,0.6202,0.5722
LinearRegression,0.4094,0.5435,0.6398,-0.0459,0.2816,0.2698,0.3294,0.4809,0.5739,0.2868,0.5401,0.4913
KNeighborsRegressor,0.3663,0.4932,0.6052,0.0641,0.3845,0.3861,0.3306,0.4562,0.5749,0.2843,0.5418,0.5356
SVR,0.3171,0.4719,0.5631,0.1899,0.4590,0.4386,0.3068,0.4554,0.5539,0.3358,0.6031,0.5924


In [161]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.047601908949891, -5.4662032617778875, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.981803340891515, -6.100766379936854, -5.8...","[-5.932429936898539, -6.150428355067814, -5.81...","[0.1588746613252465, 0.17335915787740477, 0.10..."
1,DecisionTreeRegressor,"[-6.78, -5.28, -5.27, -5.27, -5.57, -5.84, -5....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.78, -5.75, -6.46, -5.05, -5.57, -5.57, -5...","[-6.058, -5.8340000000000005, -6.2799999999999...","[0.3861295119516249, 0.33434114314573965, 0.49..."
2,RandomForestRegressor,"[-6.125799999999997, -5.505399999999995, -5.55...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.945999999999999, -5.991799999999997, -6.0...","[-6.081039999999999, -6.26476, -6.051399999999...","[0.10204919597919408, 0.15152167633708508, 0.0..."
3,GradientBoostingRegressor,"[-6.566279719922049, -5.577695681577152, -5.48...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.258864384582683, -5.906829534161897, -6.0...","[-6.083266757376705, -6.182873260790592, -5.93...","[0.08874166043867254, 0.2516021456153037, 0.12..."
4,AdaBoostRegressor,"[-6.077875000000001, -5.630000000000001, -5.60...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.077875000000001, -6.066666666666666, -5.8...","[-6.082270151515151, -6.20393778052933, -5.932...","[0.07278793369096942, 0.09615663512591754, 0.0..."
5,XGBRegressor,"[-5.852806, -5.454952, -5.463698, -5.0673847, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.711476, -6.284354, -6.010225, -5.5982947,...","[-5.965815, -6.4077387, -6.14445, -5.843674, -...","[0.17992914, 0.20862709, 0.16590866, 0.2454823..."
6,ExtraTreesRegressor,"[-6.072699999999997, -5.688699999999995, -5.48...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.893399999999998, -5.957899999999999, -6.0...","[-6.10906, -6.19118, -6.094519999999998, -5.92...","[0.1344381136434166, 0.15451365505999815, 0.10..."
7,LinearRegression,"[-6.218945927191835, -5.891954466675243, -5.72...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.208283974027019, -6.178572452614679, -5.9...","[-6.211178509376927, -6.223322865922953, -5.99...","[0.15345970548334686, 0.10323458520902304, 0.0..."
8,KNeighborsRegressor,"[-6.096666666666667, -5.6933333333333325, -5.3...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.096666666666667, -6.096666666666667, -5.6...","[-6.201333333333333, -6.201333333333333, -5.71...","[0.16151504917843112, 0.16151504917843112, 0.1..."
9,SVR,"[-6.109846647221274, -5.3753558983225815, -5.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.808558499499437, -6.0448669330887554, -5....","[-6.047085838927642, -6.18800277316345, -5.754...","[0.1885288472993148, 0.13099463246638887, 0.12..."


In [162]:
result_df.to_csv('results/Descriptors/Results_3D_RDKit_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_RDKit_desc_RRCK.csv')

In [163]:
#3d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_curated_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_curated_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 431)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 431)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16793
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 431
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2056,0.3826,0.4534,0.4747,0.6905,0.6710,0.2867,0.4143,0.5355,0.3792,0.6201,0.5641
DecisionTreeRegressor,0.3773,0.4585,0.6143,0.0359,0.4867,0.4606,0.3794,0.4902,0.6160,0.1785,0.4444,0.4279
RandomForestRegressor,0.2357,0.4054,0.4855,0.3978,0.6349,0.6011,0.2793,0.4269,0.5285,0.3952,0.6383,0.5966
GradientBoostingRegressor,0.1969,0.3677,0.4438,0.4968,0.7065,0.6720,0.2648,0.4006,0.5145,0.4268,0.6550,0.6021
AdaBoostRegressor,0.2208,0.3874,0.4699,0.4359,0.6665,0.6309,0.2946,0.4373,0.5428,0.3622,0.6038,0.5836
XGBRegressor,0.2362,0.3888,0.4860,0.3964,0.6420,0.6221,0.2736,0.4114,0.5231,0.4077,0.6420,0.6019
ExtraTreesRegressor,0.1840,0.3508,0.4290,0.5298,0.7372,0.7149,0.2575,0.4120,0.5074,0.4425,0.6763,0.6255
LinearRegression,0.8931,0.7407,0.9450,-1.2818,0.3340,0.2879,0.5618,0.5337,0.7495,-0.2163,0.4251,0.4057
KNeighborsRegressor,0.2402,0.3903,0.4901,0.3862,0.6275,0.6282,0.3968,0.5030,0.6299,0.1409,0.4447,0.4387
SVR,0.2171,0.3827,0.4659,0.4453,0.6713,0.6461,0.2915,0.4419,0.5399,0.3688,0.6261,0.5879


In [164]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.012581878433041, -5.783928738579369, -5.81...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.019713301142322, -6.021761976308937, -6.4...","[-6.171448420795714, -6.113939567722629, -6.32...","[0.11099405749032287, 0.05321785018247872, 0.1..."
1,DecisionTreeRegressor,"[-5.45, -5.45, -5.05, -6.93, -6.09, -6.09, -6....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.45, -5.76, -5.4, -6.93, -5.74, -6.34, -5....","[-6.132, -6.234, -6.174, -6.098000000000001, -...","[0.48188795378178945, 0.41720977936764636, 0.5..."
2,RandomForestRegressor,"[-6.03855, -6.001499999999998, -5.915799999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.067349999999998, -6.063299999999997, -6.4...","[-6.177469999999997, -6.1265199999999975, -6.4...","[0.07430472124972896, 0.06056865195792329, 0.1..."
3,GradientBoostingRegressor,"[-6.1387583261352825, -6.069792551580672, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.884615790027, -5.989171943238348, -6.5747...","[-6.101590961228779, -6.1872660184986366, -6.4...","[0.15671490247733302, 0.16557495483684367, 0.1..."
4,AdaBoostRegressor,"[-6.256764705882353, -6.122400000000002, -5.68...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -6.116904761904762, -6.695, -6.272500...","[-6.144386810762545, -6.1169678210678216, -6.5...","[0.12400912986300205, 0.15144363035218408, 0.0..."
5,XGBRegressor,"[-6.0075407, -5.3682575, -5.49762, -5.861085, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1067615, -5.834629, -6.584834, -6.395494,...","[-6.0878134, -6.0461817, -6.6053705, -6.133697...","[0.13484721, 0.2031243, 0.09631659, 0.2611019,..."
6,ExtraTreesRegressor,"[-6.320299999999998, -5.843700000000001, -5.52...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.081199999999995, -5.972999999999997, -6.7...","[-6.100199999999996, -6.0665799999999965, -6.5...","[0.06021272290803759, 0.1230989423187705, 0.16..."
7,LinearRegression,"[-6.151367180170692, -7.215098350106491, -7.14...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.524403545666976, -5.804168022711105, -7.0...","[-4.688673776024084, -5.873343386867058, -6.42...","[0.6769271852599065, 0.7517772496054056, 0.662..."
8,KNeighborsRegressor,"[-6.146666666666667, -5.916666666666667, -5.26...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.223333333333334, -5.919999999999999, -6.3...","[-6.144666666666667, -6.0233333333333325, -6.4...","[0.1384132299392738, 0.14883249346534266, 0.30..."
9,SVR,"[-6.185237446867153, -6.12137795690565, -5.784...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.832721034558073, -5.954955872368684, -6.3...","[-5.823736492767425, -6.014941652192598, -6.25...","[0.07768546091921767, 0.07982157988741227, 0.1..."


In [165]:
result_df.to_csv('results/Descriptors/Results_3D_padel_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_padel_desc_RRCK.csv')

In [166]:
df_train_rdkit = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_RRCK.csv')
df_train_rdkit = df_train_rdkit.fillna(0)
df_train_padel = pd.read_csv('features/Descriptors/Train_3d_padel_curated_RRCK.csv')

df_3d_descriptors = df_train_rdkit.merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,25557.550768,38719.861789,59078.304374,0.432605,0.655399,7.119995,0.000026,...,0.555662,0.361731,0.544173,0.495228,0.417788,55.253242,844.996910,3701.072059,0.376090,1.457189
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,25195.527353,38098.222481,54940.535676,0.458596,0.693445,6.976408,0.000028,...,0.514250,0.383940,0.446697,0.436831,0.348132,50.660858,741.432722,3405.746601,0.347284,1.231660
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,24627.941709,37770.489101,52166.145232,0.472106,0.724042,6.901496,0.000029,...,0.584567,0.352614,0.534974,0.490552,0.346062,57.530851,877.093731,3400.242320,0.405772,1.371588
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,27053.583076,35398.003591,54685.502768,0.494712,0.647301,6.978552,0.000024,...,0.540927,0.396766,0.520911,0.514156,0.415753,54.389916,807.742283,3013.737615,0.406540,1.450820
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,25621.505581,37274.558430,58032.092029,0.441506,0.642309,7.132297,0.000025,...,0.572679,0.345721,0.523047,0.544280,0.438872,53.667187,786.079599,3336.947160,0.377600,1.506199
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,7624.940741,7922.483693,14006.527516,0.544385,0.565628,4.855440,0.000074,...,0.558285,0.321297,0.508700,0.556253,0.295744,28.520547,232.063263,761.687250,0.337428,1.360698
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,6232.953683,8749.353631,12635.210560,0.493300,0.692458,4.747094,0.000111,...,0.492893,0.353000,0.440062,0.551455,0.349133,25.227357,193.693844,649.413156,0.268840,1.340650
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,5787.868928,7241.500975,11269.076352,0.513606,0.642599,4.474538,0.000111,...,0.540801,0.390620,0.531371,0.520052,0.472508,25.525900,179.262689,445.738032,0.397132,1.523931
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,5518.344689,6698.311541,10323.461033,0.534544,0.648844,4.360292,0.000118,...,0.434247,0.397872,0.462647,0.447369,0.419043,21.042045,138.352372,429.631984,0.248178,1.329059


In [167]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [168]:
df_3d_descriptors.to_csv('features/Descriptors/Train_3d_all_descriptors_RRCK.csv', index=False)

In [169]:
df_test_rdkit = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_RRCK.csv')
df_test_rdkit = df_test_rdkit.fillna(0)
df_test_padel = pd.read_csv('features/Descriptors/Test_3d_padel_curated_RRCK.csv')

df_3d_descriptors = df_test_rdkit.merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,25465.772297,41635.236357,57764.581274,0.440854,0.720774,7.157638,0.000028,...,0.553179,0.378046,0.621744,0.592669,0.362613,61.295253,1026.337021,4399.866189,0.396838,1.577026
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,23425.027477,36819.014519,51958.191749,0.450844,0.708628,6.829971,0.000030,...,0.593681,0.333277,0.516828,0.508872,0.413331,55.396709,814.970696,3327.239818,0.390522,1.439031
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,19228.307860,26584.788950,41557.971168,0.462686,0.639704,6.315023,0.000033,...,0.495314,0.432467,0.491430,0.560578,0.414926,48.550470,662.854459,2481.778448,0.391672,1.466935
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,18709.773342,27134.585978,41923.420928,0.446285,0.647242,6.497942,0.000035,...,0.550835,0.351295,0.412793,0.478748,0.389842,40.222215,455.898864,1728.491712,0.353195,1.281383
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,15494.460333,25480.222143,35133.289000,0.441019,0.725244,6.180220,0.000047,...,0.446396,0.420408,0.451162,0.484828,0.337762,38.674180,453.379291,1937.981514,0.300205,1.273752
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,12652.784435,21781.115461,29110.712319,0.434644,0.748217,5.773167,0.000059,...,0.564393,0.353966,0.455681,0.504722,0.490337,42.046102,485.726415,1740.123264,0.377539,1.450740
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,14906.294395,19726.009637,30917.986108,0.482124,0.638011,5.907191,0.000043,...,0.512123,0.378645,0.502782,0.546703,0.346323,37.901202,418.328104,1609.458674,0.336152,1.395808
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,14125.144457,25108.706249,32442.299466,0.435393,0.773950,6.261496,0.000055,...,0.527378,0.345681,0.461735,0.511862,0.384095,37.160707,404.790585,1629.498650,0.309589,1.357693
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,12577.621436,20415.610939,26519.217510,0.474283,0.769842,5.752513,0.000061,...,0.433018,0.413344,0.440081,0.422826,0.386581,36.776762,417.957514,1822.583332,0.269542,1.249488
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,11732.937028,20116.819627,25872.348901,0.453493,0.777541,5.739447,0.000066,...,0.538422,0.395997,0.537446,0.535345,0.393729,37.635410,388.799321,1171.822077,0.401629,1.466520


In [170]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [171]:
df_3d_descriptors.to_csv('features/Descriptors/Test_3d_all_descriptors_RRCK.csv', index=False)

In [172]:
#3d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_RRCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_3dall = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_3dall, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 442)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 442)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17222
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 442
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2016,0.3780,0.4489,0.4850,0.6978,0.6743,0.2807,0.4042,0.5298,0.3923,0.6289,0.5788
DecisionTreeRegressor,0.4269,0.4981,0.6534,-0.0907,0.4322,0.4241,0.3593,0.4654,0.5994,0.2222,0.4888,0.4767
RandomForestRegressor,0.2280,0.3973,0.4775,0.4175,0.6518,0.6167,0.2792,0.4290,0.5284,0.3954,0.6399,0.5767
GradientBoostingRegressor,0.1977,0.3694,0.4446,0.4949,0.7039,0.6741,0.2720,0.4056,0.5215,0.4112,0.6419,0.6052
AdaBoostRegressor,0.2037,0.3868,0.4513,0.4796,0.7029,0.6829,0.2807,0.4229,0.5298,0.3922,0.6346,0.6104
XGBRegressor,0.2491,0.4114,0.4991,0.3635,0.6146,0.5918,0.2966,0.4328,0.5446,0.3579,0.5985,0.5760
ExtraTreesRegressor,0.1885,0.3590,0.4342,0.5183,0.7284,0.7100,0.2542,0.4071,0.5042,0.4496,0.6860,0.6326
LinearRegression,0.8460,0.7218,0.9198,-1.1616,0.3496,0.3042,0.5022,0.5274,0.7086,-0.0872,0.4726,0.4591
KNeighborsRegressor,0.2549,0.4119,0.5049,0.3487,0.5993,0.5951,0.3999,0.4935,0.6323,0.1343,0.4403,0.4438
SVR,0.2122,0.3781,0.4607,0.4578,0.6811,0.6650,0.2882,0.4373,0.5368,0.3761,0.6274,0.5812


In [173]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.093728775500042, -5.830563332601942, -5.80...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.194450518894163, -6.138748100682937, -6.3...","[-6.2614601242768755, -6.183131907262282, -6.2...","[0.07624835849960908, 0.10478071489607321, 0.1..."
1,DecisionTreeRegressor,"[-5.45, -5.57, -5.05, -6.58, -6.09, -6.09, -6....",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.64, -5.76, -5.4, -6.93, -5.57, -6.12, -5....","[-6.063999999999999, -6.324, -6.438, -6.022, -...","[0.3566847347448445, 0.4625840464175134, 0.538..."
2,RandomForestRegressor,"[-6.001399999999998, -5.962350000000002, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0708999999999955, -6.064099999999997, -6....","[-6.145789999999996, -6.159819999999998, -6.37...","[0.06697341562142442, 0.11110455256199117, 0.1..."
3,GradientBoostingRegressor,"[-6.070813005659661, -6.085662229661206, -5.39...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.1040026966001415, -6.04373336027183, -6.5...","[-6.153155917537471, -6.187368859861542, -6.42...","[0.041538099101826646, 0.13956881751685832, 0...."
4,AdaBoostRegressor,"[-6.316122448979591, -5.7728, -5.76, -5.686086...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -6.128571428571428, -6.52422222222222...","[-6.204969696969697, -6.112745864661654, -6.34...","[0.0929075984778041, 0.1359651804864361, 0.121..."
5,XGBRegressor,"[-6.039057, -5.535313, -5.426241, -5.7300944, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0903673, -5.8134084, -6.443205, -6.271086...","[-6.067761, -6.1768947, -6.495488, -6.088774, ...","[0.2263251, 0.30221003, 0.08803449, 0.33393568..."
6,ExtraTreesRegressor,"[-6.122200000000001, -5.889749999999998, -5.46...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.046099999999996, -6.028799999999997, -6.5...","[-6.134569999999997, -6.065139999999998, -6.50...","[0.06375184389490321, 0.1411234863514927, 0.08..."
7,LinearRegression,"[-5.016712522550303, -7.483133136807219, -6.37...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.000194782142677, -6.399596744701813, -7.2...","[-5.354271728932112, -6.276781054556951, -6.57...","[1.1275780320755424, 0.8961153834287172, 0.520..."
8,KNeighborsRegressor,"[-6.21, -5.8, -5.18, -5.543333333333334, -5.87...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.223333333333334, -5.919999999999999, -6.3...","[-6.144666666666667, -6.0233333333333325, -6.2...","[0.1384132299392738, 0.14883249346534266, 0.09..."
9,SVR,"[-6.21307742930723, -6.12718350616473, -5.7379...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.820173336855668, -5.962726615378672, -6.3...","[-5.82681072666268, -6.022574287225498, -6.255...","[0.08899796460351243, 0.08702058615253269, 0.1..."


In [174]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_RRCK.csv')

In [175]:
#3d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_3dall = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_3dall, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 442)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 442)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015037 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17222
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 442
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2016,0.3780,0.4489,0.4850,0.6978,0.6743,0.2807,0.4042,0.5298,0.3923,0.6289,0.5788
DecisionTreeRegressor,0.4269,0.4981,0.6534,-0.0907,0.4322,0.4241,0.3593,0.4654,0.5994,0.2222,0.4888,0.4767
RandomForestRegressor,0.2280,0.3973,0.4775,0.4175,0.6518,0.6167,0.2792,0.4290,0.5284,0.3954,0.6399,0.5767
GradientBoostingRegressor,0.1977,0.3694,0.4446,0.4949,0.7039,0.6741,0.2720,0.4056,0.5215,0.4112,0.6419,0.6052
AdaBoostRegressor,0.2037,0.3868,0.4513,0.4796,0.7029,0.6829,0.2807,0.4229,0.5298,0.3922,0.6346,0.6104
XGBRegressor,0.2491,0.4114,0.4991,0.3635,0.6146,0.5918,0.2966,0.4328,0.5446,0.3579,0.5985,0.5760
ExtraTreesRegressor,0.1885,0.3590,0.4342,0.5183,0.7284,0.7100,0.2542,0.4071,0.5042,0.4496,0.6860,0.6326
LinearRegression,0.8460,0.7218,0.9198,-1.1616,0.3496,0.3042,0.5022,0.5274,0.7086,-0.0872,0.4726,0.4591
KNeighborsRegressor,0.2549,0.4119,0.5049,0.3487,0.5993,0.5951,0.3999,0.4935,0.6323,0.1343,0.4403,0.4438
SVR,0.2122,0.3781,0.4607,0.4578,0.6811,0.6650,0.2882,0.4373,0.5368,0.3761,0.6274,0.5812


In [176]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_const_rem_RRCK.csv')

In [177]:
#3d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 375)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 375)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005330 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14613
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 375
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-2.774644547431081


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1933,0.3726,0.4396,0.5062,0.7136,0.6883,0.2915,0.4144,0.5399,0.3689,0.6076,0.5769
DecisionTreeRegressor,0.3812,0.4581,0.6174,0.0260,0.5100,0.4940,0.4360,0.5241,0.6603,0.0560,0.3700,0.4083
RandomForestRegressor,0.2250,0.3981,0.4743,0.4251,0.6593,0.6167,0.2790,0.4276,0.5282,0.3960,0.6402,0.5916
GradientBoostingRegressor,0.1770,0.3425,0.4207,0.5479,0.7414,0.7199,0.2747,0.4025,0.5241,0.4052,0.6368,0.5937
AdaBoostRegressor,0.2120,0.3880,0.4604,0.4584,0.6867,0.6618,0.3113,0.4386,0.5579,0.3261,0.5715,0.5505
XGBRegressor,0.2459,0.4019,0.4959,0.3718,0.6245,0.5965,0.3040,0.4361,0.5514,0.3418,0.5848,0.5354
ExtraTreesRegressor,0.1803,0.3501,0.4246,0.5394,0.7453,0.7287,0.2567,0.4103,0.5066,0.4443,0.6773,0.6330
LinearRegression,0.9838,0.7818,0.9919,-1.5136,0.2820,0.2436,0.6006,0.5956,0.7750,-0.3002,0.4048,0.4351
KNeighborsRegressor,0.2296,0.3893,0.4792,0.4133,0.6495,0.6371,0.3670,0.4670,0.6058,0.2054,0.4920,0.4937
SVR,0.2074,0.3704,0.4554,0.4701,0.6908,0.6700,0.2851,0.4288,0.5339,0.3828,0.6382,0.5956


In [178]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.164326444793451, -5.767772011056359, -5.71...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.275974303283605, -6.155134023862533, -6.2...","[-6.3181531710769265, -6.213479942999202, -6.1...","[0.060062906507671314, 0.1083868423658656, 0.0..."
1,DecisionTreeRegressor,"[-6.42, -5.57, -5.35, -6.4, -6.4, -6.4, -6.4, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.13, -5.76, -6.13, -5.66, -5.57, -6.89, -6...","[-6.194000000000001, -6.013999999999999, -6.38...","[0.4037623063139997, 0.35336100520572455, 0.54..."
2,RandomForestRegressor,"[-6.0952499999999965, -5.915649999999997, -5.7...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0639999999999965, -6.107999999999997, -6....","[-6.170149999999997, -6.172899999999997, -6.37...","[0.06895057650230353, 0.07790103978766887, 0.0..."
3,GradientBoostingRegressor,"[-6.024585344533043, -6.076680377855003, -5.33...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.994850386674385, -6.010646696573272, -6.4...","[-6.166896295191497, -6.194655122579839, -6.43...","[0.15945565786788338, 0.11544478649560083, 0.0..."
4,AdaBoostRegressor,"[-6.248, -6.152666666666666, -5.63363636363636...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.083970588235294, -5.942499999999999, -6.2...","[-6.260494117647058, -6.084748717948718, -6.39...","[0.12161065582010695, 0.16731977294041206, 0.1..."
5,XGBRegressor,"[-5.954464, -5.5036564, -5.7981195, -5.498722,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.041051, -5.779446, -6.5284953, -6.4577603...","[-6.125024, -6.151274, -6.3457, -6.028202, -5....","[0.25534537, 0.18998866, 0.2342425, 0.34339672..."
6,ExtraTreesRegressor,"[-6.126799999999996, -5.8597, -5.3172999999999...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.167599999999994, -5.990199999999995, -6.6...","[-6.131399999999995, -6.108179999999996, -6.55...","[0.13164262227713291, 0.16910077941866478, 0.1..."
7,LinearRegression,"[-4.656112353546378, -7.624666029433395, -6.38...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.687695699697522, -6.291944969207815, -7.3...","[-5.977692626536745, -6.26100528129807, -6.757...","[1.6393770185091532, 1.4659291615821795, 0.379..."
8,KNeighborsRegressor,"[-5.73, -5.8, -5.266666666666667, -5.543333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.223333333333334, -6.136666666666667, -6.2...","[-6.205333333333334, -6.170666666666667, -6.32...","[0.04203702072115871, 0.04818482933224732, 0.1..."
9,SVR,"[-6.079606362798325, -6.105625156918078, -5.65...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.86316242013125, -5.965225877487537, -6.26...","[-5.851564077824125, -6.048966366041829, -6.18...","[0.075013002833377, 0.09496840458577636, 0.156..."


In [179]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_LVR_RRCK.csv')

In [180]:
#2d and 3d descriptors all
df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_RRCK.csv')
df_train_2d
df_train_3d = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_RRCK.csv')
df_train_3d

df_2d_3d_train = df_train_2d.merge(df_train_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_train.to_csv('features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv', index=False)
df_2d_3d_train

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,15.193873,15.193873,0.130769,-1.621791,0.147476,26.802326,1216.662,...,0.555662,0.361731,0.544173,0.495228,0.417788,55.253242,844.996910,3701.072059,0.376090,1.457189
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,15.152762,15.152762,0.128114,-1.816236,0.134993,26.372093,1214.646,...,0.514250,0.383940,0.446697,0.436831,0.348132,50.660858,741.432722,3405.746601,0.347284,1.231660
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,0.584567,0.352614,0.534974,0.490552,0.346062,57.530851,877.093731,3400.242320,0.405772,1.371588
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,15.028142,15.028142,0.097424,-1.231351,0.116062,27.411765,1202.635,...,0.540927,0.396766,0.520911,0.514156,0.415753,54.389916,807.742283,3013.737615,0.406540,1.450820
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,15.068092,15.068092,0.128760,-1.611421,0.157205,27.083333,1188.608,...,0.572679,0.345721,0.523047,0.544280,0.438872,53.667187,786.079599,3336.947160,0.377600,1.506199
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,13.851851,13.851851,0.023647,-1.031406,0.304960,27.600000,626.799,...,0.558285,0.321297,0.508700,0.556253,0.295744,28.520547,232.063263,761.687250,0.337428,1.360698
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,13.805528,13.805528,0.023859,-1.030988,0.318688,27.954545,612.772,...,0.492893,0.353000,0.440062,0.551455,0.349133,25.227357,193.693844,649.413156,0.268840,1.340650
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,13.850263,13.850263,0.054388,-0.925132,0.468423,29.209302,606.809,...,0.540801,0.390620,0.531371,0.520052,0.472508,25.525900,179.262689,445.738032,0.397132,1.523931
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,13.811409,13.811409,0.054806,-0.925415,0.488135,29.619048,592.782,...,0.434247,0.397872,0.462647,0.447369,0.419043,21.042045,138.352372,429.631984,0.248178,1.329059


In [181]:
df_test_2d = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_RRCK.csv')
df_test_2d
df_test_3d = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_RRCK.csv')
df_test_3d

df_2d_3d_test = df_test_2d.merge(df_test_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_test.to_csv('features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv', index=False)
df_2d_3d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,15.144490,15.144490,0.113131,-1.744209,0.128505,27.116279,1218.634,...,0.553179,0.378046,0.621744,0.592669,0.362613,61.295253,1026.337021,4399.866189,0.396838,1.577026
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,0.593681,0.333277,0.516828,0.508872,0.413331,55.396709,814.970696,3327.239818,0.390522,1.439031
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,14.943135,14.943135,0.009186,-1.218040,0.319427,27.500000,1095.438,...,0.495314,0.432467,0.491430,0.560578,0.414926,48.550470,662.854459,2481.778448,0.391672,1.466935
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,14.843366,14.843366,0.011626,-1.216334,0.396119,28.337838,1039.330,...,0.550835,0.351295,0.412793,0.478748,0.389842,40.222215,455.898864,1728.491712,0.353195,1.281383
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,14.986472,14.986472,0.005159,-1.183490,0.343286,27.394366,996.305,...,0.446396,0.420408,0.451162,0.484828,0.337762,38.674180,453.379291,1937.981514,0.300205,1.273752
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,15.005167,15.005167,0.011136,-1.161318,0.363344,26.720588,953.280,...,0.564393,0.353966,0.455681,0.504722,0.490337,42.046102,485.726415,1740.123264,0.377539,1.450740
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,14.929116,14.929116,0.022583,-1.148701,0.304153,26.582090,939.253,...,0.512123,0.378645,0.502782,0.546703,0.346323,37.901202,418.328104,1609.458674,0.336152,1.395808
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,14.694489,14.694489,0.002536,-1.355433,0.153787,21.769231,914.089,...,0.527378,0.345681,0.461735,0.511862,0.384095,37.160707,404.790585,1629.498650,0.309589,1.357693
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,14.686202,14.686202,0.009566,-1.067138,0.175293,25.412698,899.213,...,0.433018,0.413344,0.440081,0.422826,0.386581,36.776762,417.957514,1822.583332,0.269542,1.249488
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,14.705070,14.705070,0.072633,-1.128302,0.149796,22.047619,876.137,...,0.538422,0.395997,0.537446,0.535345,0.393729,37.635410,388.799321,1171.822077,0.401629,1.466520


In [182]:
#All 2d and 3d descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 3533)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 3533)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030221 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 91483
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 2626
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1428,0.3041,0.3779,0.6350,0.8005,0.7939,0.2134,0.3359,0.4619,0.5380,0.7441,0.7286
DecisionTreeRegressor,0.2957,0.4214,0.5438,0.2446,0.6181,0.5683,0.2384,0.3765,0.4883,0.4838,0.7004,0.6559
RandomForestRegressor,0.1722,0.3362,0.4149,0.5601,0.7638,0.7523,0.2100,0.3342,0.4582,0.5454,0.7609,0.7090
GradientBoostingRegressor,0.1390,0.3015,0.3728,0.6449,0.8051,0.7899,0.2099,0.3374,0.4582,0.5455,0.7518,0.7275
AdaBoostRegressor,0.1566,0.3259,0.3957,0.6000,0.7859,0.7669,0.2147,0.3505,0.4634,0.5351,0.7468,0.7153
XGBRegressor,0.1943,0.3527,0.4408,0.5036,0.7128,0.6913,0.2002,0.3204,0.4475,0.5665,0.7691,0.7432
ExtraTreesRegressor,0.1392,0.2881,0.3730,0.6444,0.8083,0.7875,0.2033,0.3365,0.4509,0.5598,0.7586,0.7270
LinearRegression,0.3663,0.4653,0.6053,0.0640,0.6443,0.6339,0.4208,0.4262,0.6487,0.0891,0.6055,0.6653
KNeighborsRegressor,0.2040,0.3450,0.4516,0.4789,0.7173,0.7224,0.2699,0.3589,0.5195,0.4157,0.6554,0.6043
SVR,0.1507,0.3021,0.3883,0.6149,0.7930,0.7960,0.2479,0.3744,0.4979,0.4633,0.6839,0.5948


In [183]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.02648919754043, -5.399638049228945, -5.286...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.056643055289098, -5.969229834308446, -6.1...","[-6.234093306006592, -6.094550514462941, -6.27...","[0.10516698328437081, 0.17953612034007416, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -4.9, -5.05, -5.74, -5.28, -5.92, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.14, -5.14, -6.89, -5.35, -5...","[-6.414, -5.9399999999999995, -5.5259999999999...","[0.49745753587617897, 0.36000000000000015, 0.4..."
2,RandomForestRegressor,"[-6.026999999999999, -5.547499999999997, -5.41...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.070599999999996, -5.933599999999997, -6.3...","[-6.163829999999998, -6.000389999999998, -6.29...","[0.06589847949687509, 0.0969591223145116, 0.14..."
3,GradientBoostingRegressor,"[-6.1817669931130474, -5.376692737933862, -5.1...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0915973762300455, -5.884553291985491, -6....","[-6.190586026773754, -5.973344788987822, -6.07...","[0.07605183833659007, 0.22992861289833047, 0.1..."
4,AdaBoostRegressor,"[-5.815714285714286, -5.363333333333333, -5.32...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.110000000000001, -5.815714285714286, -6.5...","[-6.142150000000001, -5.975386446886446, -6.56...","[0.07908287495476612, 0.1393362043001197, 0.02..."
5,XGBRegressor,"[-6.283933, -5.4378953, -5.0524917, -5.639491,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0439124, -5.780023, -6.196623, -6.016963,...","[-6.2107854, -5.8729043, -6.1814923, -6.176912...","[0.11923515, 0.11115459, 0.19877513, 0.4044085..."
6,ExtraTreesRegressor,"[-5.938299999999997, -5.346799999999999, -5.24...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.075099999999995, -5.8086999999999955, -6....","[-6.1466299999999965, -5.876799999999996, -6.2...","[0.04539215350696664, 0.12202994714413487, 0.1..."
7,LinearRegression,"[-6.629263177201542, -5.801102301460784, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-7.360072212075062, -5.631483382252142, -7.2...","[-7.074316436063542, -5.810943857640355, -6.82...","[0.3590850142785064, 0.26567928983765243, 0.31..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.18, -5.18, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.084000000000001, -5.885999999999999, -6.46...","[0.13957713916604586, 0.04873511168665874, 0.3..."
9,SVR,"[-6.017715768650095, -5.42946366643868, -5.276...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.960711355730181, -5.878746821134055, -6.0...","[-5.9817766271469335, -5.948174320716957, -6.1...","[0.06978896544689862, 0.08209283265691601, 0.2..."


In [184]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_RRCK.csv')

In [185]:
#All 2d and 3d descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 2809)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 2809)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028797 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 91483
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 2626
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1428,0.3041,0.3779,0.6350,0.8005,0.7939,0.2134,0.3359,0.4619,0.5380,0.7441,0.7286
DecisionTreeRegressor,0.2792,0.4086,0.5284,0.2867,0.6277,0.5998,0.2312,0.3567,0.4808,0.4995,0.7213,0.6930
RandomForestRegressor,0.1728,0.3386,0.4156,0.5586,0.7610,0.7506,0.2124,0.3377,0.4609,0.5400,0.7548,0.7086
GradientBoostingRegressor,0.1402,0.3030,0.3744,0.6418,0.8029,0.7915,0.2119,0.3417,0.4603,0.5412,0.7489,0.7200
AdaBoostRegressor,0.1592,0.3293,0.3990,0.5933,0.7834,0.7789,0.2121,0.3432,0.4605,0.5409,0.7519,0.6922
XGBRegressor,0.1943,0.3527,0.4408,0.5036,0.7128,0.6913,0.2002,0.3204,0.4475,0.5665,0.7691,0.7432
ExtraTreesRegressor,0.1378,0.2892,0.3713,0.6479,0.8107,0.7830,0.2046,0.3358,0.4524,0.5570,0.7561,0.7240
LinearRegression,0.3663,0.4653,0.6053,0.0640,0.6443,0.6339,0.4208,0.4262,0.6487,0.0891,0.6055,0.6653
KNeighborsRegressor,0.2040,0.3450,0.4516,0.4789,0.7173,0.7224,0.2699,0.3589,0.5195,0.4157,0.6554,0.6043
SVR,0.1507,0.3021,0.3883,0.6148,0.7930,0.7960,0.2490,0.3750,0.4990,0.4609,0.6819,0.5899


In [186]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.02648919754043, -5.399638049228945, -5.286...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.056643055289098, -5.969229834308446, -6.1...","[-6.234093306006592, -6.094550514462941, -6.27...","[0.10516698328437081, 0.17953612034007416, 0.0..."
1,DecisionTreeRegressor,"[-6.13, -5.14, -5.05, -5.92, -5.28, -5.4, -5.3...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.87, -5.76, -5.14, -5.14, -5.24, -5.35, -5...","[-6.354000000000001, -5.938, -5.47399999999999...","[0.44315234400824294, 0.3610207750254825, 0.34..."
2,RandomForestRegressor,"[-6.014899999999998, -5.584749999999999, -5.43...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.070149999999997, -5.913699999999997, -6.4...","[-6.1862299999999975, -6.011079999999997, -6.2...","[0.083638242449253, 0.10614207742455448, 0.148..."
3,GradientBoostingRegressor,"[-6.096182234804717, -5.391858884417666, -5.17...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.9709072854736895, -5.876173311960456, -6....","[-6.191563844612896, -5.980928043481134, -6.09...","[0.12502410976468295, 0.20389786061176268, 0.1..."
4,AdaBoostRegressor,"[-5.85, -5.3456521739130425, -5.37421052631579...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.07403846153846, -5.85, -6.645652173913044...","[-6.214038686459739, -5.949187114845939, -6.49...","[0.14778870859381749, 0.1501512924889352, 0.16..."
5,XGBRegressor,"[-6.283933, -5.4378953, -5.0524917, -5.639491,...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.0439124, -5.780023, -6.196623, -6.016963,...","[-6.2107854, -5.8729043, -6.1814923, -6.176912...","[0.11923515, 0.11115459, 0.19877513, 0.4044085..."
6,ExtraTreesRegressor,"[-6.021099999999999, -5.348600000000002, -5.25...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.158199999999997, -5.787599999999995, -6.4...","[-6.148079999999997, -5.897349999999996, -6.29...","[0.04304858418113339, 0.14475618121517447, 0.1..."
7,LinearRegression,"[-6.629263177201502, -5.801102301460784, -5.79...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-7.360072212075069, -5.631483382252142, -7.2...","[-7.074316436063543, -5.810943857640359, -6.82...","[0.3590850142785089, 0.26567928983765465, 0.31..."
8,KNeighborsRegressor,"[-5.920000000000001, -5.18, -5.18, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.920000000000001, -5.919999999999999, -6.4...","[-6.084000000000001, -5.885999999999999, -6.46...","[0.13957713916604586, 0.04873511168665874, 0.3..."
9,SVR,"[-6.017727004707456, -5.429448970992933, -5.27...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.960755322276252, -5.878672511098355, -6.0...","[-5.981808882096343, -5.948170992920626, -6.11...","[0.06976077268257189, 0.08209799737280174, 0.2..."


In [187]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_const_rem_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_const_rem_RRCK.csv')

In [188]:
#All 2d and 3d descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 2097)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 2097)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.085499 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 65782
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 1926
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1221,0.2800,0.3495,0.6879,0.8387,0.8334,0.2173,0.3444,0.4662,0.5295,0.7400,0.7304
DecisionTreeRegressor,0.2675,0.4162,0.5172,0.3164,0.6316,0.6224,0.2410,0.3759,0.4909,0.4782,0.7034,0.6223
RandomForestRegressor,0.1718,0.3388,0.4145,0.5611,0.7625,0.7541,0.2143,0.3415,0.4629,0.5361,0.7532,0.7104
GradientBoostingRegressor,0.1398,0.2985,0.3739,0.6428,0.8038,0.7918,0.2191,0.3448,0.4680,0.5257,0.7358,0.7094
AdaBoostRegressor,0.1581,0.3350,0.3976,0.5960,0.7815,0.7716,0.2493,0.3671,0.4993,0.4603,0.6881,0.6587
XGBRegressor,0.1743,0.3348,0.4175,0.5545,0.7455,0.7262,0.2001,0.3327,0.4473,0.5669,0.7736,0.7560
ExtraTreesRegressor,0.1431,0.2953,0.3782,0.6345,0.8023,0.7825,0.2089,0.3418,0.4571,0.5477,0.7481,0.7060
LinearRegression,0.3626,0.4607,0.6022,0.0735,0.6575,0.6602,0.3830,0.4615,0.6189,0.1707,0.6301,0.6842
KNeighborsRegressor,0.2092,0.3467,0.4574,0.4655,0.7088,0.7112,0.2903,0.3856,0.5388,0.3715,0.6247,0.5678
SVR,0.1508,0.3037,0.3884,0.6146,0.7939,0.7905,0.2652,0.3880,0.5150,0.4258,0.6529,0.5640


In [189]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_LVR_RRCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_LVR_RRCK.csv')

In [190]:
#Stacked architecture model
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [191]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    # Identify columns with variance below the threshold
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [192]:
from tqdm import tqdm
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_all_atomic_desc_RRCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_all_atomic_desc_RRCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
target_column = 'Permeability'
def scale_features(df_train, df_test):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test)
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test)
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test)
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test)
models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),   
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(140, 250)
(35, 250)
(140, 347)
(35, 347)
(140, 637)
(35, 637)
(140, 13)
(35, 13)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
107    24  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...       -6.3000   
80     26  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...       -5.3900   
72     27  CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...       -5.4600   
73     28  CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N2CCC[...       -5.2100   
98     29  CC(C)C[C@@H]1NC(=O)[C@H](CC(

In [193]:
dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
target_column = 'Permeability'


meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_train, df_test in tqdm(dataframes, desc="Processing dataframe pairs"):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    # Storing predictions for the current dataframe
    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models"):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for train_index, val_index in kf.split(X_weak):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)

            # Predictions for validation set
            fold_predictions[val_index] =  np.clip( model.predict(X_val), -10, -4.0)

            # Predictions for test set
            test_predictions_fold =  np.clip( model.predict(X_eval), -10, -4.0)
            test_predictions_folds.append(test_predictions_fold)

        # Store predictions for the meta-learner
        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)

# Convert lists to arrays for the meta-learner
meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for train_index, val_index in kf.split(meta_features_train):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        # Predictions for test set
        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -4.0)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    print(f'{model_name} Evaluation completed: Test R2 score: {r2_test}')

    predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_eval,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T
prediction_df = pd.DataFrame(predictions)
results_df

Processing dataframe pairs:   0%|          | 0/4 [00:00<?, ?it/s]
Training models: 0it [00:00, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001953 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8013
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 226
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,


Training models: 1it [00:00,  2.30it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf



Training models: 2it [00:01,  1.07s/it]
Training models: 3it [00:05,  1.99s/it]
Training models: 4it [00:06,  1.70s/it]
Training models: 5it [00:07,  1.56s/it]
Training models: 6it [00:08,  1.50s/it]
Training models: 7it [00:09,  1.07s/it]
Training models: 10it [00:09,  1.02it/s][A
Processing dataframe pairs:  25%|██▌       | 1/4 [00:09<00:29,  9.84s/it]
Training models: 0it [00:00, ?it/s]

...

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 40
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.145423,0.311869,0.381343,0.628445,0.793582,0.788659,0.238448,0.344958,0.488311,0.483754,0.697730,0.631785
DecisionTreeRegressor,0.266562,0.408982,0.516297,0.318935,0.642611,0.622614,0.212366,0.324529,0.460832,0.540221,0.741163,0.665032
RandomForestRegressor,0.144394,0.301671,0.379992,0.631073,0.794820,0.783606,0.212107,0.319904,0.460551,0.540782,0.735393,0.659943
GradientBoostingRegressor,0.152658,0.317255,0.390715,0.609960,0.784365,0.774550,0.218359,0.321696,0.467289,0.527246,0.727270,0.637529
AdaBoostRegressor,0.159250,0.318322,0.399061,0.593117,0.770580,0.759341,0.206275,0.320572,0.454175,0.553408,0.744065,0.668628
XGBRegressor,0.184102,0.338447,0.429071,0.529620,0.742906,0.727974,0.217850,0.336184,0.466744,0.528349,0.734373,0.683897
ExtraTreesRegressor,0.144100,0.304761,0.379605,0.631825,0.795041,0.786300,0.206769,0.313548,0.454719,0.552338,0.743530,0.682076
LinearRegression,0.204930,0.351839,0.452692,0.476405,0.719295,0.697490,0.240193,0.353328,0.490094,0.479976,0.694943,0.645234
KNeighborsRegressor,0.170177,0.320782,0.412525,0.565198,0.753546,0.739164,0.235006,0.340411,0.484774,0.491206,0.704579,0.621393
SVR,0.154153,0.313345,0.392623,0.606140,0.779064,0.774994,0.237294,0.329936,0.487128,0.486251,0.704370,0.638790


In [194]:
results_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/Stacked/Results_5_folds_stacked_archi_RRCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/RRCK/results/Stacked/Prediction_data_5_folds_stacked_archi_RRCK.csv')

In [196]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm
import joblib

# Ensure the models directory exists
os.makedirs('/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/', exist_ok=True)

# Assuming remove_low_variance_columns and features functions are defined elsewhere
# 2D and 3D descriptors dataframes
df_desc_train =pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_desc = features(train, "Permeability")
joblib.dump(selected_features_desc, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/selected_features_descriptors.joblib')
df_desc_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_desc]], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test = df_desc_test.dropna()
df_desc_test = df_desc_test[df_desc_train.columns]

# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_fp = features(train, "Permeability")
joblib.dump(selected_features_fp, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/selected_features_fingerprints.joblib')
df_fp_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_fp]], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test = df_fp_test[df_fp_train.columns]

# Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_emb = features(train, "Permeability")
joblib.dump(selected_features_emb, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/selected_features_embeddings.joblib')
df_emb_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_emb]], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test = df_emb_test[df_emb_train.columns]

# Atomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_all_atomic_desc_RRCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_atomic = features(train, "Permeability")
joblib.dump(selected_features_atomic, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/selected_features_atomic.joblib')
df_atomic_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_atomic]], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_all_atomic_desc_RRCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test = df_atomic_test[df_atomic_train.columns]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Filter dataframes to have consistent IDs
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]
df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]
df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

target_column = 'Permeability'

def scale_features(df_train, df_test, feature_type):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    # Save the scaler
    joblib.dump(scaler, f'/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/scaler_{feature_type}.joblib')
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test, 'Descriptor')
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test, 'Fingerprints')
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test, 'Embeddings')
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test , 'Atomic')

models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]

dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
data_names = ['descriptors', 'fingerprints', 'embeddings', 'atomic']

meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_idx, (df_train, df_test) in enumerate(tqdm(dataframes, desc="Processing dataframe pairs")):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models", total=len(models_weak)):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for fold_idx, (train_index, val_index) in enumerate(kf.split(X_weak)):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)
            
            model_name = model.__class__.__name__
            joblib.dump(model, f'/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/weak_{data_names[df_idx]}_{model_name}_fold_{fold_idx}.joblib')

            fold_predictions[val_index] = np.clip(model.predict(X_val), -10, -4.0)

            test_predictions_fold = np.clip(model.predict(X_eval), -10, -4.0)
            test_predictions_folds.append(test_predictions_fold)

        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)
    
    joblib.dump(fold_meta_features_train, f'/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/meta_features_train_{data_names[df_idx]}.joblib')
    joblib.dump(fold_meta_features_test, f'/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/meta_features_test_{data_names[df_idx]}.joblib')

meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

joblib.dump(meta_features_train, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/meta_features_train_combined.joblib')
joblib.dump(meta_features_test, '/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/meta_features_test_combined.joblib')

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for fold_idx, (train_index, val_index) in enumerate(kf.split(meta_features_train)):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)
        
        joblib.dump(model, f'/home/users/akshay/PCPpred/RRCK/models_rrck_stacked_ensemble/meta_{model_name}_fold_{fold_idx}.joblib')

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -4.0)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    

    predictions.append({
        'Model': model_name,
        'Y Train pred': predictions_train,
        'Y Test actual': y_eval,
        'Test prediction folds': test_predictions_folds,
        'Test Predictions Mean': predictions_test_mean,
        'Test Predictions Std': predictions_test_mean,
    })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(140, 250)
(35, 250)
(140, 347)
(35, 347)
(140, 637)
(35, 637)
(140, 13)
(35, 13)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
107    24  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...       -6.3000   
80     26  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...       -5.3900   
72     27  CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...       -5.4600   
73     28  CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N2CCC[...       -5.2100   
98     29  CC(C)C[C@@H]1NC(=O)[C@H](CC(

Training models:   0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003847 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8013
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 226
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,


Training models:  10%|█         | 1/10 [00:00<00:04,  2.00it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...


Training models:  60%|██████    | 6/10 [00:04<00:03,  1.06it/s]/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimize

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Dimensions of meta_features_train: (140, 40)
Dimensions of meta_features_test: (35, 40)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Stage 1 completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.079940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1485
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 40
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000585 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1477
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 40
[LightGBM] [Info] Start training from score -5.688080
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [197]:
#Ablation study
import os
import joblib
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

In [198]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [199]:
from tqdm import tqdm
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_3d_all_descriptors_RRCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_3d_all_descriptors_RRCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/All_fingerprints_train_RRCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/All_fingerprints_test_RRCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_rrck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_all_atomic_desc_RRCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_all_atomic_desc_RRCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
target_column = 'Permeability'
def scale_features(df_train, df_test):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test)
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test)
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test)
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test)
models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),   
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]
dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(140, 250)
(35, 250)
(140, 347)
(35, 347)
(140, 637)
(35, 637)
(140, 13)
(35, 13)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
107    24  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...       -6.3000   
80     26  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...       -5.3900   
72     27  CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...       -5.4600   
73     28  CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N2CCC[...       -5.2100   
98     29  CC(C)C[C@@H]1NC(=O)[C@H](CC(

In [200]:
ablation_results = {}

for ablation_idx in range(len(dataframes)):
    print(f"========== Ablation: Excluding feature at index {ablation_idx} ==========")
    feature_names = ['Descriptor', 'Fingerprints', 'Embeddings', 'Atomic']
    print(f"========== Ablation: Excluding feature :-- {feature_names[ablation_idx]} ==========")

    ablated_dataframes = [pair for i, pair in enumerate(dataframes) if i != ablation_idx]

    meta_features_train = []
    meta_features_test = []

    # Stage 1
    for df_train, df_test in tqdm(ablated_dataframes, desc="Processing ablated dataframes"):
        X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
        y_weak = df_train[target_column]
        X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
        y_eval = df_test[target_column]

        kf = KFold(n_splits=5, shuffle=True, random_state=101)

        fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
        fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

        for i, model in tqdm(enumerate(models_weak), desc="Training weak models", total=len(models_weak)):
            fold_predictions = np.zeros(X_weak.shape[0])
            test_predictions_folds = []

            for train_index, val_index in kf.split(X_weak):
                X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
                y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]

                model.fit(X_train, y_train)

                fold_predictions[val_index] = np.clip(model.predict(X_val), -10, -4.0)
                test_predictions_fold = np.clip(model.predict(X_eval), -10, -4.0)
                test_predictions_folds.append(test_predictions_fold)

            fold_meta_features_train[:, i] = fold_predictions
            fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)
            print(f'Model training done {i}: {model.__class__.__name__}')

        meta_features_train.append(fold_meta_features_train)
        meta_features_test.append(fold_meta_features_test)
        print('Dataframe training completed')

    # Stack all meta-features
    meta_features_train = np.hstack(meta_features_train)
    meta_features_test = np.hstack(meta_features_test)

    print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
    print('Stage 1 completed (Weak Learners)')
    print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

    # Stage 2
    results = {}
    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    for model in models_meta:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        for train_index, val_index in kf.split(meta_features_train):
            X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
            y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]

            model.fit(X_fold_train, y_fold_train)
            y_pred_fold = np.clip(model.predict(X_fold_val), -10, -4.0)

            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_fold_val)

            test_predictions_fold = model.predict(meta_features_test)
            test_predictions_fold = np.clip(test_predictions_fold, -10, -4.0)
            test_predictions_folds.append(test_predictions_fold)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        mse_test = mean_squared_error(y_eval, predictions_test_mean)
        mae_test = mean_absolute_error(y_eval, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_eval, predictions_test_mean)
        pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
        spearman_test, _ = spearmanr(y_eval, predictions_test_mean)

        results[model_name] = {
            'Train MSE (5 fold CV)': mse_train,
            'Train MAE (5 fold CV)': mae_train,
            'Train RMSE (5 fold CV)': rmse_train,
            'Train R2 (5 fold CV)': r2_train,
            'Train PCC (5 fold CV)': pearson_train,
            'Train SCC (5 fold CV)': spearman_train,
            'Test MSE': mse_test,
            'Test MAE': mae_test,
            'Test RMSE': rmse_test,
            'Test R2': r2_test,
            'Test PCC': pearson_test,
            'Test SCC': spearman_test,
        }

    ablation_results[f"Ablation_{feature_names[ablation_idx]}"] = pd.DataFrame(results).T

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Ablation Study Completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# To view the results
ablation_results_df = {key: value for key, value in ablation_results.items()}


========== Ablation: Excluding feature at index 0 ==========
========== Ablation: Excluding feature :-- Descriptor ==========


Training weak models:   0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002106 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 827
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 153
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 


Training weak models:  10%|█         | 1/10 [00:00<00:03,  2.31it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

Model training done 9: DecisionTreeRegressor
Dataframe training completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Stage 1 completed (Weak Learners)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000815 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1124
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 30
[LightGBM] [Info] Start training from score -5.658237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf



/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: 

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: F

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Ablation Study Completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [201]:
import os
import pickle

ablation_result_dir = '/home/users/akshay/PCPpred/RRCK/results/Ablation/'
os.makedirs(ablation_result_dir, exist_ok=True)

pickle_path = os.path.join(ablation_result_dir, 'ablation_results.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(ablation_results, f)


with open(pickle_path, 'rb') as f:
    ablation_results = pickle.load(f)


ablation_results

{'Ablation_Descriptor':                            Train MSE (5 fold CV)  Train MAE (5 fold CV)  \
 LGBMRegressor                           0.197021               0.359529   
 DecisionTreeRegressor                   0.277550               0.405393   
 RandomForestRegressor                   0.174022               0.331177   
 GradientBoostingRegressor               0.195507               0.345701   
 AdaBoostRegressor                       0.189692               0.346669   
 XGBRegressor                            0.218648               0.367890   
 ExtraTreesRegressor                     0.182434               0.329761   
 LinearRegression                        0.190728               0.333497   
 KNeighborsRegressor                     0.184178               0.342593   
 SVR                                     0.182775               0.341978   
 MLPRegressor                            0.536169               0.632428   
 
                            Train RMSE (5 fold CV)  Train R2 (5

In [202]:
ablation_result_dir = '/home/users/akshay/PCPpred/RRCK/results/Ablation/'
os.makedirs(ablation_result_dir, exist_ok=True)

for ablation_label, df in ablation_results.items():
    print(f"Results for {ablation_label}: \n")
    safe_label = ablation_label.replace(" ", "_").replace("/", "_")
    file_path = os.path.join(ablation_result_dir, f"{safe_label}.csv")
    df.to_csv(file_path)

Results for Ablation_Descriptor: 

Results for Ablation_Fingerprints: 

Results for Ablation_Embeddings: 

Results for Ablation_Atomic: 



In [203]:
from IPython.display import display
for ablation_label, df in ablation_results.items():
    print(f"Results for {ablation_label}: \n")
    display(df)

Results for Ablation_Descriptor: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.197021,0.359529,0.443871,0.496611,0.708960,0.715423,0.204444,0.309757,0.452154,0.557374,0.747261,0.700707
DecisionTreeRegressor,0.277550,0.405393,0.526830,0.290861,0.640421,0.646671,0.219153,0.312400,0.468138,0.525527,0.728708,0.737742
RandomForestRegressor,0.174022,0.331177,0.417159,0.555375,0.745852,0.749438,0.208343,0.321747,0.456446,0.548932,0.742515,0.729285
GradientBoostingRegressor,0.195507,0.345701,0.442162,0.500479,0.715603,0.729033,0.203600,0.313366,0.451220,0.559201,0.749707,0.722701
AdaBoostRegressor,0.189692,0.346669,0.435536,0.515339,0.719640,0.725556,0.224460,0.328565,0.473772,0.514038,0.721301,0.721860
XGBRegressor,0.218648,0.367890,0.467598,0.441356,0.687009,0.687144,0.214407,0.332547,0.463041,0.535803,0.735699,0.704910
ExtraTreesRegressor,0.182434,0.329761,0.427122,0.533883,0.732209,0.733660,0.189268,0.314460,0.435049,0.590229,0.769878,0.749037
LinearRegression,0.190728,0.333497,0.436725,0.512690,0.730168,0.716161,0.210186,0.339614,0.458460,0.544942,0.739321,0.708412
KNeighborsRegressor,0.184178,0.342593,0.429160,0.529426,0.728675,0.711884,0.202461,0.307137,0.449957,0.561666,0.749872,0.710513
SVR,0.182775,0.341978,0.427522,0.533011,0.731412,0.729433,0.193496,0.312720,0.439881,0.581077,0.763323,0.736990


Results for Ablation_Fingerprints: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.147760,0.304641,0.384395,0.622474,0.789822,0.783412,0.246777,0.352395,0.496767,0.465720,0.687319,0.634447
DecisionTreeRegressor,0.289676,0.423661,0.538216,0.259879,0.627204,0.584480,0.184418,0.311757,0.429439,0.600729,0.775777,0.717568
RandomForestRegressor,0.141792,0.295522,0.376553,0.637722,0.799374,0.788119,0.214341,0.319853,0.462970,0.535945,0.732457,0.657001
GradientBoostingRegressor,0.155116,0.311020,0.393848,0.603679,0.782717,0.771215,0.217020,0.317298,0.465854,0.530145,0.730716,0.636128
AdaBoostRegressor,0.152585,0.311924,0.390621,0.610147,0.783511,0.764530,0.204742,0.318904,0.452485,0.556727,0.746545,0.693983
XGBRegressor,0.182745,0.337175,0.427487,0.533087,0.746718,0.733918,0.216763,0.322244,0.465578,0.530702,0.734612,0.680675
ExtraTreesRegressor,0.138237,0.297908,0.371803,0.646804,0.804451,0.795014,0.212917,0.314888,0.461429,0.539029,0.734440,0.683197
LinearRegression,0.172601,0.331827,0.415453,0.559005,0.755506,0.745889,0.223108,0.337092,0.472344,0.516964,0.719215,0.639210
KNeighborsRegressor,0.144874,0.302436,0.380623,0.629848,0.794347,0.779814,0.253985,0.354634,0.503969,0.450115,0.674979,0.605309
SVR,0.147022,0.303863,0.383434,0.624359,0.790909,0.790022,0.253808,0.341079,0.503794,0.450498,0.680447,0.620298


Results for Ablation_Embeddings: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.170406,0.336995,0.412803,0.564612,0.757037,0.743458,0.248183,0.363983,0.498179,0.462678,0.681579,0.602367
DecisionTreeRegressor,0.238972,0.379232,0.488847,0.389428,0.698068,0.683403,0.271932,0.394314,0.521471,0.411260,0.650657,0.531624
RandomForestRegressor,0.161184,0.323493,0.401477,0.588176,0.768768,0.744943,0.228765,0.344990,0.478294,0.504716,0.710435,0.614975
GradientBoostingRegressor,0.180552,0.339858,0.424914,0.538691,0.748058,0.722293,0.240182,0.353769,0.490084,0.479999,0.693640,0.567346
AdaBoostRegressor,0.167832,0.329176,0.409673,0.571191,0.758646,0.750486,0.215954,0.344488,0.464709,0.532453,0.731031,0.659802
XGBRegressor,0.206516,0.349788,0.454441,0.472351,0.707723,0.689925,0.231137,0.349493,0.480767,0.499582,0.707025,0.594383
ExtraTreesRegressor,0.163785,0.327675,0.404704,0.581529,0.765208,0.742462,0.212701,0.325147,0.461195,0.539496,0.736287,0.628423
LinearRegression,0.215530,0.370999,0.464252,0.449321,0.706833,0.692302,0.259240,0.362942,0.509156,0.438737,0.662818,0.625482
KNeighborsRegressor,0.177825,0.331929,0.421692,0.545659,0.739904,0.727627,0.258263,0.346297,0.508196,0.440854,0.667829,0.586398
SVR,0.186623,0.347102,0.431999,0.523179,0.726337,0.713287,0.238770,0.319843,0.488641,0.483057,0.700128,0.626182


Results for Ablation_Atomic: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.144988,0.303464,0.380772,0.629557,0.794852,0.781978,0.244533,0.361262,0.494503,0.470579,0.689781,0.626882
DecisionTreeRegressor,0.277616,0.397446,0.526893,0.290692,0.639043,0.620156,0.198031,0.315714,0.445007,0.571256,0.759345,0.693003
RandomForestRegressor,0.139228,0.300176,0.373133,0.644272,0.802909,0.787006,0.218663,0.324637,0.467615,0.526587,0.726057,0.644393
GradientBoostingRegressor,0.152454,0.312947,0.390454,0.610480,0.786688,0.779245,0.218770,0.330436,0.467728,0.526358,0.726681,0.637389
AdaBoostRegressor,0.155883,0.313521,0.394821,0.601718,0.777389,0.758155,0.201483,0.314040,0.448868,0.563784,0.751794,0.669608
XGBRegressor,0.167545,0.312322,0.409322,0.571924,0.764343,0.750083,0.214609,0.337148,0.463259,0.535365,0.733653,0.682496
ExtraTreesRegressor,0.140872,0.301049,0.375329,0.640073,0.800295,0.784104,0.212531,0.313193,0.461011,0.539864,0.735019,0.652378
LinearRegression,0.171093,0.326918,0.413634,0.562857,0.765331,0.733216,0.238410,0.355349,0.488272,0.483836,0.697317,0.632346
KNeighborsRegressor,0.167853,0.322725,0.409698,0.571137,0.757806,0.758163,0.245387,0.348937,0.495366,0.468729,0.688601,0.612454
SVR,0.146226,0.302014,0.382395,0.626393,0.791746,0.786285,0.247179,0.344512,0.497171,0.464851,0.691615,0.629544


In [206]:
#Saving best model
#KlekotaRoth fingerprints
import os
import joblib 


def train_and_test_predict(models, X_train, y_train, X_test, y_test, save_dir='models_rrck_KR'):
   
    os.makedirs(save_dir, exist_ok=True)

    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        fold_no = 1
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            fold_model_path = os.path.join(save_dir, f"{model_name}_fold{fold_no}_RRCK.joblib")
            joblib.dump(model, fold_model_path)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)

            fold_no += 1

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,
        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

#KlekotaRoth Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRoth_train_RRCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test_RRCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)



# Saving the scaler
joblib.dump(scaler, '/home/users/akshay/PCPpred/RRCK/models_rrck_KR/scaler_rrck_kr.joblib')
models = [
    xgb.XGBRegressor(random_state=101),
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df


X_train shape:  (140, 4860)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 4860)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
XGBRegressor,0.2970,0.3921,0.5449,0.2413,0.5503,0.5733,0.1596,0.3173,0.3995,0.6545,0.8134,0.8095


In [207]:
models_dir = '/home/users/akshay/PCPpred/RRCK/models_rrck_KR' 
scaler_path = '/home/users/akshay/PCPpred/RRCK/models_rrck_KR/scaler_rrck_kr.joblib' 
model_base_name = 'XGBRegressor'                   
n_folds = 5                                    

df_new_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test_RRCK.csv') 

X_new_test_features = df_new_test.drop(columns=['ID', 'SMILES','Permeability'], errors='ignore')
y_test = df_new_test['Permeability']

scaler = joblib.load(scaler_path)
X_new_scaled = scaler.transform(X_new_test_features)
X_new_scaled = pd.DataFrame(X_new_scaled, columns=X_new_test_features.columns,index=X_new_test_features.index)

all_fold_preds = []

for fold in range(1, n_folds + 1):
    fold_model_path = os.path.join(models_dir, f"{model_base_name}_fold{fold}_RRCK.joblib")
    fold_model = joblib.load(fold_model_path)
    preds = fold_model.predict(X_new_scaled)
    preds = np.clip(preds, -10, -4.0)  
    all_fold_preds.append(preds)


all_fold_preds = np.array(all_fold_preds)
mean_prediction = np.mean(all_fold_preds, axis=0)

mse_test = mean_squared_error(y_test, mean_prediction)
print(f"{mse_test:.4f}")
mae_test = mean_absolute_error(y_test, mean_prediction)
print(f"{mae_test:.4f}")
rmse_test = np.sqrt(mse_test)
print(f"{rmse_test:.4f}")
r2_test = r2_score(y_test, mean_prediction)
print(f"{r2_test:.4f}")
pearson_test, _ = pearsonr(y_test, mean_prediction)
print(f"{pearson_test:.4f}")
spearman_test, _ = spearmanr(y_test, mean_prediction)
print(f"{spearman_test:.4f}")

print("Prediction on new data complete.")


0.1596
0.3173
0.3995
0.6545
0.8134
0.8095
Prediction on new data complete.
